> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# v6.9 ·  How can semantic labels enter tools for decision-making? 

 Ben.  notebook  Includes data, a frozen plan, the previous model and tools, candidate heads, random controls, and three successive experiments: 

- **A： Two-way back to the same historical tab. **  Exchange Only  `preview` / `preferred`  Activities generated. 
- **B： Query identity at the same decision-making stage. **  The goal is always to get to it.  X； Check it out first.  Y  or  Z， Both conditions require rechecking  X。 Query identity and end directory refer to independent change. 
- **C： Information source intervention. **  Separately block attention from the current record, other record, tool call, and tool response to the decision position to explore source differences. 

** Run by default first  24  A development scenarios. **  These check new-task competence, natural-generation coverage, and runtime, and do not belong to  96  An independent confirmation of the scenarios. After checking the development report, the grid will be set  `RUN_STAGE`  Change it to...  `"confirmation"`， Keep the same.  `RUN_LABEL`  Again.  Run all， i.e. run built-in and stand-alone  96  confirmation scenarios. Both stages use the same frozen plan; design changes after development require a new version and independent confirmation set. 

 Development Phase Include  **1,456  tool trajectories  + 2,432  A fractional probe. **； Identification phase included  **5,568  tool trajectories  + 7,680  A fractional probe. **。B  Each official world keeps two separate catalogues in order to check correctly first.  X  , the original model capability. C  and expand the random collection to use predefined  8/24  A subset of the world. Two separate scenarios.  smoke  Not into the official denominator. 

 in  Colab  Selection  **A100 80 GB**， Readable  Llama-3.1-8B   `HF_TOKEN`  Enter  Colab Secrets  And turn this on.  notebook  . Select  **Runtime → Run all**， Mount original item  Drive。 Start the session after the first installation if you want to restart  Run all。 Use here  Colab GPU  Amount, not called  GPT API。

** Followed by: **  Preserve  Drive  Accounts, GPU、 Software and  `RUN_LABEL`。 Full Track / Write each probe separately and validate before reuse. If disconnection leaves a write lock, confirm the old session has stopped before entering the reported identifier in  `TAKEOVER_LOCK_ID`。 Complete the current selection, verify the truth  ZIP  and synchronize  Drive  After default automatically disconnects  runtime； Some results or failures do not automatically break. 


In [ ]:
#@title Drive、 Output Position and Run Tabs 
import base64, hashlib, os, re, zlib
from pathlib import Path
from google.colab import drive
RUN_LABEL = "main"
RUN_STAGE = "development"  # development / confirmation
MAX_WALL_HOURS = 0.0  # 0 =  sets no active time limit; a positive value stops and exports at a completed branch boundary  PARTIAL
if RUN_STAGE not in ("development", "confirmation"):
    raise ValueError(" Please choose.  development  or  confirmation")
if not isinstance(MAX_WALL_HOURS, (int,float)) or not 0 <= MAX_WALL_HOURS <= 1000:
    raise ValueError("MAX_WALL_HOURS  Yes  0–1000  Limited number of hours ")
RUN_MODE = "all"  # all / smoke / export
TAKEOVER_LOCK_ID = ""
if RUN_MODE not in ("all","smoke","export"):
    raise ValueError(" Please choose.  all、smoke  or  export")
AUTO_DISCONNECT = True
V69_RESULT = None
if not re.fullmatch(r"[A-Za-z0-9_-]{1,48}",RUN_LABEL):
    raise ValueError("RUN_LABEL  Please use  1–48  Letter, numeral, underlined or short line ")
DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT))
if not os.path.ismount(str(DRIVE_MOUNT)) or not (DRIVE_MOUNT/"MyDrive").is_dir():
    raise RuntimeError(" Please mount first.  MyDrive")
BASE_DIR = DRIVE_MOUNT/"MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p9_semantic_routes"
V69_RUN = BASE_DIR/("routes-"+RUN_LABEL)
V6_CFG = dict(experiment_version="v6_truthful_binding_1",model_name="meta-llama/Llama-3.1-8B",
              model_revision="d04e592bb4f6aa9cfee91e2e20afa771667e1d4b", dtype="bfloat16",
              context_window=4096,max_prompt_tokens=3072,actor_max_new_tokens=128,writer_max_new_tokens=384,
              max_actions=6,use_cache=True,hook_kind="z",data_seed=961103)

print(" Current cycle  Drive  output ",V69_RUN,flush=True)


In [ ]:
if RUN_MODE != "export":
    import sys, subprocess, importlib.metadata as v5_metadata
    if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
        raise RuntimeError(' Fixed-dependent combination needs  Python 3.10–3.13。')
    V6_REQUIRED_VERSIONS = {
        'transformer-lens': '2.17.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0',
        'numpy': '2.1.3' if sys.version_info[:2] == (3, 13) else '1.26.4',
        'pandas': '2.2.3', 'plotly': '6.1.2', 'scipy': '1.15.3',
        'scikit-learn': '1.6.1', 'datasets': '3.6.0', 'pyarrow': '20.0.0',
    }
    V6_INSTALL_PROBLEMS = []
    for v5_name, v5_version in V6_REQUIRED_VERSIONS.items():
        try:
            v5_installed = v5_metadata.version(v5_name)
        except v5_metadata.PackageNotFoundError:
            v5_installed = None
        if v5_installed != v5_version:
            V6_INSTALL_PROBLEMS.append((v5_name, v5_installed, v5_version))
    if V6_INSTALL_PROBLEMS:
        print(' The original experimental fixed version will be installed: ', V6_INSTALL_PROBLEMS)
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        *[f'{name}=={version}' for name, version in V6_REQUIRED_VERSIONS.items()], 'tqdm'], check=True)
        raise SystemExit(' Installation complete. Please restart  Colab runtime， Runs from this block; versions are not repeated after matching. ')
    print(' Fixed reliance matches, skips installation. ')

else:
    print(' Read-only export: Skip this step. ')


In [ ]:
if RUN_MODE != "export":
    #  Run this block after restarting: check if the installation is currently in place  Python  Environment. 
    import sys
    import importlib.metadata as metadata
    import importlib.util

    print("Python:", sys.version.split()[0])
    print(" Interpreter :", sys.executable)
    if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
        raise RuntimeError(" Yes.  Python 3.10–3.13； Please re-implement the installment block after running. ")

    expected_versions = {
        "transformer-lens": "2.17.0", "transformers": "4.57.1",
        "huggingface-hub": "0.36.0",
        "numpy": "2.1.3" if sys.version_info[:2] == (3, 13) else "1.26.4",
        "pandas": "2.2.3",
        "plotly": "6.1.2", "scipy": "1.15.3", "scikit-learn": "1.6.1",
        "datasets": "3.6.0", "pyarrow": "20.0.0",
    }
    problems = []
    for package, expected in expected_versions.items():
        try:
            installed = metadata.version(package)
        except metadata.PackageNotFoundError:
            installed = None
        print(f"{package}: {installed if installed is not None else ' not installed '} ( Yes.  {expected})")
        if installed != expected:
            problems.append(f"{package}:  current  {installed}， Yes.  {expected}")

    for package in ("torch", "tqdm"):
        try:
            print(f"{package}: {metadata.version(package)}")
        except metadata.PackageNotFoundError:
            problems.append(f"{package}:  not installed ")

    try:
        torch_version = metadata.version("torch")
        torch_major_minor = tuple(int(part) for part in torch_version.split("+")[0].split(".")[:2])
        if torch_major_minor < (2, 6):
            problems.append(f"torch:  current  {torch_version}， Yes.  >=2.6")
    except metadata.PackageNotFoundError:
        pass  #  The record is missing. 
    except ValueError:
        problems.append(" Cannot parse  torch  Version, please check the current version  Colab  Run when. ")

    if importlib.util.find_spec("transformer_lens") is None:
        problems.append(" current  Python  Cannot find  transformer_lens  module ")
    if problems:
        raise RuntimeError(
            " Reliance check failed: \n- " + "\n- ".join(problems)
            + "\n Run the first cell and inspect its complete installation log. After successful installation, restart the runtime, "
              " Go on from this block. "
        )
    print(" Reliance on a version check pass; continues to run the third code block. ")

else:
    print(' Read-only export: Skip this step. ')


In [ ]:
# Authenticate before accessing gated Hugging Face model files.
from google.colab import userdata
from huggingface_hub import login
try:
    _archive_hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.") from None
if not isinstance(_archive_hf_token, str) or not _archive_hf_token.strip():
    raise RuntimeError("HF_TOKEN is empty. Set it in Colab Secrets and enable notebook access.")
login(token=_archive_hf_token.strip(), add_to_git_credential=False)
del _archive_hf_token


In [ ]:
if RUN_MODE != "export":
    from google.colab import userdata
    from huggingface_hub import login
    def authenticate_hub():
        hint = " Please.  Colab Secrets  Set Readable  Llama  Model.  HF_TOKEN， ♪ And open this book ♪  notebook  Access rights. "
        try:
            token = userdata.get("HF_TOKEN")
        except Exception:
            raise RuntimeError(hint) from None
        if not isinstance(token,str) or not token.strip():
            raise RuntimeError(hint)
        try:
            login(token=token.strip(), add_to_git_credential=False)
        except Exception:
            raise RuntimeError(hint) from None
    authenticate_hub()

else:
    print(' Read-only export: Skip this step. ')


In [ ]:
if RUN_MODE != "export":
    #  Formal Import: Keep the whole package if there is still an error report here  traceback  In the past few years, many of the worlds leading players have been able to identify specific conflicts. 
    import gc, json, math, os, random, sys, time
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    from dataclasses import dataclass
    from pathlib import Path
    from typing import List, Dict, Tuple, Optional
    import numpy as np
    import pandas as pd
    import plotly.express as px
    import plotly.graph_objects as go
    import torch
    import torch.nn.functional as F
    from scipy import stats
    from tqdm.auto import tqdm
    from transformer_lens import HookedTransformer, utils
    from huggingface_hub import notebook_login, get_token, model_info
    import importlib.metadata as metadata
    from IPython.display import display

    V6_PACKAGE_NAMES = ["torch", "transformer-lens", "transformers", "huggingface-hub", "numpy", "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly"]
    V6_PACKAGE_VERSIONS = {name: metadata.version(name) for name in V6_PACKAGE_NAMES}
    print(V6_PACKAGE_VERSIONS)
    print("Python:", sys.version.split()[0])
    print("CUDA available:", torch.cuda.is_available())
    print(" Reliance on import success. ")

else:
    import gc, json, math, os, random, sys, time
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import torch
    from IPython.display import display
    HookedTransformer = get_token = notebook_login = None
    V6_PACKAGE_VERSIONS = {}
    print(" Read-only export: no model, no access  Hugging Face、 Not requested  GPU。")


# v6.9： Semantic labels, identity queries and sources of information 

 Status: experimental plan frozen before execution. No new model results are included here. 

##  Research issues and evidence available 

 Formerly selected for completion of the mandate.  80  Head. Yes.  v6.8  history-donor swaps can change item-selection scores and final orders. Remaining questions are whether label-induced activity affects actions through these nodes, whether history swaps transfer item identity or action stage, and which input segment causally affects candidate activity. 

 Design lessons  [Stochastic Chameleons（ACL 2025）](https://aclanthology.org/2025.acl-long.1458/)  by source of information, and  [Semantic Bandits](https://arxiv.org/html/2608.16707v1)  separation of task structure from linguistic cues. The former appendix  H.1  source blocking renormalizes remaining attention; the latter is primarily behavioral evidence. Here, for  agent  interventions are newly designed and are not direct replications of those internal mechanisms. 

##  Data, Split and Run Ranges 

-  Development: original fixed generator  `confirm-00265`  to  `confirm-00288`，24  An independent world. 
-  Confirms that: `confirm-00289`  to  `confirm-00384`，96  An independent world. 
-  Technology  smoke：`confirm-00385`、`confirm-00386`， Influencing statistics. 
-  Templates, record sequences and current target locations are formalized at each location  cohort  balance within strata. Freeze new world identifiers, items, targets, and random controls before inspecting model results. 
- A、B  Run every world in the selected stage. The larger random-set score survey and  C  Source Intervention Use Predefined Subset: Before Development  8  One, before confirmation.  24  A world; this is not a subset of choices based on whether or not they were wrong. C  The stand-alone sample number is...  8/24， No, it is not.  24/96。
-  Development phase  1,456  complete tool trajectories and  2,432  A point-only probe; confirmation phase is  5,568  Complete trajectory and  7,680  A probe with a fraction only.  smoke  report separately and exclude from these counts. Multiple conditions within a world are not new independent samples. 
-  Default  `RUN_STAGE="development"`。 The confirmation stage is adopted by the same  notebook、 Switch to the same tab  `"confirmation"`。 The confirmation set is embedded. Do not use its results to choose heads, thresholds, or positions during development. If the design changes after development, preserve old results and create a new version with an independent confirmation set. 

##  Models, Environments and Runners 

 Preserve  `meta-llama/Llama-3.1-8B` BASE， Revision  `d04e592bb4f6aa9cfee91e2e20afa771667e1d4b`，bfloat16、greedy， and the fixed software combinations that history runs. Use the original  A100 80 GB  Configure. The original three tools are  `check_stock`、`reserve_item`、`create_order`； The tool status and parameter validation rules remain as they are. 

 The six-step budget includes one externally executed genuine lookup per trajectory. All subsequent calls are generated by  Actor  Free generation until actual creation of orders, formatting / The agreement fails or the budget is exhausted. X  is in stock in all main experiments. A fallback item does not become correct merely because of an incorrect reservation. 

 Only from  Colab Secrets  Read  `HF_TOKEN`， ♪ In any ♪  Hub  Login before calling. Not source, output, breakpoint or  ZIP  Write the document.  notebook  Do not call for fees  GPT API。

## A： Exchange with two-way labels of history 

 Keep the current request, X/Y  Keep items, tool history, stock, and tool state unchanged; change only the other request field between  `preview_item`（Clex） and  `preferred_item`（C1） Switch between. Check first.  X、 Check it out first.  Y  Both reservations. A  inputs match the original tool-prompt assembly verbatim except for this field. Before execution, verify equal length and a single  token  Replace. 

 Every Tab / first-lookup conditions retain intact execution, candidate self-patching, and patches from the other label. Candidate heads come from the previous work  top80  And the original three groups randomly 80。 The two directions are measured separately: C1  Receive  Clex  Increase in activity  X  The advantages of the process; Clex  Receive  C1  Whether activities are reduced  X  advantage. Donor and recipient query items and actual observations are identical. 

 Decree  `m = logit(X) - logit(Y)`， Its the first distinction.  SKU token   logits。 Label transfer for each head set averages the two directed swap contrasts, then averages over first lookups  X/Y  Equal rights average: 

`T_A = mean_start{[m(C1←Clex)-m(C1)] + [m(Clex)-m(Clex←C1)]}/2`。

 Confirm main estimate  A  as  `T_A(top80) - mean(T_A(random80_1..3))`。 Report both directions, both first lookups, each fixed control separately, and the intact-model label difference. Do not divide by the total label effect to report a mediation percentage. Define secondary order risk differences using the same directed contrast and retain absolute counts. 

## B： Query identity at the same action stage and last reference 

 Current target is  X， The other two requests are for the same relationship.  Y、Z， Stand-alone supplies are  W。 The records of all three requests were authentic. Y/Z  Same stock, same role, first number  token  Different. Check it out first.  Y  Or first.  Z  After that, X  None; both are in the right position and the next step is to check back.  X。 History swapping therefore no longer changes the appropriate next action from rechecking to reservation. 

 Add an explicitly non-executable display catalog containing one occurrence of each  Y  and  Z， In exchange for only the order of the two, so that the final reference to the commodity is  Y  or  Z。 Its part of the search identity.  2×2  paired design. It neither fabricates tool responses nor changes request fields, stock, or tool-call counts. Subsequent calls retain the same catalog condition. 

 For each starting state and catalog order, retain intact execution, candidate self-patching, and swaps of actual query history while keeping the catalog unchanged  Y/Z  four patching groups. Separately swap catalog order in score probes while holding the actual query fixed to measure activity transfer associated with mentioned identity. Query arguments and actual returns still change together, so  B  The result of the call cannot be distinguished separately. 

 Two other original model capabilities per formal world: first physical check  X， and to retain the reference at the end of the sentence.  Y  or  Z  . The same extension is used for this comparison  schema， has no head intervention. Report correct orders, format failures, and tool failures separately to assess basic execution of the new task. It does not enter  Y/Z  Identification transfer owner estimate. 

 Decree  `q = logit(Y)-logit(Z)`。 Query identity transfer is: 

`T_B = mean_mention{[q(H=Z←H=Y)-q(H=Z)] + [q(H=Y)-q(H=Y←H=Z)]}/2`。

 Confirm main estimate  B  as  `T_B(top80) - mean(T_B(random80_1..3))`。 Positive values indicate transfer of score advantage toward the donor queried item, not necessarily purchase of that item. Also report  X  Scores relative to competing commodities, check back  X、 Reservation  Y/Z  Im buying it.  X/Y/Z/W/ Other / no orders. Directly construct a paired query-versus-mention contrast; significance in one condition but not the other does not establish a difference between them. 

## C： Source-directed interventions, exploratory 

 As pre-designated  8/24  In the world, follow.  A  Well check first.  Y  inputs and both labels, separately marking the current record, other record, and history within the actual request block  action  And history.  observation。 The same name segment in the example should not be mislabelled as the current source.  tokenizer  Map to actual  token  position, saves inter-zone and verification information. C  Were not going anywhere else.  X  source blocking therefore does not establish preserved success from normal starts. This evaluates source localization, not a deployable repair. 

 The source contains only those that fall entirely within the character interval.  token； Exclude tokens crossing adjacent-record or field boundaries from blocking to avoid assigning the same  JSON  limits  token  Assigned to two sources. Original characters and  token  Save all spans and verify that the four source categories are disjoint; the intervention still includes record / The point within the field cannot be claimed to touch only the plain semantic word. 

 At the first time of autonomy  SKU  at the decision position, block a source for specified heads and renormalize attention over remaining visible positions. Candidate  top80  Fixed random with three sets 80 Use the same operation. The tool process continues normally after one intervention. C  sources and head sets are frozen before execution; do not reselect layers on confirmation data. 

C  localizes the manipulable influence of a source at these nodes; it does not directly identify a minimal circuit or exclusive semantic encoding. Source hidden states may already contain information from other positions, and blocking redistributes attention weights. The complete  path patching、QK/V  Leave decomposition and additional layer-position searches until localization evidence exists. These unexecuted extensions cannot support the current  notebook  Experiments completed. 

##  Head set and contrast range 

-  candidate  80  The head is used as a fixed coordinate, not at all.  agent  As a result, re-screened. 
-  The first three groups of random candidates that do not overlap 80 For full trajectories, maintenance is comparable to existing research. 
-  Adds to the set of selected surveys  32  Group Synchronized Random 80（ seed  76910–76941）， sample uniformly within layers to match candidate counts, allowing overlap with candidates. These controls generate only first-step score probes and are not full-order controls. 
-  Report candidates against each group and their rank in the fixed survey of sets. Intervals for the original three sets are conditional on those sets; 32  the distribution of these sets is not an exact population description of every head set. The two random references have different sampling limits and are reported separately. 
- 32  The group survey provides a cross-section of the world of independence and head gathering at the same time.  bootstrap， as exploratory uncertainty analysis. Keep it separate from world-only resampling conditional on fixed sets. With a small subset, it is approximate and does not guarantee generalization to all possible head sets. 

##  Intervention location and real generation overlay 

 Preserve  v6.8   item  Before, tool  Its a natural-generated route later on. Its open.  stem `{"item":"sku-`  ♪ And the last ♪  token  when the first differentiated number is projected  token。 The whole thing is filled in before the attention is combined and the output is projected.  `hook_z`；C  Use attention paths corresponding to that individual query. Intervene at this position only, with no later replacement. 

 Same public as the supply.  stem， And verify if it is natural for the original model.  greedy  prefix. Let the recipient generate freely; if it does not naturally reach the designated prefix, do not insert it or restart generation. Keep that trajectory in the order denominator and record its fixed-prefix score and miss flag separately. Report score probes, actual generation, and final orders separately. Self-patching must reproduce the intact model  logits、 actions and terminal states; stop on failure as a technical obstacle, without silently changing tolerances or dropping scenarios. 

##  Statistics, successful findings and cessation rules 

 Independent worlds are the resampling unit; first pair and aggregate directions, labels, first lookups, and catalog orders within each world. Confirmation  A/B  The two masters are expected to do it.  4,000  Number of times  bootstrap， seed  76902， Report on both sides  97.5%  Percentage interval (percentage) Bonferroni  Family name  95%）。 Other compartments are descriptive  95%， Clear marking; full labelling for the development phase. 

 Retain every formal world. Format failures, tool rejections, no orders, and prefix misses remain in their respective denominators. Strict success requires the final tool order, current request, customer, city, and other fields to match the task. Moving an incorrect order to a fallback item or no order is not recovery. Report counts for risk differences / denominators and percentage points, difference in points used  logits， Do not combine these descriptions. 

 Technology  smoke  Inspection  B  Well check it out first.  X  Post-basic completion capacity; retention of the above-mentioned in the formal phase throughout the world  X  The starting point is a comparison of the original model capabilities. None of the two types of checks are mixed.  B   Y/Z  The same phase is based on estimates that the world has not been reduced as a result. 

 Report both templates, both record orders, the intact model, the candidate set, and every random set. Group by prespecified conditions rather than selecting scenarios where the candidate intervention repairs behavior. New  B  Report basic task competence and format performance separately to avoid treating general task incapacity as semantic interference. Errors after an externally fixed first lookup remain conditional risks rather than autonomous error rates. 

 The maximum optional duration limit will only stop at the full branch boundary, and  PARTIAL  export. Resuming with the same label reuses validated results. Stop on failed technical preflight, self-patching, or signatures. Do not add confirmation samples or change heads because the primary effect is nonsignificant. If development suggests changing the task, freeze a new version and run an independent confirmation. 

##  Engineering, time-consuming and archiving 

 Each complete trajectory / probes receive individual atomic checkpoints with a run signature and content digest. Validate existing branches before reusing them. Reject reuse of the same label when the configuration, model identity, frozen data, or source code changes. Retain a single-writer lock and never automatically take over a live session. 

 Record completed and planned units, runtime, and actual generation-path coverage for every module. smoke  And the development phase is used to measure this.  GPU  The real cost is not a fixed number of hours in advance. `export`  Mode only authenticates and packs the saved results, no need  GPU、Hub  Login or model load. 

 The archive should contain frozen inputs, all required source code, head coordinates, protocol, and raw branches / Probes, condition counts, world pairings and  `reports/report.json`。 Yeah.  ZIP  verify  CRC、 List, manifest、 input / Code summary, branch-by-branch signature, and resets the tool terminal. / Identification of separate marking stages; PARTIAL、 Contents, title and  smoke  It does not represent a complete result. 

 Only the chosen stage is complete and true.  ZIP  Verify sum  Drive  Read back successfully, without a writing lock and  Drive  Only release with default settings after synchronisation has been successful  runtime。 Abnormal and partial retention  runtime。Notebook  Only local validation, modeling.  smoke  With formal  GPU  The reasoning is on the user.  Colab  Run. 


In [ ]:
import linecache
V6_CORE_SOURCE = r'''import copy, gc, hashlib, json, math, os, random, re, time
from pathlib import Path


def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def stable_hash(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_heads(heads, n_layers, n_heads):
    result = sorted(set((int(layer), int(head)) for layer, head in heads))
    if any(not (0 <= l < n_layers and 0 <= h < n_heads) for l, h in result):
        raise ValueError("Head coordinates out of bounds")
    return result

def ablation_hooks(model, heads=(), drop_mask=None, hook_kind="z"):
    """Mask query-head outputs, including Llama GQA; never mask KV-head indices."""
    if hook_kind not in {"z", "result"}:
        raise ValueError("hook_kind must be z or result")
    if drop_mask is not None and tuple(drop_mask.shape) != (model.cfg.n_layers, model.cfg.n_heads):
        raise ValueError("Drop mask shape must be [layers, query heads]")
    selected = normalize_heads(heads, model.cfg.n_layers, model.cfg.n_heads)
    if drop_mask is not None and selected:
        raise ValueError("Specify hard heads OR differentiable drop mask")
    by_layer = {}
    for layer, head in selected:
        by_layer.setdefault(layer, []).append(head)
    layers = range(model.cfg.n_layers) if drop_mask is not None else sorted(by_layer)
    hooks = []
    for layer in layers:
        def mask_output(value, hook, layer=layer):
            if drop_mask is not None:
                keep = (1 - drop_mask[layer]).to(device=value.device, dtype=value.dtype)
            else:
                keep = torch.ones(model.cfg.n_heads, device=value.device, dtype=value.dtype)
                keep[by_layer[layer]] = 0
            return value * keep.view(1, 1, -1, 1)
        hooks.append((f"blocks.{layer}.attn.hook_{hook_kind}", mask_output))
    return hooks

class TLBackend:
    def __init__(self, model, cfg):
        self.model = model
        self.tokenizer = model.tokenizer
        self.cfg = dict(cfg)
        self.device = next(model.parameters()).device
        self.hook_kind = cfg.get("hook_kind", "z")
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        model.eval()
        model.set_use_attn_result(self.hook_kind == "result")

    def encode(self, text):
        # One unpadded sequence per forward: no padding-mask/last-position ambiguity.
        return self.tokenizer.encode(text, add_special_tokens=True)

    def tokens(self, text):
        ids = self.encode(text)
        limit = min(int(self.cfg.get("max_prompt_tokens", 3072)), int(self.model.cfg.n_ctx))
        if len(ids) > limit:
            raise ValueError(f"Prompt has {len(ids)} tokens, limit={limit}; evidence is never silently truncated")
        return torch.tensor([ids], dtype=torch.long, device=self.device)

    def one_token_id(self, full_prefix, value):
        before, after = self.encode(full_prefix), self.encode(full_prefix + value)
        if len(after) != len(before) + 1 or after[:-1] != before:
            return None
        token = after[-1]
        if token in set(self.tokenizer.all_special_ids):
            return None
        return token

    def margin(self, snapshot, heads=(), drop_mask=None):
        gold = self.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = self.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            raise ValueError("Snapshot candidates are not two distinct exact-prefix single tokens")
        hooks = ablation_hooks(self.model, heads, drop_mask, self.hook_kind)
        with self.model.hooks(fwd_hooks=hooks):
            logits = self.model(self.tokens(snapshot["prompt"]), return_type="logits")
        # Logit difference equals log P(gold)-log P(distractor).
        return (logits[0, -1, gold] - logits[0, -1, wrong]).float()

    def generate(self, prompt, role="actor", heads=(), max_new_tokens=None, use_cache=True):
        from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
        if role not in {"actor", "writer"}:
            raise ValueError("role must be actor or writer")
        budget = int(max_new_tokens or self.cfg[f"{role}_max_new_tokens"])
        prefix = self.tokens(prompt)
        if prefix.shape[1] + budget > int(self.model.cfg.n_ctx):
            raise ValueError("Prompt plus generation budget exceeds model context")
        cache = HookedTransformerKeyValueCache.init_cache(self.model.cfg, self.device, batch_size=1) if use_cache else None
        generated = []
        sequence = prefix
        stop = "max_new_tokens"
        started = time.perf_counter()
        hooks = ablation_hooks(self.model, heads, hook_kind=self.hook_kind)
        # Fresh cache for every call, same intervention for prefill AND decode.
        with torch.inference_mode(), self.model.hooks(fwd_hooks=hooks):
            for step in range(budget):
                tokens = sequence if step == 0 or not use_cache else sequence[:, -1:]
                logits = self.model(tokens, past_kv_cache=cache, return_type="logits")
                next_id = int(logits[0, -1].argmax().item())
                if next_id == self.tokenizer.eos_token_id:
                    stop = "eos"
                    break
                generated.append(next_id)
                sequence = torch.cat([sequence, torch.tensor([[next_id]], device=self.device)], dim=1)
                decoded = self.tokenizer.decode(generated, skip_special_tokens=False)
                parsed, error = parse_json_output(decoded, dict if role == "actor" else list)
                if error is None:
                    stop = "complete_json"
                    break
        text = self.tokenizer.decode(generated, skip_special_tokens=False)
        del cache
        return {"text": text, "token_count": len(generated), "prompt_tokens": int(prefix.shape[1]),
                "seconds": time.perf_counter() - started, "stop_reason": stop}

def _json_no_duplicates(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result

def parse_json_output(text, expected_type):
    """Strictly parse one JSON object/list; never repair factual values.

    A single enclosing Markdown fence is tolerated. Trailing explanations,
    duplicate keys, NaN/Infinity, and mismatched types are explicit failures.
    """
    if not isinstance(text, str):
        return None, "output_not_text"
    candidate = text.strip()
    if candidate.startswith("```"):
        match = re.fullmatch(r"```(?:json)?\s*\n?(.*?)\n?```", candidate, flags=re.DOTALL | re.IGNORECASE)
        if match is None:
            return None, "invalid_json_fence"
        candidate = match.group(1).strip()
    try:
        value = json.loads(candidate, object_pairs_hook=_json_no_duplicates,
                           parse_constant=lambda token: (_ for _ in ()).throw(ValueError(f"nonfinite JSON: {token}")))
    except (ValueError, TypeError, json.JSONDecodeError) as exc:
        return None, f"invalid_json:{str(exc)[:160]}"
    if not isinstance(value, expected_type):
        return None, f"wrong_json_type:expected_{getattr(expected_type, '__name__', str(expected_type))}"
    return value, None

class OrderEnvironment:
    """A local, deterministic world; it knows inventory, not customer truth."""

    def __init__(self, initial_state):
        self.state = copy.deepcopy(initial_state)
        self.state.setdefault("stock", {})
        self.state.setdefault("reservations", {})
        self.state.setdefault("orders", [])
        self.state.setdefault("next_reservation", 1)
        self.state.setdefault("next_order", 1)
        if any(type(v) is not int or v < 0 for v in self.state["stock"].values()):
            raise ValueError("stock quantities must be nonnegative integers")
        self.tool_log = []

    def snapshot(self):
        return copy.deepcopy(self.state)

    def step(self, action):
        observation = self._execute(action)
        self.tool_log.append({"action": copy.deepcopy(action), "observation": copy.deepcopy(observation)})
        return copy.deepcopy(observation)

    def _execute(self, action):
        if not isinstance(action, dict):
            return {"ok": False, "error": "action_must_be_object"}
        tool = action.get("tool")
        schemas = {
            "check_stock": {"tool", "item"},
            "reserve_item": {"tool", "item"},
            "create_order": {"tool", "customer", "item", "city", "reservation"},
        }
        if not isinstance(tool, str) or tool not in schemas:
            return {"ok": False, "error": "unknown_tool"}
        if set(action) != schemas[tool] or any(not isinstance(v, str) or not v for v in action.values()):
            return {"ok": False, "error": "invalid_arguments", "tool": tool}
        item = action["item"]
        if item not in self.state["stock"]:
            return {"ok": False, "error": "unknown_item", "item": item}
        if tool == "check_stock":
            return {"ok": True, "item": item, "available": self.state["stock"][item]}
        if tool == "reserve_item":
            if self.state["stock"][item] <= 0:
                return {"ok": False, "error": "out_of_stock", "item": item}
            reservation = f"R{self.state['next_reservation']:04d}"
            self.state["next_reservation"] += 1
            self.state["stock"][item] -= 1
            self.state["reservations"][reservation] = {"item": item, "consumed": False}
            return {"ok": True, "item": item, "reservation": reservation}
        reservation = self.state["reservations"].get(action["reservation"])
        if reservation is None or reservation["consumed"] or reservation["item"] != item:
            return {"ok": False, "error": "invalid_reservation"}
        # No oracle city/customer correction: a semantically wrong order can
        # succeed as a tool call. The held-out evaluator scores it afterwards.
        order = {"order_id": f"O{self.state['next_order']:04d}",
                 "customer": action["customer"], "item": item,
                 "city": action["city"], "reservation": action["reservation"]}
        self.state["next_order"] += 1
        self.state["orders"].append(order)
        reservation["consumed"] = True
        return {"ok": True, "order": copy.deepcopy(order)}

def runtime_smoke(backend):
    """Small actual-model interface checks; no scientific outcome is assumed."""
    model = backend.model
    probe = backend.tokens('A short interface check: the city is')
    head = [(min(1, model.cfg.n_layers - 1), 0)]
    original_kind = backend.hook_kind
    try:
        with torch.no_grad():
            model.set_use_attn_result(False)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="z")):
                z_logits = model(probe, return_type="logits")[0, -1].float()
            model.set_use_attn_result(True)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="result")):
                result_logits = model(probe, return_type="logits")[0, -1].float()
        max_difference = float((z_logits - result_logits).abs().max())
        if not torch.allclose(z_logits, result_logits, atol=0.25, rtol=0.02):
            raise RuntimeError(f"z/result ablation disagreement (max={max_difference}); inspect precision before search")
        model.set_use_attn_result(original_kind == "result")
        with torch.enable_grad():
            mask_logits = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=backend.device, requires_grad=True)
            drop = torch.sigmoid(mask_logits)
            with model.hooks(fwd_hooks=ablation_hooks(model, drop_mask=drop, hook_kind=original_kind)):
                score = model(probe, return_type="logits")[0, -1, :16].float().sum()
            score.backward()
            gradient_norm = float(mask_logits.grad.norm())
            if not math.isfinite(gradient_norm) or gradient_norm == 0:
                raise RuntimeError("Mask gradient is zero or nonfinite")
        # Different cache paths may expose tiny dtype differences. A mismatch
        # is a failed software check, never evidence of an intervention effect.
        generation_prompt = 'Complete one JSON object.\n{"check":'
        cache_agreement = None
        if backend.cfg.get("use_cache", True):
            cached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=True)
            uncached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
            cache_agreement = cached["text"] == uncached["text"]
            if not cache_agreement:
                raise RuntimeError("Cached/uncached greedy text differs. Inspect precision or start an explicitly uncached run (use_cache=False); do not silently continue.")
        else:
            backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
        return {"passed": True, "z_result_max_abs_difference": max_difference,
                "mask_gradient_norm": gradient_norm, "cache_text_agreement": cache_agreement,
                "cache_test_skipped": not backend.cfg.get("use_cache", True),
                "cuda_peak_gb": torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None}
    finally:
        backend.hook_kind = original_kind
        model.set_use_attn_result(original_kind == "result")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
'''
V6_RUNTIME_SOURCE = r'''"""Independent, pinned v4 runtime with checked live-model reuse.

No function in this module loads weights until v4_load_runtime is called.
The original backend/tools are supplied verbatim in the same isolated namespace.
"""
import ast as v4_ast
import copy as v4_copy
import hashlib as v4_hashlib
import importlib.metadata as v4_metadata
import json as v4_json
from pathlib import Path as V4Path
import sys as v4_sys
import types as v4_types


V4_MODEL_NAME = "meta-llama/Llama-3.1-8B"
V4_MODEL_REVISION = "d04e592bb4f6aa9cfee91e2e20afa771667e1d4b"
V4_PACKAGE_NAMES = (
    "torch", "transformer-lens", "transformers", "huggingface-hub", "numpy",
    "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly",
)
V4_FIXED_CONFIG = {
    "model_name": V4_MODEL_NAME, "model_revision": V4_MODEL_REVISION,
    "dtype": "bfloat16", "context_window": 4096, "max_prompt_tokens": 3072,
    "actor_max_new_tokens": 128, "writer_max_new_tokens": 384,
    "max_actions": 6, "use_cache": True, "hook_kind": "z",
}


def v4_backend_contract(cfg):
    """Only settings that can change the original generation implementation."""
    keys = ("context_window", "max_prompt_tokens", "actor_max_new_tokens",
            "writer_max_new_tokens", "use_cache", "hook_kind", "dtype", "model_name")
    if not isinstance(cfg, dict) or any(key not in cfg for key in keys):
        raise ValueError("Incomplete backend generation contract")
    return {key: cfg[key] for key in keys}


def v4_code_state(code):
    """Executable-code identity independent of notebook filename/line numbers."""
    def v4_constant(value):
        if isinstance(value, v4_types.CodeType):
            return {"code": v4_code_state(value)}
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if isinstance(value, tuple):
            return {"tuple": [v4_constant(item) for item in value]}
        if isinstance(value, frozenset):
            return {"frozenset": sorted((v4_constant(item) for item in value), key=repr)}
        if value is None or isinstance(value, (bool, int, float, str)):
            return value
        return {"type": type(value).__name__, "repr": repr(value)}
    return {
        "bytecode": code.co_code.hex(), "constants": [v4_constant(x) for x in code.co_consts],
        "names": code.co_names, "variables": code.co_varnames,
        "free": code.co_freevars, "cells": code.co_cellvars,
        "args": code.co_argcount, "posonly": code.co_posonlyargcount,
        "kwonly": code.co_kwonlyargcount, "flags": code.co_flags & 0xffff,
        "exception_table": getattr(code, "co_exceptiontable", b"").hex(),
    }


def v4_definition_state(value):
    if isinstance(value, type):
        methods = {}
        for name, member in vars(value).items():
            member = member.__func__ if isinstance(member, (staticmethod, classmethod)) else member
            if isinstance(member, v4_types.FunctionType):
                methods[name] = v4_definition_state(member)
        return {"class": value.__name__, "bases": [base.__name__ for base in value.__bases__],
                "methods": methods}
    if isinstance(value, v4_types.MethodType):
        value = value.__func__
    if isinstance(value, v4_types.FunctionType):
        return {"code": v4_code_state(value.__code__),
                "defaults": v4_copy.deepcopy(value.__defaults__),
                "kwdefaults": v4_copy.deepcopy(value.__kwdefaults__)}
    return v4_copy.deepcopy(value)


def v4_source_exports(source):
    names = []
    for node in v4_ast.parse(source).body:
        if isinstance(node, (v4_ast.FunctionDef, v4_ast.ClassDef)):
            names.append(node.name)
        elif isinstance(node, v4_ast.Assign):
            names.extend(target.id for target in node.targets if isinstance(target, v4_ast.Name))
    return names


def v4_check_sources(core_source, runtime_source):
    """Check live definitions against the bundled sources, without inference."""
    if not isinstance(core_source, str) or not core_source.strip() or not isinstance(runtime_source, str) or not runtime_source.strip():
        raise ValueError("Both exact runtime source strings are required")
    expected = dict(globals())
    names = []
    for label, source in (("core", core_source), ("runtime", runtime_source)):
        exec(compile(source, "<v4-expected-" + label + ">", "exec", dont_inherit=True), expected)
        names.extend(v4_source_exports(source))
    if len(names) != len(set(names)):
        raise RuntimeError("Bundled core/runtime definitions overlap")
    states = {name: v4_definition_state(expected[name]) for name in names}
    for name in names:
        if name not in globals() or stable_hash(v4_definition_state(globals()[name])) != stable_hash(states[name]):
            raise RuntimeError("Live definition does not match bundled source: " + name)
    return states


def v4_check_packages(package_versions):
    if not isinstance(package_versions, dict) or set(package_versions) != set(V4_PACKAGE_NAMES):
        raise ValueError("Use the original complete PACKAGE_VERSIONS schema")
    actual = {name: v4_metadata.version(name) for name in V4_PACKAGE_NAMES}
    if actual != package_versions or str(torch.__version__) != actual["torch"]:
        raise RuntimeError("Installed/imported packages differ from the supplied package identity")
    for name, required in (("transformer-lens", "2.17.0"), ("transformers", "4.57.1"),
                           ("huggingface-hub", "0.36.0")):
        if actual[name] != required:
            raise RuntimeError("Original model stack required: " + name + "==" + required)
    return actual


def v4_plain_state(value):
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    if isinstance(value, dict):
        return {str(key): v4_plain_state(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [v4_plain_state(item) for item in value]
    return {"type": type(value).__module__ + "." + type(value).__qualname__, "repr": str(value)}


def v4_tokenizer_state(tokenizer):
    fields = ("name_or_path", "vocab_size", "model_max_length", "padding_side", "truncation_side",
              "bos_token_id", "eos_token_id", "pad_token_id", "unk_token_id", "all_special_ids",
              "special_tokens_map", "added_tokens_encoder", "add_bos_token", "add_eos_token",
              "clean_up_tokenization_spaces")
    return {"class": type(tokenizer).__module__ + "." + type(tokenizer).__qualname__,
            **{key: v4_plain_state(getattr(tokenizer, key, None)) for key in fields}}


def v4_no_active_hooks(model):
    hook_points = getattr(model, "hook_dict", None)
    if not isinstance(hook_points, dict):
        raise RuntimeError("Model lacks the original TransformerLens hook registry")
    for name, point in hook_points.items():
        if getattr(point, "fwd_hooks", ()) or getattr(point, "bwd_hooks", ()):
            raise RuntimeError("Active TransformerLens hooks at " + str(name))
    if not callable(getattr(model, "modules", None)):
        raise RuntimeError("Model does not expose its module hook state")
    for module in model.modules():
        for field in ("_forward_hooks", "_forward_pre_hooks", "_backward_hooks", "_backward_pre_hooks"):
            if getattr(module, field, None):
                raise RuntimeError("Active module hooks: " + field)


def v4_parameter_state(model):
    result = []
    for parameter in model.parameters():
        result.append({"object_id": id(parameter), "version": getattr(parameter, "_version", None),
                       "shape": list(getattr(parameter, "shape", ())),
                       "dtype": str(parameter.dtype), "device": str(parameter.device),
                       "requires_grad": bool(parameter.requires_grad),
                       "data_ptr": int(parameter.data_ptr()) if callable(getattr(parameter, "data_ptr", None)) else None})
    if not result:
        raise RuntimeError("Loaded model has no parameters")
    return result


def v4_check_model(model, cfg):
    if isinstance(HookedTransformer, type) and type(model) is not HookedTransformer:
        raise RuntimeError("The loaded model is not the original HookedTransformer class")
    v4_no_active_hooks(model)
    if bool(getattr(model, "training", True)):
        raise RuntimeError("Model must already be in evaluation mode")
    if any(bool(getattr(model, key, False)) for key in ("is_loaded_in_4bit", "is_loaded_in_8bit")):
        raise RuntimeError("Quantized models cannot be reused")
    if int(model.cfg.n_ctx) != cfg["context_window"]:
        raise RuntimeError("Model context differs from the frozen backend contract")
    for field, expected in (("n_layers", 32), ("n_heads", 32), ("d_model", 4096), ("d_head", 128)):
        if getattr(model.cfg, field, None) != expected:
            raise RuntimeError("Model architecture differs at " + field)
    if getattr(model.cfg, "use_attn_result", False):
        raise RuntimeError("Model attention-result setting differs from the original z backend")
    parameters = v4_parameter_state(model)
    if any(item["dtype"] != str(torch.bfloat16) or item["requires_grad"]
           or item["device"].split(":")[0] != "cuda" for item in parameters):
        raise RuntimeError("Model parameters must all be frozen, CUDA, and bfloat16")
    if len({item["device"] for item in parameters}) != 1:
        raise RuntimeError("The original single-device model contract is required")
    tokenizer = model.tokenizer
    if not callable(getattr(tokenizer, "encode", None)) or not callable(getattr(tokenizer, "decode", None)):
        raise RuntimeError("Tokenizer encode/decode interface is missing")
    if tokenizer.eos_token_id is None or tokenizer.pad_token_id is None:
        raise RuntimeError("Tokenizer eos/padding setup is incomplete")
    module_code, modules = {}, []
    for module in model.modules():
        forward = getattr(module, "forward", None)
        function = forward.__func__ if isinstance(forward, v4_types.MethodType) else forward
        function_id = id(function)
        if function_id not in module_code:
            module_code[function_id] = (stable_hash(v4_code_state(function.__code__))
                                       if hasattr(function, "__code__") else None)
        modules.append({"object_id": id(module), "class": type(module).__module__ + "." + type(module).__qualname__,
                        "forward_object_id": function_id, "forward_code": module_code[function_id]})
    return {"parameters": parameters, "config": v4_plain_state(vars(model.cfg)),
            "tokenizer": v4_tokenizer_state(tokenizer), "modules": modules}


def v4_check_backend(backend, cfg, model_identity, expected_identity):
    if stable_hash(model_identity) != stable_hash(expected_identity):
        raise RuntimeError("Candidate model/revision/dtype/packages/Python identity differs")
    if type(backend).__name__ != "TLBackend" or stable_hash(v4_definition_state(type(backend))) != stable_hash(v4_definition_state(TLBackend)):
        raise RuntimeError("Candidate backend class differs from the copied original TLBackend")
    for name, method in vars(TLBackend).items():
        if isinstance(method, v4_types.FunctionType):
            actual = getattr(backend, name, None)
            if stable_hash(v4_definition_state(actual)) != stable_hash(v4_definition_state(method)):
                raise RuntimeError("Candidate backend method differs: " + name)
            function = actual.__func__ if isinstance(actual, v4_types.MethodType) else actual
            for dependency in ("normalize_heads", "ablation_hooks", "parse_json_output", "_json_no_duplicates"):
                reference = function.__globals__.get(dependency)
                if reference is None or stable_hash(v4_definition_state(reference)) != stable_hash(v4_definition_state(globals()[dependency])):
                    raise RuntimeError("Candidate backend dependency differs: " + dependency)
    if v4_backend_contract(backend.cfg) != v4_backend_contract(cfg) or backend.hook_kind != cfg["hook_kind"]:
        raise RuntimeError("Candidate generation settings differ")
    if backend.tokenizer is not backend.model.tokenizer:
        raise RuntimeError("Candidate backend/model tokenizers differ")
    state = v4_check_model(backend.model, cfg)
    if str(backend.device) != state["parameters"][0]["device"]:
        raise RuntimeError("Candidate backend device differs from its weights")
    return state


def v4_load_runtime(cfg, runtime_dir, package_versions, core_source, runtime_source, candidates=()):
    """Load/reuse pinned weights and return an independent parent namespace.

    The same original TLBackend wraps a compatible live model with a new config
    dictionary. No old backend configuration, run directory, or result is changed.
    Live checks protect code, identities, package versions, model/tokenizer config,
    parameter objects/versions/storage and hooks. They do not hash all 8B weight
    values; reuse trusts the supplied reviewed pinned model-run identity.
    """
    if not isinstance(cfg, dict):
        raise TypeError("cfg must be a dictionary")
    frozen_cfg = v4_copy.deepcopy(cfg)
    for key, expected in V4_FIXED_CONFIG.items():
        if key not in frozen_cfg or type(frozen_cfg[key]) is not type(expected) or frozen_cfg[key] != expected:
            raise ValueError("This notebook preserves the original fixed setting: " + key)
    if type(frozen_cfg.get("data_seed")) is not int or not 0 <= frozen_cfg["data_seed"] < 2**32:
        raise ValueError("data_seed must be an integer in [0, 2**32)")
    if not isinstance(frozen_cfg.get("experiment_version"), str) or not frozen_cfg["experiment_version"].strip():
        raise ValueError("A standalone experiment_version is required")
    packages = v4_check_packages(v4_copy.deepcopy(package_versions))
    source_states = v4_check_sources(core_source, runtime_source)
    identity = {"name": frozen_cfg["model_name"], "revision": frozen_cfg["model_revision"],
                "dtype": frozen_cfg["dtype"], "packages": packages,
                "python": v4_sys.version.split()[0]}
    implementation = stable_hash({"core_source": core_source, "runtime_source": runtime_source})
    fingerprint = stable_hash({"config": frozen_cfg, "model": identity, "implementation": implementation})
    directory = V4Path(runtime_dir).expanduser().resolve()
    manifest = {
        "schema": "conflict_v4_standalone_runtime_v1", "config": frozen_cfg,
        "model": identity, "implementation": implementation, "fingerprint": fingerprint,
        "generation_contract": v4_backend_contract(frozen_cfg),
        "sources": {"core": core_source, "runtime": runtime_source},
        "source_sha256": {"core": v4_hashlib.sha256(core_source.encode("utf-8")).hexdigest(),
                          "runtime": v4_hashlib.sha256(runtime_source.encode("utf-8")).hexdigest()},
    }
    manifest_path = directory / "manifest.json"
    if manifest_path.exists() and v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Locked runtime identity differs. Choose a fresh v4 directory before loading weights.")
    # This guard precedes login, downloads, rewrapping, or any inference.
    if not torch.cuda.is_available():
        raise RuntimeError("The unchanged 8B runtime requires a CUDA GPU")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("The unchanged runtime requires bfloat16 support")
    attempts, wrapped, mode, selected_label = [], None, None, None
    for index, candidate in enumerate(candidates):
        label = str(candidate.get("label", "candidate-" + str(index))) if isinstance(candidate, dict) else "candidate-" + str(index)
        try:
            if not isinstance(candidate, dict) or "backend" not in candidate or "model_identity" not in candidate:
                raise ValueError("Candidate must contain backend and model_identity")
            old_backend = candidate["backend"]
            v4_check_backend(old_backend, frozen_cfg, candidate["model_identity"], identity)
            old_cfg = v4_copy.deepcopy(old_backend.cfg)
            wrapped = TLBackend(old_backend.model, frozen_cfg)
            if old_backend.cfg != old_cfg:
                raise RuntimeError("Wrapping unexpectedly modified the previous backend config")
            v4_check_backend(wrapped, frozen_cfg, identity, identity)
            mode, selected_label = "reuse_live_model", label
            attempts.append({"label": label, "accepted": True})
            break
        except Exception as error:
            wrapped = None
            attempts.append({"label": label, "accepted": False,
                             "reason": type(error).__name__ + ": " + str(error)})
    login_called = False
    if wrapped is None:
        if not get_token():
            notebook_login()
            login_called = True
        seed_everything(frozen_cfg["data_seed"])
        model = HookedTransformer.from_pretrained(
            frozen_cfg["model_name"], revision=frozen_cfg["model_revision"],
            device="cuda", dtype=torch.bfloat16, n_ctx=frozen_cfg["context_window"],
            fold_ln=False, center_writing_weights=False, center_unembed=False,
        )
        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        if model.tokenizer.pad_token is None:
            model.tokenizer.pad_token = model.tokenizer.eos_token
        wrapped = TLBackend(model, frozen_cfg)
        v4_check_backend(wrapped, frozen_cfg, identity, identity)
        mode = "load_pinned_model"
    model = wrapped.model
    model_state = v4_check_model(model, frozen_cfg)
    tokenizer = model.tokenizer
    source_state_hash = stable_hash(source_states)
    manifest_snapshot = v4_copy.deepcopy(manifest)
    # Recheck before committing in case a concurrent cell changed the lock.
    if manifest_path.exists() and v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest_snapshot:
        raise RuntimeError("Runtime manifest changed while initializing the model")
    atomic_json(manifest_path, manifest_snapshot)
    metadata = {
        "mode": mode, "reused_candidate": selected_label, "candidate_attempts": attempts,
        "login_called": login_called, "device": str(wrapped.device),
        "weight_download": "not requested" if mode == "reuse_live_model" else "HuggingFace normal local cache; download only if required",
        "model_identity_policy": "pinned loader or compatible live model with reviewed run identity",
        "live_weight_guard": "parameter identity/version/storage checks; no full weight-content digest",
        "original_backend_config_changed": False,
    }
    parent = {
        "backend": wrapped, "model": model, "RUN_DIR": directory,
        "CFG": v4_copy.deepcopy(frozen_cfg), "MODEL_IDENTITY": v4_copy.deepcopy(identity),
        "IMPLEMENTATION_HASH": implementation, "RUN_FINGERPRINT": fingerprint,
        "PACKAGE_VERSIONS": v4_copy.deepcopy(packages),
        "runtime_manifest": v4_copy.deepcopy(manifest_snapshot),
        "loader_reuse": v4_copy.deepcopy(metadata), "runtime_metadata": v4_copy.deepcopy(metadata),
    }

    def v4_verify_live_run():
        if parent.get("backend") is not wrapped or parent.get("model") is not model:
            raise RuntimeError("Standalone runtime backend/model was replaced")
        if V4Path(parent["RUN_DIR"]).expanduser().resolve() != directory:
            raise RuntimeError("Standalone runtime directory changed")
        if (stable_hash(parent["CFG"]) != stable_hash(frozen_cfg)
                or stable_hash(parent["MODEL_IDENTITY"]) != stable_hash(identity)
                or parent["IMPLEMENTATION_HASH"] != implementation
                or parent["RUN_FINGERPRINT"] != fingerprint):
            raise RuntimeError("Standalone runtime identity/config was modified")
        if wrapped.cfg != frozen_cfg:
            raise RuntimeError("Live backend configuration changed")
        if model.tokenizer is not tokenizer or wrapped.tokenizer is not tokenizer:
            raise RuntimeError("Live tokenizer object changed")
        live_states = {name: v4_definition_state(globals().get(name)) for name in source_states}
        if stable_hash(live_states) != source_state_hash:
            raise RuntimeError("Live backend/tools/runtime implementation changed")
        if v4_sys.version.split()[0] != identity["python"] or v4_check_packages(packages) != packages:
            raise RuntimeError("Live package/Python identity changed")
        v4_check_backend(wrapped, frozen_cfg, identity, identity)
        if stable_hash(v4_check_model(model, frozen_cfg)) != stable_hash(model_state):
            raise RuntimeError("Live model parameters, configuration, or tokenizer settings changed")
        if (not manifest_path.exists()
                or v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest_snapshot):
            raise RuntimeError("Standalone runtime manifest changed on disk")
        if stable_hash(parent["runtime_manifest"]) != stable_hash(manifest_snapshot):
            raise RuntimeError("In-memory runtime manifest changed")
        return True

    parent["verify_live_run"] = v4_verify_live_run
    v4_verify_live_run()
    reuse_path = directory / "loader-reuse.json"
    previous = v4_json.loads(reuse_path.read_text(encoding="utf-8")) if reuse_path.exists() else {"run_fingerprint": fingerprint, "loads": []}
    if previous.get("run_fingerprint") != fingerprint or not isinstance(previous.get("loads"), list):
        raise RuntimeError("Loader reuse provenance belongs to another runtime")
    previous["loads"].append(metadata)
    atomic_json(reuse_path, previous)
    return parent
'''
V6_RUNTIME_NAMESPACE = {'__name__': 'v6_runtime', 'np': np, 'torch': torch,
                        'HookedTransformer': HookedTransformer, 'get_token': get_token, 'notebook_login': notebook_login}
for v5_source, v5_filename in ((V6_CORE_SOURCE, '<v5-core>'), (V6_RUNTIME_SOURCE, '<v5-runtime>')):
    linecache.cache[v5_filename] = (len(v5_source), None, v5_source.splitlines(True), v5_filename)
    exec(compile(v5_source, v5_filename, 'exec'), V6_RUNTIME_NAMESPACE)


In [ ]:
V6_LEGACY_SOURCE = r'''from functools import lru_cache
import copy as gx_copy

GX_ATTRIBUTES = ("delivery_city", "order_customer", "preferred_item")

GX_DEMO_CITIES = ("Valletta", "Reykjavik", "Suva", "La Paz")

GX_ORDER_FIELDS = {
    "delivery_city": "city", "order_customer": "customer", "preferred_item": "item",
}

def gx_task_payload(case):
    """Only the other two fixed fields and fallback accompany the request ID."""
    return {"request_id": case["request_id"], "resolve_attribute": case["attribute"],
            "fixed_fields": {attribute: case["gold_fields"][attribute]
                             for attribute in GX_ATTRIBUTES if attribute != case["attribute"]},
            "fallback_item": case["fallback_item"]}

def gx_actor_examples():
    """Independent tool examples cover all field mappings and real fallback.

    These short teaching cases use only demo cities and identifiers. They never
    depend on evaluation cases, and do not furnish actions for evaluated tasks.
    """
    examples = []
    for index, attribute in enumerate(GX_ATTRIBUTES):
        request = ("req-demo-71d3", "req-demo-a926", "req-demo-4f80")[index]
        customer = ("acct-demo-8c31", "acct-demo-90ea", "acct-demo-2d74")[index]
        preferred = ("sku-demo-31ab", "sku-demo-e072", "sku-demo-64dc")[index]
        fallback = ("sku-demo-8f62", "sku-demo-72de", "sku-demo-0a13")[index]
        city = GX_DEMO_CITIES[index]
        fields = {"delivery_city": city, "order_customer": customer, "preferred_item": preferred}
        available = index != 2
        env = OrderEnvironment({"stock": {preferred: int(available), fallback: 2},
                                "next_reservation": (7, 19, 41)[index]})
        events = []

        def gx_demo_step(action):
            observation = env.step(action)
            events.append({"action": action, "observation": observation})
            return observation

        observation = gx_demo_step({"tool": "check_stock", "item": preferred})
        chosen = preferred if observation["available"] > 0 else fallback
        if chosen != preferred:
            gx_demo_step({"tool": "check_stock", "item": fallback})
        reservation = gx_demo_step({"tool": "reserve_item", "item": chosen})["reservation"]
        gx_demo_step({"tool": "create_order", "customer": customer, "item": chosen,
                      "city": city, "reservation": reservation})
        examples.append({
            "task": {"request_id": request, "resolve_attribute": attribute,
                     "fixed_fields": {key: value for key, value in fields.items() if key != attribute},
                     "fallback_item": fallback},
            "memory": [{"kind": "verified_profile", "entity": request, "attribute": attribute,
                        "value": fields[attribute], "source_event": "evt-demo-" + str(index)}],
            "events": events,
        })
    return examples

@lru_cache(maxsize=1)
def gx_actor_example_text():
    """Teach every next-call phase, including the initially empty history.

    The v5 smoke model copied the final-call pattern before any reservation.
    Render every legal prefix of the same three independent teaching cases,
    not only the final create_order prefix. Every observation is produced by
    the unchanged tools. All ten examples are static and identical across arms;
    no demonstration is chosen using an evaluated task or its hidden truth.
    """
    continuations = []
    for example in gx_actor_examples():
        for step, event in enumerate(example["events"]):
            continuations.append(
                "INDEPENDENT EXAMPLE:\nTASK_JSON:\n" + canonical_json(example["task"])
                + "\nMEMORY_JSON:\n" + canonical_json(example["memory"])
                + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(example["events"][:step])
                + "\nNEXT_TOOL_CALL:\n" + canonical_json(event["action"])
                + "\nEND_EXAMPLE")
    return "\n".join(continuations)

def actor_prompt(case, memory, events):
    """Present the effective arm's memory verbatim; do not insert focal gold."""
    visible = [{"action": event.get("action"), "observation": event.get("observation")}
               for event in events if event.get("action") is not None]
    return (
        "You are an order-processing assistant. Execute one order using the tools.\n"
        "A request_id identifies a request, not a customer account. Resolve the indicated attribute "
        "for that request from its records, then combine it with the task's fixed_fields. "
        "delivery_city maps to order.city; order_customer maps to order.customer; preferred_item "
        "is the SKU to check and reserve when available. The fallback_item is allowed only if "
        "the resolved preferred SKU is out of stock.\n"
        "The records belong to one system. Match the request ID and the indicated attribute. Actions and tool observations do not update the request fields.\n"
        "Check the preferred SKU's stock. If unavailable, check the fallback. Reserve the chosen "
        "available SKU, then create exactly one order. Copy the reservation ID only from its "
        "successful reserve_item observation; a stock count is not a reservation.\n"
        "Output exactly one complete JSON object for the next tool call, without explanation. "
        "Every create_order requires all four arguments: customer, item, city, reservation. "
        "Use city as the tool key even when the task field is named delivery_city.\n"
        'Tools: {"tool":"check_stock","item":"SKU"}; '
        '{"tool":"reserve_item","item":"SKU"}; '
        '{"tool":"create_order","customer":"ACCOUNT","item":"SKU","city":"CITY","reservation":"RESERVATION"}.\n'
        "Independent next-call examples (never copy their values or IDs):\n"
        + gx_actor_example_text()
        + "\nEND_EXAMPLES. Execute this independent request:\n"
        + case["template_text"].format(request_id=case["request_id"])
        + "\nOnly this request's EXECUTED_TOOL_HISTORY_JSON records completed actions. "
        "An empty history means nothing has been checked or reserved: start with check_stock "
        "for the resolved preferred SKU. After a positive stock result use reserve_item. "
        "If the preferred SKU is unavailable, check the fallback before reserving it. "
        "Use create_order only after reserve_item succeeded in this request's history. "
        "Never invent a reservation ID or take one from an example."
        + "\nGenerate only the next call. For create_order include customer, item, city, "
        "and the reservation from a successful reserve_item observation. Resolve city from "
        "this request's delivery_city, including when it is supplied in fixed_fields."
        + "\nTASK_JSON:\n" + canonical_json(gx_task_payload(case))
        + "\nMEMORY_JSON:\n" + canonical_json(memory)
        + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(visible)
        + "\nNEXT_TOOL_CALL:\n"
    )

def gx_expected_order(case, initial_state=None):
    """Evaluator-only final fields, including stock-contingent SKU truth."""
    state = case["initial_env"] if initial_state is None else initial_state
    fields = case["gold_fields"]
    preferred = fields["preferred_item"]
    item = preferred if state["stock"].get(preferred, 0) > 0 else case["fallback_item"]
    return {"city": fields["delivery_city"], "customer": fields["order_customer"], "item": item}

def gx_value_missing(value):
    """Absent/null/blank values are not explicit choices of a wrong value.

    Nonstring values are schema/type failures, not choices of a string-valued
    attribute. They are excluded from wrong-choice rates by gx_wrong_value;
    the schema-complete indicators still expose their invalidity.
    """
    return value is None or (isinstance(value, str) and not value.strip())

def gx_wrong_value(value, expected):
    """Count only an explicitly supplied, nonblank string that is incorrect."""
    return isinstance(value, str) and bool(value.strip()) and value != expected

def gx_order_schema_complete(action):
    """A complete order attempt has exactly five keys with nonblank strings.

    This is a diagnostic; the original executable schema remains unchanged.
    It deliberately treats a whitespace-only value as missing even if the
    executable tool's nonempty-string check would accept that value.
    """
    return (isinstance(action, dict)
            and set(action) == {"tool", "customer", "item", "city", "reservation"}
            and action.get("tool") == "create_order"
            and all(isinstance(value, str) and bool(value.strip()) for value in action.values()))

def gx_score_actor(case, events, orders, initial_state):
    """Score accepted fields and the executed prerequisite subsequence.

    A wrong focal field is counted on any accepted order, independently of the
    correctness of the other fields. Missing orders are failures and have their
    own indicator; they never masquerade as correct field decisions. Tool
    recovery is allowed: strict success requires a valid final check/reserve/
    order subsequence, while all earlier tool errors remain reported separately.

    Intermediate rates are source-level booleans, not counts of repeated calls.
    The first order attempt is the first parsed create_order action, including
    a rejected one; any-attempt metrics cover all such actions. Missing or
    nonstring focal values do not count as explicit wrong choices. Reservation
    acceptance requires that reserve_item's own observation has ok=True.
    Expected items include the legal initial-stock fallback. Conflict-SKU
    reservation metrics apply only to preferred_item sources (None otherwise).
    """
    expected = gx_expected_order(case, initial_state)
    focal = GX_ORDER_FIELDS[case["attribute"]]
    correct = len(orders) == 1 and all(orders[0].get(key) == value for key, value in expected.items())
    preferred = case["gold_fields"]["preferred_item"]
    chosen = expected["item"]
    checked, grounded = {}, set()
    checked_preferred = checked_chosen = reserved_chosen = path_valid = False
    tool_errors = []
    actions = []
    order_attempts, reservation_attempts, accepted_reservations = [], [], []
    for step, event in enumerate(events):
        action = event.get("action")
        observation = event.get("observation")
        if isinstance(action, dict):
            actions.append(action)
            if action.get("tool") == "create_order":
                order_attempts.append(action)
            elif action.get("tool") == "reserve_item":
                reservation_attempts.append(action)
                if isinstance(observation, dict) and observation.get("ok") is True:
                    accepted_reservations.append(action)
        if not isinstance(action, dict) or not isinstance(observation, dict):
            continue
        if not observation.get("ok", False):
            tool_errors.append({"step": step, "error": observation.get("error")})
            continue
        tool, item = action.get("tool"), action.get("item")
        if tool == "check_stock":
            checked[item] = observation["available"]
            checked_preferred |= item == preferred
            checked_chosen |= item == chosen and observation["available"] > 0
        elif tool == "reserve_item" and item == chosen:
            reserved_chosen = True
            priority_ok = (checked.get(preferred, -1) > 0 if chosen == preferred
                           else checked.get(preferred, -1) == 0)
            if priority_ok and checked.get(chosen, 0) > 0:
                grounded.add(observation["reservation"])
        elif tool == "create_order" and item == chosen:
            path_valid |= action.get("reservation") in grounded
    protocol_errors = {"action_must_be_object", "unknown_tool", "invalid_arguments"}
    format_failure = any(event.get("parse_error") is not None for event in events) or any(
        error["error"] in protocol_errors for error in tool_errors)
    first_order = order_attempts[0] if order_attempts else None
    order_fields = ("customer", "item", "city", "reservation")
    first_missing_attribute = first_order is not None and gx_value_missing(first_order.get(focal))
    first_missing_city = first_order is not None and gx_value_missing(first_order.get("city"))
    first_missing_required = first_order is not None and any(
        gx_value_missing(first_order.get(field)) for field in order_fields)
    conflict_item_applicable = case["attribute"] == "preferred_item"
    return {
        "task_success": bool(correct), "strict_task_success": bool(correct and path_valid),
        "accepted_wrong_attribute": any(order.get(focal) != expected[focal] for order in orders),
        "accepted_conflict_value": any(order.get(focal) == case["conflict_value"] for order in orders),
        "accepted_correct_attribute": any(order.get(focal) == expected[focal] for order in orders),
        "accepted_wrong_other_fields": any(any(order.get(key) != value for key, value in expected.items()
                                                if key != focal) for order in orders),
        "no_accepted_order": not orders, "accepted_order_count": len(orders),
        "format_failure": bool(format_failure), "tool_error": bool(tool_errors),
        "tool_error_count": len(tool_errors), "action_count": len(actions),
        "order_attempted": first_order is not None,
        "order_accepted": bool(orders), "order_attempt_count": len(order_attempts),
        "first_order_attribute_correct": first_order is not None and first_order.get(focal) == expected[focal],
        "first_order_conflict_value": first_order is not None and first_order.get(focal) == case["conflict_value"],
        "any_order_conflict_value": any(action.get(focal) == case["conflict_value"] for action in order_attempts),
        "first_order_wrong_attribute": first_order is not None and gx_wrong_value(first_order.get(focal), expected[focal]),
        "any_order_wrong_attribute": any(gx_wrong_value(action.get(focal), expected[focal]) for action in order_attempts),
        "first_order_missing_attribute": bool(first_missing_attribute),
        "any_order_missing_attribute": any(gx_value_missing(action.get(focal)) for action in order_attempts),
        "first_order_missing_city": bool(first_missing_city),
        "any_order_missing_city": any(gx_value_missing(action.get("city")) for action in order_attempts),
        "first_order_missing_required_fields": bool(first_missing_required),
        "any_order_missing_required_fields": any(any(gx_value_missing(action.get(field)) for field in order_fields)
                                                  for action in order_attempts),
        "first_order_schema_complete": first_order is not None and gx_order_schema_complete(first_order),
        "any_order_schema_complete": any(gx_order_schema_complete(action) for action in order_attempts),
        "reservation_attempted": bool(reservation_attempts),
        "reservation_accepted": bool(accepted_reservations),
        "reservation_attempt_count": len(reservation_attempts),
        "accepted_reservation_count": len(accepted_reservations),
        "attempted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)
                                                 for action in reservation_attempts),
        "accepted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)
                                                for action in accepted_reservations),
        "attempted_conflict_reservation_item": (
            any(action.get("item") == case["conflict_value"] for action in reservation_attempts)
            if conflict_item_applicable else None),
        "accepted_conflict_reservation_item": (
            any(action.get("item") == case["conflict_value"] for action in accepted_reservations)
            if conflict_item_applicable else None),
        "prerequisite_path_valid": bool(path_valid),
        "checked_preferred": bool(checked_preferred), "checked_chosen_available": bool(checked_chosen),
        "reserved_chosen": bool(reserved_chosen),
        "subgoal_progress": (int(checked_preferred) + int(checked_chosen) + int(reserved_chosen) + int(correct)) / 4,
    }
'''


In [ ]:
V6_TASKS_SOURCE = r'''"""Truthful request pairs, matched contexts, and the original three-tool workflow."""
import copy
import json
import random
from collections import Counter
from functools import lru_cache


TEMPLATES = {
    "smoke": ("Process request {request_id}. Place exactly one order using its resolved fields.",),
    "pilot": ("Fulfil request {request_id} by placing one order using its resolved fields.",
              "Arrange one shipment for request {request_id} using its resolved order details."),
    "confirm": ("Submit one order for request {request_id} using the fields belonging to that request.",
                "Complete the purchase specified by request {request_id} using its recorded details."),
}
SPLIT_RANGES = {"smoke": (100000, 199999), "pilot": (200000, 399999), "confirm": (400000, 999999)}
DATA_SEED = 961103


def make_worlds(split, count):
    if split not in SPLIT_RANGES or type(count) is not int or not 0 < count <= 10000:
        raise ValueError("Unknown split or invalid number of independent worlds")
    rng = random.Random(f"v6/worlds/{split}/{DATA_SEED}")
    lo, hi = SPLIT_RANGES[split]
    used = set()

    def identifier(kind):
        while True:
            digits = str(rng.randint(lo, hi))
            if digits not in used:
                used.add(digits)
                return f"{kind}-{digits[:3]}-{digits[3:]}"

    worlds = []
    for i in range(count):
        # The first n worlds are stable if a larger manifest is generated later.
        # A/B, gold, and competitor appear only in evaluator metadata.
        requests = [identifier("req"), identifier("req")]
        items = [identifier("sku"), identifier("sku")]
        lots = [identifier("lot"), identifier("lot")]
        worlds.append({
            "id": f"{split}-{i+1:05d}", "index": i, "split": split,
            "requests": requests, "items": items, "lots": lots,
            "events": [identifier("evt"), identifier("evt")],
            "fallback": identifier("sku"), "customer": identifier("acct"),
            "city": rng.choice(("Lisbon", "Osaka", "Cairo", "Prague", "Mumbai", "Nairobi")),
            "template": i % len(TEMPLATES[split]), "reverse": bool((i // 2) % 2),
        })
    return worlds


def case_for(w, target, stocked=True):
    other = 1 - target
    stock = {sku: 2 for sku in (*w["items"], w["fallback"])}
    stock[w["items"][target]] = 2 if stocked else 0
    return {
        "source_id": w["id"], "stage": w["split"], "attribute": "preferred_item",
        "request_id": w["requests"][target], "template_id": str(w["template"]),
        "template_text": TEMPLATES[w["split"]][w["template"]],
        "gold_fields": {"preferred_item": w["items"][target], "delivery_city": w["city"],
                        "order_customer": w["customer"]},
        "conflict_value": w["items"][other], "fallback_item": w["fallback"],
        "profile_event_id": w["events"][target], "note_event_id": w["events"][other],
        "stock_available": stocked, "initial_env": {"stock": stock, "orders": [], "reservations": {},
                                                   "next_reservation": 1, "next_order": 1},
    }


def records_for(w, target, context):
    if context not in ("C0", "C1") or target not in (0, 1):
        raise ValueError("Invalid target/context")
    records = []
    for index in range(2):
        has_item = index == target or context == "C1"
        records.append({"kind": "verified_profile", "entity": w["requests"][index],
                        "attribute": "preferred_item" if has_item else "dispatch_batch",
                        "value": w["items"][index] if has_item else w["lots"][index],
                        "source_event": w["events"][index]})
    return list(reversed(records)) if w["reverse"] else records


@lru_cache(maxsize=1)
def direct_examples():
    lines = []
    for w in make_worlds("smoke", 3):
        # Distinct demo-only identifiers even from the usability smoke.
        for group in ("requests", "items", "lots", "events"):
            w[group] = [v.replace("-", "-demo-", 1) for v in w[group]]
        for t in (0, 1):
            lines.append("EXAMPLE REQUEST " + w["requests"][t] + "\nRECORDS " +
                         canonical_json(records_for(w, t, ("C0", "C1")[t])) +
                         "\nANSWER " + canonical_json({"item": w["items"][t]}))
    return "\n".join(lines)


def direct_prompt(w, target, context):
    # Same target, fixed fields, records and order as Actor. Read-only, one model call.
    case = case_for(w, target)
    return ("Read the request records. Answer with one JSON object containing only the item field. "
            "Return the preferred SKU for the specified request.\n" + direct_examples() +
            "\nTASK_JSON:\n" + canonical_json(gx_task_payload(case)) +
            "\nMEMORY_JSON:\n" + canonical_json(records_for(w, target, context)) + "\nANSWER:\n")


def arm_sets(global_heads, random_count):
    global_heads = [tuple(h) for h in global_heads]
    if (len(global_heads) != 80 or len(set(global_heads)) != 80
            or any(not (0 <= l < 32 and 0 <= h < 32) for l, h in global_heads)):
        raise ValueError("Expected 80 distinct query heads of the original Llama-3.1-8B")
    arms = {"none": [], "global80": global_heads}
    for draw in range(random_count):
        rng = random.Random(61401 + draw)
        pick = []
        for layer, count in sorted(Counter(l for l, h in global_heads).items()):
            pool = [h for h in range(32) if (layer, h) not in global_heads]
            pick.extend((layer, h) for h in rng.sample(pool, count))
        arms[f"random{draw+1}"] = sorted(pick)
    return arms


def run_actor(backend, case, records, heads, forced_check=False, *, first_lookup_item=None):
    env = OrderEnvironment(case["initial_env"])
    events = []
    if first_lookup_item is not None and (not forced_check or first_lookup_item not in
                                          (case["gold_fields"]["preferred_item"], case["conflict_value"])):
        raise ValueError("A controlled first lookup must use one of this world's two SKUs")
    if forced_check:
        item = case["gold_fields"]["preferred_item"] if first_lookup_item is None else first_lookup_item
        action = {"tool": "check_stock", "item": item}
        events.append({"action": action, "observation": env.step(action), "parse_error": None,
                       "origin": "fixed_prefix", "step": 0})
    end = "action_budget"
    while len(events) < backend.cfg["max_actions"]:
        prompt = actor_prompt(case, records, events)
        if len(backend.encode(prompt)) > backend.cfg["max_prompt_tokens"]:
            # A pathological generated argument can be echoed in multiple tool errors.
            # Score this execution failure rather than truncate evidence or lose a world.
            end = "prompt_budget"
            break
        out = backend.generate(prompt, role="actor", heads=heads, use_cache=True)
        action, error = parse_json_output(out["text"], dict)
        obs = None if error else env.step(action)
        events.append({"step": len(events), "origin": "model", "prompt": prompt,
                       "action": action, "observation": obs, "parse_error": error, "generation": out})
        if error:
            end = "format_failure"
            break
        if obs.get("error") in ("action_must_be_object", "unknown_tool", "invalid_arguments"):
            end = "protocol_failure"
            break
        if action.get("tool") == "create_order" and obs.get("ok"):
            end = "order_created"
            break
    orders = env.state["orders"]
    metrics = gx_score_actor(case, events, orders, case["initial_env"])
    first = (events[0]["action"] or {}) if events else {}
    first_ok = bool(events and events[0]["observation"] and events[0]["observation"].get("ok"))
    checked = first_ok and first.get("tool") == "check_stock"
    metrics.update(first_lookup_other=bool(checked and first.get("item") == case["conflict_value"]),
                   first_lookup_gold=bool(checked and first.get("item") == case["gold_fields"]["preferred_item"]),
                   first_valid_lookup=bool(checked), prompt_budget_hit=end=="prompt_budget",
                   action_budget_hit=end=="action_budget")
    return {"metrics": metrics, "events": events, "initial_state": case["initial_env"],
            "final_state": env.snapshot(), "terminal": end, "forced_check": forced_check}


def direct_choice(backend, w, target, context):
    prompt = direct_prompt(w, target, context)
    out = backend.generate(prompt, role="actor", heads=(), max_new_tokens=48, use_cache=True)
    answer, error = parse_json_output(out["text"], dict)
    schema_ok = not error and set(answer) == {"item"} and isinstance(answer["item"], str) and bool(answer["item"].strip())
    item = answer["item"] if schema_ok else None
    valid = schema_ok and item in (*w["items"],w["fallback"])
    return {"prompt": prompt, "generation": out, "item": item, "valid": bool(valid), "schema_valid": bool(schema_ok),
            "parse_error": error, "other": item == w["items"][1-target], "correct": item == w["items"][target]}


def candidate_tokens(backend, prefix, item):
    # Every candidate begins with sku-. The identical stem cancels from the log odds.
    if not item.startswith("sku-"):
        raise ValueError("Expected a SKU")
    ids = backend.encode(prefix)
    full = backend.encode(prefix + item[4:] + '"}')
    if full[:len(ids)] != ids or not len(ids) < len(full) <= backend.cfg["max_prompt_tokens"]:
        raise ValueError("Candidate retokenized the controlled prefix, or exceeded the context budget")
    return ids, full


def score_pair(backend, prompt, gold, other, heads):
    prefix = prompt + '{"tool":"check_stock","item":"sku-'
    alternatives = [candidate_tokens(backend, prefix, v) for v in (gold, other)]
    if len(alternatives[0][1]) != len(alternatives[1][1]):
        raise ValueError("The two complete value/terminator sequences must have equal token counts")
    scores = []
    with torch.inference_mode(), backend.model.hooks(fwd_hooks=ablation_hooks(backend.model, heads)):
        for before, full in alternatives:
            tokens = torch.tensor([full[:-1]], dtype=torch.long, device=backend.device)
            # Each row predicts the NEXT token. Include the identical JSON terminator.
            logits = backend.model(tokens, return_type="logits")[0, len(before)-1:, :].float()
            targets = torch.tensor(full[len(before):], device=backend.device)
            selected = logits.log_softmax(-1).gather(1, targets[:, None])
            scores.append(float(selected.sum().item()))
    return {"gold_logp": scores[0], "other_logp": scores[1], "margin": scores[0]-scores[1],
            "candidate_tokens": len(alternatives[0][1])-len(alternatives[0][0])}


def check_inputs(backend, worlds):
    maximum = 0
    for w in worlds:
        for target in (0, 1):
            case = case_for(w, target)
            lengths = []
            for context in ("C0", "C1"):
                records = records_for(w, target, context)
                prompt = actor_prompt(case, records, [])
                dp = direct_prompt(w, target, context)
                prefix = prompt + '{"tool":"check_stock","item":"sku-'
                counts = [len(candidate_tokens(backend, prefix, item)[1]) for item in w["items"]]
                if counts[0] != counts[1]:
                    raise ValueError(f"SKU tokenizer mismatch in {w['id']}")
                lengths.append((len(backend.encode(prompt)), len(backend.encode(dp))))
                maximum = max(maximum, *counts, *lengths[-1])
            if lengths[0] != lengths[1]:
                raise ValueError(f"C0/C1 token counts differ in {w['id']}, target {target}: {lengths}")
    if maximum > backend.cfg["max_prompt_tokens"]:
        raise ValueError("Untruncated prompt exceeds budget")
    return {"worlds": len(worlds), "equal_lengths": True, "max_initial_tokens": maximum,
            "later_steps": "original backend checks each full prompt before generation"}
'''


In [ ]:
V6_STUDY_SOURCE = r'''"""One atomic checkpoint per condition/target/arm, plus separate forced-history controls."""
import csv
import hashlib
import math
import os
import tempfile
import time
import zipfile
from pathlib import Path


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(dir=path.parent, prefix=".pending-", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(value, f, ensure_ascii=False, allow_nan=False, separators=(",", ":"))
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(k for row in rows for k in row))
    fd, temporary = tempfile.mkstemp(dir=path.parent, prefix=".pending-", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


def paper_reference(backend, paper, arms):
    def margin(prompt, gold, other, heads):
        with torch.inference_mode(), backend.model.hooks(fwd_hooks=ablation_hooks(backend.model, heads)):
            logits = backend.model(backend.tokens(prompt), return_type="logits")[0, -1]
            return float((logits[gold].float() - logits[other].float()).item())
    rows = []
    for c in paper["cases"]:
        gold, other = c["gold_token_id"], c["distractor_token_id"]
        for name, tok in (("gold", gold), ("distractor", other)):
            if backend.tokenizer.encode(c[name+"_token_str"], add_special_tokens=False) != [tok]:
                raise ValueError("Paper reference token ID mismatch")
        row = dict(c)
        row["clean_none"] = margin(c["clean_prompt"], gold, other, [])
        for arm in ("none", "global80", "random1"):
            row["attacked_"+arm] = margin(c["attacked_prompt"], gold, other, arms[arm])
        rows.append(row)
    drop = np.mean([r["clean_none"]-r["attacked_none"] for r in rows])
    gain = np.mean([r["attacked_global80"]-r["attacked_none"] for r in rows])
    rand = np.mean([r["attacked_random1"]-r["attacked_none"] for r in rows])
    return {"rows": rows, "mean_attack_drop": float(drop), "global80_gain": float(gain),
            "random1_gain": float(rand), "direction_pass": bool(drop > 0 and gain > 0)}


class Study:
    def __init__(self, root, backend, worlds, smoke_worlds, arms, identity, verify, scorer=score_pair):
        self.root, self.backend = Path(root), backend
        self.worlds, self.smoke_worlds = worlds, smoke_worlds
        self.arms, self.verify, self.scorer = arms, verify, scorer
        self.manifest = dict(identity, worlds=worlds, smoke_worlds=smoke_worlds, arms=arms)
        self.signature = stable_hash(self.manifest)
        lock = self.root / "manifest.json"
        if lock.exists():
            if stable_hash(read_json(lock)) != self.signature:
                raise RuntimeError(" Runs different identities or samples. Please select a new one.  RUN_LABEL， Keeps the original directory. ")
        else:
            if self.root.exists() and any(self.root.iterdir()):
                raise RuntimeError(" Turns out the directory is not empty but missing  manifest， Please keep the original directory and select a new label. ")
            save_json(lock, self.manifest)
        self.new_units = 0

    def check(self):
        self.verify()
        if stable_hash(self.manifest) != self.signature or stable_hash(read_json(self.root/"manifest.json")) != self.signature:
            raise RuntimeError(" Run configuration or disk  manifest  Change ")

    def unit(self, key, fn=None):
        path = self.root / (key + ".json")
        tag = stable_hash([self.signature, key])
        if path.exists():
            obj = read_json(path)
            if obj["signature"] != tag or obj["digest"] != stable_hash(obj["result"]):
                raise RuntimeError(" Checkpoint check failed : " + str(path))
            return obj["result"]
        if fn is None:
            return None
        value = fn()
        save_json(path, {"signature": tag, "digest": stable_hash(value), "result": value})
        self.new_units += 1
        return value

    def smoke(self):
        self.check()
        rows = []
        for w in self.smoke_worlds:
            t = w["index"] % 2
            def execute():
                case = case_for(w, t, stocked=w["index"] % 2 == 0)
                act = run_actor(self.backend, case, records_for(w, t, "C0"), [])
                return {"actor": act, "direct": direct_choice(self.backend, w, t, "C0")}
            out = self.unit("smoke/"+w["id"], execute)
            m = out["actor"]["metrics"]
            rows.append({"world": w["id"], "stocked": w["index"] % 2 == 0,
                         "strict": m["strict_task_success"], "first_valid": m["first_lookup_gold"],
                         "format_failure": m["format_failure"], "direct_correct": out["direct"]["correct"]})
        ok = (sum(r["strict"] for r in rows) >= 5 and all(r["first_valid"] for r in rows)
              and not any(r["format_failure"] for r in rows) and sum(r["direct_correct"] for r in rows) >= 5
              and all(sum(r["strict"] for r in rows if r["stocked"] == stock) >= 2 for stock in (True, False)))
        result = {"pass": bool(ok), "rows": rows}
        if self.unit("controls/readiness",lambda: result) != result:
            raise RuntimeError("Readiness summary disagrees with saved smoke trajectories")
        save_json(self.root/"smoke-report.json", result)
        return result

    def prepare(self, paper):
        self.check()
        audit = self.unit("controls/input-audit", lambda: check_inputs(self.backend, self.worlds+self.smoke_worlds))
        reference = self.unit("controls/old-paper", lambda: paper_reference(self.backend, paper, self.arms))
        smoke = self.smoke()
        print(" Enter pair check ", audit)
        print(" The original paper is in contrast. ", {k:v for k,v in reference.items() if k != "rows"})
        print(" Tools and a reply pre-check ", smoke)
        return smoke, reference

    def main_unit(self, w, t, context, arm):
        case, records, heads = case_for(w, t), records_for(w, t, context), self.arms[arm]
        result = {"world": w["id"], "target": t, "context": context, "arm": arm,
                  "template": w["template"], "records": records, "target_id": case["request_id"]}
        result["probe"] = self.scorer(self.backend, actor_prompt(case, records, []),
                                      case["gold_fields"]["preferred_item"], case["conflict_value"], heads)
        result["actor"] = run_actor(self.backend, case, records, heads)
        if arm == "none":
            result["direct"] = direct_choice(self.backend, w, t, context)
        return result

    def run(self, arm_names, limit=None):
        self.check()
        if limit is not None and (type(limit) is not int or limit < 1):
            raise ValueError("WORLDS_PER_CALL must be a positive integer or None")
        readiness = self.unit("controls/readiness")
        if not readiness or not readiness["pass"] or self.unit("controls/input-audit") is None:
            raise RuntimeError(" First complete and pass six independent pre-screens, check.  smoke-report.json  And the original output. ")
        if any(a != "none" for a in arm_names) and not self.unit("controls/old-paper")["direction_pass"]:
            raise RuntimeError(" The original-paper positive control did not recover the expected direction. Intact results were retained; inspect the model and masks first. ")
        for arm in arm_names:
            done = 0
            for w in self.worlds:
                keys = [(t,c,f"main/{w['id']}/{arm}-{t}-{c}") for t in (0,1) for c in ("C0","C1")]
                # First eight prespecified worlds; one balanced target; identical prefix except stock observation.
                forks = [(stock,f"fork/{w['id']}/{arm}-{int(stock)}") for stock in (True,False)] if w["index"] < 8 else []
                if all((self.root/(k+".json")).exists() for *_,k in [*keys,*forks]):
                    continue
                for t,c,k in keys:
                    self.unit(k, lambda: self.main_unit(w, t, c, arm))
                for stock,k in forks:
                    t = w["index"] % 2
                    self.unit(k, lambda: {"world":w["id"], "target":t, "arm":arm, "stocked":stock,
                        "actor":run_actor(self.backend, case_for(w,t,stock), records_for(w,t,"C1"),
                                          self.arms[arm], forced_check=True)})
                done += 1
                print(f"[{arm}] {w['id']} complete | {self.new_units} new checkpoint units", flush=True)
                if limit is not None and done >= limit:
                    break
        self.check()

    def collect(self):
        main, forks = [], []
        for w in self.worlds:
            for arm in self.arms:
                for t in (0,1):
                    for ctx in ("C0","C1"):
                        r = self.unit(f"main/{w['id']}/{arm}-{t}-{ctx}")
                        if r is None:
                            continue
                        m = r["actor"]["metrics"]
                        calls = [e["generation"] for e in r["actor"]["events"] if e["origin"]=="model"]
                        row = {k:r[k] for k in ("world","target","context","arm","template")}
                        row.update(m, **r["probe"], model_calls=len(calls), output_tokens=sum(c["token_count"] for c in calls),
                                   generation_seconds=sum(c["seconds"] for c in calls))
                        if "direct" in r:
                            row.update({"direct_"+k:r["direct"][k] for k in ("valid","schema_valid","correct","other")})
                        main.append(row)
                if w["index"] < 8:
                    for stock in (True,False):
                        r = self.unit(f"fork/{w['id']}/{arm}-{int(stock)}")
                        if r is not None:
                            forks.append({"world":w["id"], "target":r["target"], "arm":arm,
                                          "stocked":stock, **r["actor"]["metrics"]})
        return main, forks

    def export(self):
        main, forks = self.collect()
        report = analyze(self, main, forks)
        dest = self.root/"reports"
        write_csv(dest/"main.csv", main)
        write_csv(dest/"stock-forks.csv", forks)
        write_csv(dest/"conditions.csv", report["conditions"])
        write_csv(dest/"paired-effects.csv", report["effects"])
        write_csv(dest/"power-planning.csv", report["planning"])
        save_json(dest/"report.json", report)
        archive = self.root.parent/("agent_entrainment_v6_"+self.root.parent.name+"_"+report["status"]+"_results.zip")
        tmp = archive.with_suffix(".zip.pending")
        with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as z:
            for f in sorted(self.root.rglob("*")):
                if f.is_file() and f.suffix in (".json",".csv"):
                    z.write(f, f.relative_to(self.root))
        os.replace(tmp,archive)
        return report, archive
'''


In [ ]:
V6_REPORTS_SOURCE = r'''"""Paired, world-level pilot estimation. A and B never count as independent worlds."""
import math
from functools import lru_cache
from statistics import NormalDist
import numpy as np


def effect_row(name, values, unit):
    a = np.asarray(values, dtype=float)
    out = {"contrast": name, "unit": unit, "n_worlds": len(a),
           "estimate": None, "ci95_low": None, "ci95_high": None,
           "positive_worlds": int((a>0).sum()), "negative_worlds": int((a<0).sum()),
           "zero_worlds": int((a==0).sum())}
    if len(a):
        out["estimate"] = float(a.mean())
    if len(a) > 1 and np.ptp(a) > 0:
        rng = np.random.default_rng(76102)
        draws = np.concatenate([a[rng.integers(len(a),size=(100,len(a)))].mean(axis=1) for _ in range(30)])
        lo, hi = np.quantile(draws,[.025,.975])
        out.update(ci95_low=float(lo), ci95_high=float(hi))
    return out


@lru_cache(maxsize=512)
def exact_scenario_power(n, q, delta):
    """Unconditional power of the two-sided exact conditional McNemar test.

    One predesignated binary observation per world in each of two conditions.
    q = P(10)+P(01); delta = P(01)-P(10). Normality is not assumed.
    """
    from scipy.stats import binom
    if not (n > 0 and 0 < q < 1 and 0 < delta <= q):
        raise ValueError("Require 0 < delta <= discordance < 1")
    lo, hi = binom.ppf([1e-13,1-1e-13], n, q).astype(int)
    d = np.arange(lo,hi+1)
    k = binom.ppf(.025,d,.5).astype(int)
    k -= (binom.cdf(k,d,.5) > .025 + 1e-14).astype(int)
    p = (1 + delta/q)/2
    reject = binom.cdf(k,d,p) + binom.sf(d-k-1,d,p)
    return float(np.sum(binom.pmf(d,n,q)*reject))


@lru_cache(maxsize=1)
def sensitivity_table():
    table = []
    # Search a grid of independent worlds, in increments of 10. No pilot outcomes used.
    for q in (.05,.10):
        for delta in (.01,.02,.05):
            lo, hi = 0, 5
            while exact_scenario_power(10*hi,q,delta) < .80:
                lo,hi = hi,2*hi
            while hi-lo > 1:
                mid = (hi+lo)//2
                if exact_scenario_power(10*mid,q,delta) >= .80:
                    hi = mid
                else:
                    lo = mid
            table.append({"method":"independent_world_single_target_exact_McNemar_scenario",
                          "discordance":q, "target_pp":delta*100, "worlds_for_80pct":10*hi,
                          "power_at_suggested_n":exact_scenario_power(10*hi,q,delta),
                          "power_at_24":exact_scenario_power(24,q,delta)})
    return table


def analyze(study, main, forks):
    frames = {(r["world"],r["target"],r["context"],r["arm"]):r for r in main}
    expected = 4*len(study.worlds)*len(study.arms)
    expected_forks = 2*min(8,len(study.worlds))*len(study.arms)
    conditions, effects, planning, swaps, zero_bounds = [], [], [], [], []
    power_metrics = {}

    # Only combine conditions/arms on the same world; every contrast requires BOTH target requests.
    def contrast(name, terms, unit="pp", plan=False):
        # Each term is (coefficient, arm, context, field).
        vals = []
        for w in study.worlds:
            pair = []
            for target in (0,1):
                indices = [(coef,frames.get((w["id"],target,c,a)),m) for coef,a,c,m in terms]
                if all(row is not None and m in row for _,row,m in indices):
                    pair.append(sum(coef*float(row[m]) for coef,row,m in indices))
            if len(pair)==2:
                vals.append(sum(pair)/2*(100 if unit=="pp" else 1))
        effects.append(effect_row(name,vals,unit))
        if plan:
            power_metrics[name] = (vals,unit)

    def shift(arm, metric):
        return [(1,arm,"C1",metric),(-1,arm,"C0",metric)]

    def minus(terms):
        return [(-coef,a,c,m) for coef,a,c,m in terms]

    summary_metrics = ("first_lookup_gold","first_lookup_other","first_valid_lookup",
         "attempted_conflict_reservation_item","accepted_conflict_reservation_item",
         "any_order_conflict_value","accepted_conflict_value","strict_task_success",
         "no_accepted_order","format_failure","tool_error","prompt_budget_hit","action_budget_hit","model_calls","output_tokens","generation_seconds",
         "gold_logp","other_logp","margin")
    for arm in study.arms:
        for ctx in ("C0","C1"):
            rows = [r for r in main if r["arm"]==arm and r["context"]==ctx]
            if rows:
                summary = {"arm":arm,"context":ctx,"episodes":len(rows)}
                fields = list(summary_metrics) + (["direct_valid","direct_correct","direct_other"] if arm=="none" else [])
                for m in fields:
                    summary[m+"_mean"] = float(np.mean([r[m] for r in rows]))
                    if isinstance(rows[0][m],bool):
                        summary[m+"_count"] = sum(r[m] for r in rows)
                conditions.append(summary)
        contrast("mechanism_drop_C0_minus_C1/"+arm, minus(shift(arm,"margin")), "log_odds",plan=arm=="none")
        contrast("wrong_reservation_C1_minus_C0/"+arm,shift(arm,"accepted_conflict_reservation_item"),
                 plan=arm=="none")
        contrast("first_wrong_lookup_C1_minus_C0/"+arm,shift(arm,"first_lookup_other"))
        contrast("wrong_accepted_order_C1_minus_C0/"+arm,shift(arm,"accepted_conflict_value"))
        pairs = [[frames[(w["id"],t,"C1",arm)] for t in (0,1)] for w in study.worlds
                 if all((w["id"],t,"C1",arm) in frames for t in (0,1))]
        if pairs:
            n = len(pairs)
            swaps.append({"arm":arm,"complete_worlds":n,"both_targets_strict":sum(all(r["strict_task_success"] for r in pair) for pair in pairs),
                          "both_first_lookup_gold":sum(all(r["first_lookup_gold"] for r in pair) for pair in pairs)})
            k = sum(any(r["accepted_conflict_reservation_item"] for r in pair) for pair in pairs)
            from scipy.stats import beta
            upper = 1.0 if k==n else float(beta.ppf(.95,k+1,n-k))
            zero_bounds.append({"arm":arm,"event":"one_or_both_targets_wrong_reservation_under_C1",
                                "worlds":n,"worlds_with_event":k,"upper_one_sided_95":upper})
    for arm in ("none", *[a for a in study.arms if a.startswith("random")]):
        contrast("mechanism_drop_reduction_"+arm+"_minus_global80",
                 minus(shift(arm,"margin"))+shift("global80","margin"),"log_odds",plan=arm=="none")
        contrast("C1_margin_global80_minus_"+arm,[(1,"global80","C1","margin"),(-1,arm,"C1","margin")],"log_odds")
        contrast("wrong_reservation_context_effect_"+arm+"_minus_global80",
                 shift(arm,"accepted_conflict_reservation_item")+minus(shift("global80","accepted_conflict_reservation_item")),
                 plan=arm=="none")
    contrast("C0_strict_success_global80_minus_none",[(1,"global80","C0","strict_task_success"),(-1,"none","C0","strict_task_success")])
    contrast("direct_first_choice_C1_minus_C0",shift("none","direct_other"))
    contrast("direct_minus_actor_context_effect_on_first_SKU",
             shift("none","direct_other")+minus(shift("none","first_lookup_other")),plan=True)
    # Positive gap means a larger change in wrong-SKU selections for the direct protocol.
    # Invalid outputs remain in denominators. Separately report direct_valid / first_valid_lookup.
    fork_pairs = {(r["world"],r["arm"],r["stocked"]):r for r in forks}
    fork_report = []
    for arm in study.arms:
        pairs = [[fork_pairs[(w["id"],arm,s)] for s in (True,False)] for w in study.worlds[:8]
                 if all((w["id"],arm,s) in fork_pairs for s in (True,False))]
        if pairs:
            fork_report.append({"arm":arm,"paired_worlds":len(pairs),
                 "stocked_correct":sum(p[0]["strict_task_success"] for p in pairs),
                 "out_of_stock_correct":sum(p[1]["strict_task_success"] for p in pairs),
                 "both_stock_branches_correct":sum(all(r["strict_task_success"] for r in p) for p in pairs)})
    for name,(vals,unit) in power_metrics.items():
        if study.worlds[0]["split"]!="pilot":
            continue
        n=len(vals)
        informative=sum(v!=0 for v in vals)
        # A small pilot can estimate variance very poorly. Never produce n=0 or promote a partial pilot.
        ready = n==len(study.worlds) and n>=24 and informative>=6 and np.std(vals,ddof=1)>0
        for target in ((1,2,5) if unit=="pp" else (.25,.5,1.0)):
            size = None
            if ready:
                z = NormalDist().inv_cdf(.975)+NormalDist().inv_cdf(.80)
                size = max(2,math.ceil((z*np.std(vals,ddof=1)/target)**2))
            planning.append({"method":"pilot_cluster_SD_normal_approximation","contrast":name,"worlds":n,
                             "nonzero_net_worlds":informative,"target_effect":target,"effect_unit":unit,
                             "target_pp":target if unit=="pp" else None,"worlds_for_80pct":size,
                             "status":"rough planning only" if ready else "insufficient pilot information"})
    planning.extend(sensitivity_table())
    complete = len(main)==expected and len(forks)==expected_forks
    return {"status":"COMPLETE" if complete else "PARTIAL","independent_worlds_planned":len(study.worlds),
            "main_units":[len(main),expected],"fixed_history_units":[len(forks),expected_forks],
            "primary_mechanism":"C0-C1 paired whole-SKU log odds drop; compare global80 with none and matched random",
            "primary_action":"tool-accepted reservation of other request SKU, one indicator per autonomous episode",
            "conditions":conditions,"effects":effects,"target_swap":swaps,"stock_fork":fork_report,
            "world_event_bounds":zero_bounds,"planning":planning,
            "notes":["Intervals resample whole worlds (3000 draws); constant samples have no bootstrap CI. Rare-event CIs are descriptive.",
                     "Formal sample-size scenarios concern one prespecified target per world; they are not power guarantees for the cluster interaction.",
                     "Stage planning uses full paired-world SD and an external target difference, not observed power. Small or sparse pilots leave N blank.",
                     "Incomplete contrasts use only fully paired worlds. All observed attempts, invalid outputs and denominators remain in main.csv.",
                     "Direct/agent comparison concerns the first SKU under two different prompts using the same weights. Stock forks have a forced first lookup."]}
'''


In [ ]:
V61_ALIGNED_SOURCE = r'''"""v6.1 conditions and interventions in the frozen v6 namespace."""
from collections import Counter, defaultdict
from types import SimpleNamespace


CONDITIONS = ("C0", "Clex", "C1")
ITEM_PREFIX = '{"item":"sku-'
v6_records_for = records_for


def records_for(world, target, condition):
    if condition != "Clex":
        return v6_records_for(world, target, condition)
    records = v6_records_for(world, target, "C1")
    other_id = world["requests"][1-target]
    next(r for r in records if r["entity"] == other_id)["attribute"] = "preview_item"
    return records


def decision_tokens(backend, gold, other):
    """Tokenize output separately from the prompt, exactly as autoregressive decoding starts."""
    enc = lambda s: list(backend.tokenizer.encode(s, add_special_tokens=False))
    stem = enc(ITEM_PREFIX)
    texts = ['{"item":"' + sku for sku in (gold, other)]
    ids = [enc(text) for text in texts]
    if len(ids[0]) != len(ids[1]) or any(seq[:len(stem)] != stem for seq in ids):
        raise ValueError("Unmatched SKU tokenization or retokenized decision stem")
    for text, seq in zip(texts, ids):
        if backend.tokenizer.decode(seq, skip_special_tokens=False) != text:
            raise ValueError("Output token round trip failed")
    split = next((i for i,(a,b) in enumerate(zip(*ids)) if a != b), None)
    if split is None or split < len(stem):
        raise ValueError("Expected distinct numeric SKU values")
    return stem, ids, split


def prompts_for(world, target, condition):
    case = case_for(world, target)
    records = records_for(world, target, condition)
    return case, records, {"actor": actor_prompt(case, records, []),
                           "direct": direct_prompt(world, target, condition)}


def audit_aligned_inputs(backend, worlds, prior):
    max_tokens = 0
    for w in worlds:
        for t in (0, 1):
            gold, other = w["items"][t], w["items"][1-t]
            _, candidates, _ = decision_tokens(backend, gold, other)
            texts = {}
            for c in CONDITIONS:
                case, recs, texts[c] = prompts_for(w,t,c)
                if c != "Clex":
                    prev = prior[(w["id"],t,c)]
                    if (prev["actor"]["events"][0]["prompt"] != texts[c]["actor"] or
                            prev["direct"]["prompt"] != texts[c]["direct"] or
                            prev["records"] != recs or prev["actor"]["initial_state"] != case["initial_env"]):
                        raise ValueError("v6 prompt, records or environment changed: "+w["id"])
                for prompt in texts[c].values():
                    if (prompt.count(gold),prompt.count(other)) != (1,int(c!="C0")):
                        raise ValueError("Unexpected SKU exposure: "+w["id"])
                    max_tokens = max(max_tokens,len(backend.encode(prompt))+len(candidates[0]))
            for role in ("actor","direct"):
                neutral, exposure, relevant = [texts[c][role] for c in CONDITIONS]
                if exposure.count('"attribute":"preview_item"') != 1 or exposure.replace(
                        '"attribute":"preview_item"', '"attribute":"preferred_item"') != relevant:
                    raise ValueError("Clex/C1 must differ only in the other request's relation")
                seq = [backend.encode(x) for x in (neutral,exposure,relevant)]
                if len(set(map(len,seq))) != 1 or sum(a!=b for a,b in zip(seq[1],seq[2])) != 1:
                    raise ValueError("Clex/C1 must substitute exactly one token at the same position")
    if max_tokens > backend.cfg["max_prompt_tokens"]:
        raise ValueError("Initial input exceeds original context budget")
    return {"worlds":len(worlds),"max_tokens_including_candidate":max_tokens,
            "old_prompt_pairs_checked":len(prior), "equal_length_and_position":True}


def local_arms(ranking):
    top = [tuple(h) for h in ranking[:10]]
    if len(set(top)) != 10 or len(set(map(tuple,ranking))) != 80:
        raise ValueError("Expected the original ordered 80-head ranking")
    arms = {"none":[], "top10":top}
    for draw in range(3):
        rng = random.Random(61401 + draw)
        chosen = []
        for layer,n in sorted(Counter(l for l,h in top).items()):
            pool = [h for h in range(32) if (layer,h) not in set(map(tuple,ranking))]
            chosen.extend((layer,h) for h in rng.sample(pool,n))
        arms[f"random10_{draw+1}"] = chosen
    return arms


def tl_cache(backend):
    from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
    return HookedTransformerKeyValueCache.init_cache(backend.model.cfg, backend.device, batch_size=1)


def aligned_score(backend, prompt, gold, other, heads=(), cache_factory=tl_cache):
    """Original one-sequence cached decoder. Score only SKU digits/hyphens, never JSON punctuation."""
    stem, choices, split = decision_tokens(backend,gold,other)
    prompt_tokens = backend.tokens(prompt)
    if prompt_tokens.shape[1] + len(choices[0]) > backend.cfg["max_prompt_tokens"]:
        raise ValueError("Probe exceeds input budget")
    scores = []
    key_prefix = choices[0][:split]
    observed_prefix = backend.tokenizer.decode(key_prefix, skip_special_tokens=False)
    info = {"prefix_text":observed_prefix, "decision_output_offset":split,
            "shared_output_ids":key_prefix, "sku_tokens":len(choices[0])-len(stem),
            "gold_decision_id":choices[0][split],"other_decision_id":choices[1][split]}
    with torch.inference_mode():
        for branch, out_ids in enumerate(choices):
            cache = cache_factory(backend)
            step_tokens = prompt_tokens
            total = 0.0
            mismatch = None
            for i, token in enumerate(out_ids):
                # i is the index of the token predicted by THIS forward pass.
                hooks = ablation_hooks(backend.model, heads) if heads and i==split else []
                if hooks and step_tokens.shape[1] != 1:
                    raise RuntimeError("A local mask may touch only one query position")
                with backend.model.hooks(fwd_hooks=hooks):
                    logits = backend.model(step_tokens,past_kv_cache=cache,return_type="logits")[0,-1].float()
                if branch == 0 and i <= split:
                    argmax = int(logits.argmax().item())
                    if i < split and argmax != token and mismatch is None:
                        mismatch = i
                    if i == split:
                        info.update(first_margin=float((logits[out_ids[i]]-logits[choices[1][i]]).item()),
                                    greedy_text=backend.tokenizer.decode(key_prefix+[argmax],skip_special_tokens=False),
                                    greedy_decision_id=argmax, replay_prefix_match=mismatch is None,
                                    first_prefix_mismatch=mismatch)
                if i >= len(stem):
                    total += float(logits.log_softmax(-1)[token].item())
                step_tokens = torch.tensor([[token]], device=backend.device, dtype=torch.long)
            scores.append(total)
            del cache
    return dict(info, gold_sku_logp=scores[0], other_sku_logp=scores[1], sku_margin=scores[0]-scores[1])


class FirstActionMask:
    """Delegate generation, stopping and tool execution to v6. Arm one cached forward only."""
    def __init__(self, backend, prompt, common_ids, heads):
        self.backend, self.prompt, self.common, self.heads = backend,prompt,common_ids,heads
        self.cfg, self.encode = backend.cfg,backend.encode
        self.fired = False
        self.used = False

    def generate(self, prompt, role, heads, use_cache=True):
        if self.used:
            return self.backend.generate(prompt,role=role,heads=heads,use_cache=use_cache)
        if prompt!=self.prompt or not self.common or heads or not use_cache or role!="actor":
            raise ValueError("Local intervention requires the original first free Actor call")
        self.used = True
        state = SimpleNamespace(prefilled=False, offset=0, matching=True, active=False)
        model = self.backend.model
        layer_hits = Counter()
        frozen_prompt = self.backend.tokens(prompt)

        def before(module, args):
            tokens = args[0]
            state.active = False
            if not state.prefilled:
                if not torch.equal(tokens,frozen_prompt):
                    raise RuntimeError("Unexpected prefill in local decode")
                state.prefilled = True
            elif state.matching and state.offset < len(self.common):
                if tuple(tokens.shape) != (1,1):
                    raise RuntimeError("Expected one cached decode token")
                state.matching = int(tokens.item()) == self.common[state.offset]
                state.offset += 1
                state.active = state.matching and state.offset == len(self.common)
                if state.active:
                    self.fired = True

        def gated(name, fn):
            def apply(value, hook):
                if not state.active:
                    return value
                if value.shape[1] != 1:
                    raise RuntimeError("Unscoped position mask")
                layer_hits[name] += 1
                return fn(value,hook)
            return apply

        hooks = [(name,gated(name,fn)) for name,fn in ablation_hooks(model,self.heads)]
        handle = model.register_forward_pre_hook(before)
        try:
            with model.hooks(fwd_hooks=hooks):
                out = self.backend.generate(prompt,role=role,heads=(),use_cache=True)
        finally:
            handle.remove()
        if not state.prefilled or any(layer_hits[name] != int(self.fired) for name,_ in hooks):
            raise RuntimeError("Free-action intervention did not have its declared coverage")
        return out
'''


In [ ]:
V63_CONFIRM_SOURCE = r'''"""Confirmatory runner, using the frozen v6 model, environment and v6.1 SKU scorer."""

from collections import Counter

FORK_CONTEXTS = ("Clex", "C1")
FIRST_LOOKUPS = ("gold", "foil")
CHECK_KEYS = ("checks/inputs", "checks/development-smoke")


def audit_confirm_inputs(backend, inputs):
    worlds, targets = inputs["worlds"], inputs["fork_target_by_world"]
    if worlds != make_worlds("confirm", 96):
        raise ValueError(" The confirmation set is different from the freeze generator. ")
    strata, fork_strata = Counter(), Counter()
    comparisons = longest = 0
    for w in worlds:
        stratum = (w["template"], w["reverse"])
        strata[stratum] += 1
        if w["id"] in targets:
            fork_strata[stratum + (targets[w["id"]],)] += 1
        for t in (0, 1):
            gold, foil = w["items"][t], w["items"][1-t]
            _, candidates, _ = decision_tokens(backend, gold, foil)
            prompts = [prompts_for(w, t, c)[2] for c in CONDITIONS]
            for role in ("actor", "direct"):
                base, lex, rel = [p[role] for p in prompts]
                if ([p.count(gold) for p in (base, lex, rel)] != [1, 1, 1]
                        or [p.count(foil) for p in (base, lex, rel)] != [0, 1, 1]
                        or lex.count('"attribute":"preview_item"') != 1
                        or lex.replace('"attribute":"preview_item"', '"attribute":"preferred_item"') != rel):
                    raise ValueError(" Unsatisfactory comparison of scheduled exposure: " + w["id"])
                a, b, c = [backend.encode(p) for p in (base, lex, rel)]
                if not (len(a) == len(b) == len(c) and sum(x != y for x, y in zip(b, c)) == 1):
                    raise ValueError(" Relationship labels failed to replace only one in the same location  token：" + w["id"])
                longest = max(longest, len(c) + len(candidates[0]))
                comparisons += 1
    if (len(targets) != 48 or sorted(strata.values()) != [24]*4
            or sorted(fork_strata.values()) != [6]*8 or comparisons != 384
            or longest > backend.cfg["max_prompt_tokens"]):
        raise ValueError(" Unexpected confirmation balance, context length, or paired-sample count ")
    return dict(worlds=96, fork_worlds=48, matched_prompt_pairs=comparisons,
                max_initial_tokens_including_candidate=longest)


def natural_cells(w):
    for t in (0, 1):
        for context in CONDITIONS:
            for role in (("actor",) if context == "C0" else ("actor", "direct")):
                yield t, context, role, f"{w['id']}/{t}-{context}-{role}"


def fork_cells(w, targets):
    if w["id"] in targets:
        t = targets[w["id"]]
        for context in FORK_CONTEXTS:
            for lookup in FIRST_LOOKUPS:
                yield t, context, lookup, f"fork/{w['id']}/{t}-{context}-{lookup}"


def checkpoint_keys(study):
    yield from CHECK_KEYS
    for w in study.worlds:
        for _, _, _, suffix in natural_cells(w):
            yield "natural/" + suffix
            yield "probe/" + suffix
        for _, _, _, key in fork_cells(w, study.fork_targets):
            yield key


def run_lookup_branch(backend, w, t, context, lookup):
    case = case_for(w, t)
    if lookup not in FIRST_LOOKUPS:
        raise ValueError("Unknown lookup assignment")
    sku = w["items"][t if lookup == "gold" else 1-t]
    output = run_actor(backend, case, records_for(w, t, context), [],
                       forced_check=True, first_lookup_item=sku)
    first, *rest = output["events"]
    stock = case["initial_env"]["stock"]
    if (first["origin"] != "fixed_prefix" or "generation" in first
            or first["action"] != {"tool":"check_stock", "item":sku}
            or first["observation"] != {"ok":True, "item":sku, "available":stock[sku]}
            or stock[w["items"][t]] != stock[w["items"][1-t]] or stock[sku] <= 0
            or any(e["origin"] != "model" for e in rest)):
        raise RuntimeError(" Initial implementation log or follow-up autonomous action source does not match ")
    return output


def natural_row(w, t, context, role, output, probe):
    row = dict(world=w["id"], target=t, template=w["template"],
               target_position=(1-t if w["reverse"] else t)+1, context=context, role=role)
    if not all(math.isfinite(probe[k]) for k in ("first_margin","sku_margin","gold_sku_logp","other_sku_logp")):
        raise ValueError("Non-finite SKU score")
    row.update(probe)
    if role == "actor":
        first = output["events"][0] if output["events"] else {}
        raw = first.get("generation", {}).get("text", "")
        m = output["metrics"]
        row.update(m, first_gold=m["first_lookup_gold"], first_other=m["first_lookup_other"],
                   first_valid=m["first_valid_lookup"], terminal=output["terminal"])
        row.update(action_outcomes(w, t, output))
    else:
        raw = output["generation"]["text"]
        row.update(first_gold=output["correct"], first_other=output["other"],
                   first_valid=output["valid"], schema_valid=output["schema_valid"])
    row["text_prefix_match"] = raw.startswith(probe["prefix_text"])
    row["on_path"] = row["text_prefix_match"] and probe["replay_prefix_match"]
    row["decision_agrees_with_trace"] = row["on_path"] and raw.startswith(probe["greedy_text"])
    return row


def action_outcomes(w, t, out):
    gold, foil = w["items"][t], w["items"][1-t]
    orders = out["final_state"]["orders"]
    if len(orders) > 1:
        raise RuntimeError(" The environment should be terminated immediately after the successful billing. ")
    sku = orders[0]["item"] if orders else None
    later_checks = [e["action"].get("item") for e in out["events"][1:]
                    if e["origin"] == "model" and (e["action"] or {}).get("tool") == "check_stock"
                    and (e["observation"] or {}).get("ok")]
    return dict(final_item=sku, final_gold_sku=sku == gold, final_foil_sku=sku == foil,
                final_fallback_sku=sku == w["fallback"], final_other_sku=sku not in (None,gold,foil,w["fallback"]),
                final_no_order=not orders, gold_lookup_after_first=gold in later_checks,
                foil_lookup_after_first=foil in later_checks)


def fork_row(w, t, context, lookup, out):
    return dict(world=w["id"], target=t, template=w["template"],
                target_position=(1-t if w["reverse"] else t)+1,
                context=context, lookup=lookup, forced_item=out["events"][0]["action"]["item"],
                terminal=out["terminal"], model_steps=sum(e["origin"] == "model" for e in out["events"]),
                **out["metrics"], **action_outcomes(w,t,out))


class ConfirmStudy(Study):
    def __init__(self, root, backend, inputs, identity, verify, scorer=aligned_score):
        self.fork_targets = inputs["fork_target_by_world"]
        super().__init__(root, backend, inputs["worlds"], [], {"none":[]},
                         dict(identity, fork_target_by_world=self.fork_targets), verify, scorer)
        self.inputs = inputs

    def unit(self, key, fn=None):
        path = self.root/(key+".json")
        if fn is not None and path.is_file():
            try:
                return super().unit(key)
            except (KeyError, TypeError, ValueError, RuntimeError) as error:
                # A lost Drive write may leave a truncated final file. Preserve it outside the ZIP.
                backup = path.with_name(path.name+f".invalid-{time.time_ns()}")
                os.replace(path,backup)
                print(" Damaged check points have been retained and this module has been restated: ",key,type(error).__name__,flush=True)
        return super().unit(key,fn)

    def behavior(self, w, t, c, role):
        if role == "actor":
            return run_actor(self.backend, case_for(w,t), records_for(w,t,c), [])
        return direct_choice(self.backend, w, t, c)

    def score(self, w, t, c, role):
        return self.scorer(self.backend, prompts_for(w,t,c)[2][role], w["items"][t], w["items"][1-t], [])

    def prepare_confirm(self):
        self.check()
        audit = self.unit(CHECK_KEYS[0], lambda: audit_confirm_inputs(self.backend, self.inputs))
        def smoke():
            w, t, c = make_worlds("pilot", 1)[0], 0, "Clex"
            rows = [natural_row(w,t,c,role,self.behavior(w,t,c,role),self.score(w,t,c,role))
                    for role in ("actor","direct")]
            for row in rows:
                if not (row["first_valid"] and row["on_path"] and row["decision_agrees_with_trace"]):
                    raise RuntimeError(" Autonomous output or probe replay for the development of samples is not aligned, retention  runtime")
            forks = {a: run_lookup_branch(self.backend,w,t,c,a) for a in FIRST_LOOKUPS}
            return dict(world=w["id"], context=c, target=t, rows=rows, forks=forks, passed=True)
        readiness = self.unit(CHECK_KEYS[1], smoke)
        if not readiness["passed"]:
            raise RuntimeError(" Development of sample pre-screening failed ")
        print(" Enter and develop sample pre-screening through ", audit, flush=True)

    def run_confirm(self, limit=None):
        self.check()
        if limit is not None and (type(limit) is not int or limit < 1):
            raise ValueError("WORLDS_PER_CALL  Must be a positive or integer or  None")
        if not self.unit(CHECK_KEYS[0]) or not (self.unit(CHECK_KEYS[1]) or {}).get("passed"):
            raise RuntimeError(" Please run input and develop a sample pre-check first. ")
        advanced = 0
        for w in self.worlds:
            initial = self.new_units
            for t,c,role,suffix in natural_cells(w):
                for prefix,compute in (("natural",self.behavior),("probe",self.score)):
                    key = prefix + "/" + suffix
                    self.unit(key, lambda: compute(w,t,c,role))
            for t,c,lookup,key in fork_cells(w, self.fork_targets):
                self.unit(key, lambda: run_lookup_branch(self.backend,w,t,c,lookup))
            if self.new_units > initial:
                advanced += 1
                print(f"[{w['index']+1}/96] {w['id']}  Saved  |  Add this  {self.new_units-initial}  Checkpoint ",flush=True)
                if limit is not None and advanced >= limit:
                    break
        self.check()
'''


In [ ]:
V63_CONFIRM_REPORTS_SOURCE = r'''"""World-paired analyses and verified exports for the frozen v6.3 confirmation experiment."""

import shutil
from scipy.stats import binomtest

CONFIRM_COUNTS = {"actor_trajectories":576, "direct_answers":384, "actor_probes":576,
                  "direct_probes":384, "natural_scored":960, "fork_trajectories":192,
                  "natural_complete_worlds":96, "complete_fork_quartets":48}
OUTCOME_METRICS = ("strict_task_success","task_success","gold_lookup_after_first",
                   "attempted_conflict_reservation_item","accepted_conflict_reservation_item",
                   "any_order_conflict_value","accepted_conflict_value","final_gold_sku","final_foil_sku",
                   "final_fallback_sku","final_other_sku","final_no_order","format_failure","tool_error")
SCORE_METRICS = ("first_margin","sku_margin","gold_sku_logp","other_sku_logp")
REPORT_TABLES = ("natural-scores-actions","fork-actions","natural-conditions","fork-conditions",
                 "natural-effects","fork-effects","world-contrasts")


def group_summary(group, metrics, **labels):
    row = dict(labels, n=len(group))
    for metric in metrics:
        vals = [r[metric] for r in group]
        row[metric+"_mean"] = sum(vals)/len(vals) if vals else None
        if metric not in SCORE_METRICS:
            row[metric+"_count"] = sum(vals)
    return row


def natural_effects(worlds, rows):
    index = {(r["world"],r["target"],r["role"],r["context"]):r for r in rows}
    effects, details = [], []

    def paired(name, roles, metric, left, right, subset="all", template=None):
        values = []
        for w in worlds:
            if template is not None and w["template"] != template:
                continue
            block = [(t,sign,condition_sign,index.get((w["id"],t,role,ctx)))
                     for t in (0,1) for role,sign in roles for ctx,condition_sign in ((left,1),(right,-1))]
            if any(r is None or (subset == "on_path" and not r["on_path"]) for _,_,_,r in block):
                continue
            value = sum(s*d*r[metric] for _,s,d,r in block)/2
            if metric not in SCORE_METRICS:
                value *= 100
            values.append(value)
            details.append(dict(contrast=name, world=w["id"], metric=metric, subset=subset,
                                template_group=template if template is not None else "all", value=value))
        row = effect_row(name,values,"logit" if metric=="first_margin" else
                         "log_probability_difference" if metric in SCORE_METRICS else "percentage_points")
        effects.append(dict(row,metric=metric,subset=subset,template_group=template if template is not None else "all"))

    for role in ("actor","direct"):
        for metric in SCORE_METRICS:
            for subset in (("all","on_path") if metric in ("first_margin","sku_margin") else ("all",)):
                for tmpl in ((None,0,1) if metric=="first_margin" else (None,)):
                    paired(role+"/Clex-C1",[(role,1)],metric,"Clex","C1",subset,tmpl)
        metrics = ("first_other",) + (("accepted_conflict_reservation_item","final_foil_sku",
                                     "final_no_order","strict_task_success") if role=="actor" else ())
        for metric in metrics:
            paired(role+"/C1-Clex",[(role,1)],metric,"C1","Clex")
    for c in ("Clex","C1"):
        paired("actor/C0-"+c,[("actor",1)],"first_margin","C0",c)
    for subset in ("all","on_path"):
        paired("actor_minus_direct/relation_drop",[("actor",1),("direct",-1)],"first_margin","Clex","C1",subset)
    return effects, details


def fork_effects(worlds, targets, rows):
    index = {(r["world"],r["context"],r["lookup"]):r for r in rows}
    effects, details = [], []
    for metric in ("final_foil_sku","gold_lookup_after_first","final_gold_sku","strict_task_success",
                   "accepted_conflict_reservation_item","final_fallback_sku","final_no_order"):
        differences = {c:{} for c in FORK_CONTEXTS}
        for c in FORK_CONTEXTS:
            counts = Counter()
            label = "forced_foil_minus_gold/"+c
            for wid in targets:
                gold, foil = [index.get((wid,c,k)) for k in FIRST_LOOKUPS]
                if gold is None or foil is None:
                    continue
                g, f = bool(gold[metric]), bool(foil[metric])
                counts[g,f] += 1
                value = 100*(int(f)-int(g))
                differences[c][wid] = value
                details.append(dict(world=wid,target=targets[wid],contrast=label,metric=metric,value=value))
            foil_only, gold_only = counts[False,True], counts[True,False]
            discordant = foil_only + gold_only
            p = float(binomtest(foil_only, discordant, .5).pvalue) if discordant else None
            effects.append(dict(effect_row(label,list(differences[c].values()),"percentage_points"),
                                metric=metric, both=counts[True,True], neither=counts[False,False],
                                foil_start_only=foil_only, gold_start_only=gold_only,
                                discordant=discordant, exact_mcnemar_p=p))
        interaction = []
        label = "forced_foil_minus_gold/C1-minus-Clex"
        for w in worlds:
            wid = w["id"]
            if all(wid in differences[c] for c in FORK_CONTEXTS):
                v = differences["C1"][wid]-differences["Clex"][wid]
                interaction.append(v)
                details.append(dict(world=wid,target=targets[wid],contrast=label,metric=metric,value=v))
        effects.append(dict(effect_row(label,interaction,"percentage_points"),metric=metric))
    return effects, details


def read_checkpoint(z, available, study, key):
    path = key+".json"
    if path not in available:
        return None
    record = json.loads(z.read(path))
    if record["signature"] != stable_hash([study.signature,key]) or record["digest"] != stable_hash(record["result"]):
        raise ValueError(" Checkpoint validation failed: " + key)
    return record["result"]


def analyze_snapshot(study, z):
    names = set(z.namelist())
    load = lambda k: read_checkpoint(z,names,study,k)
    if not load(CHECK_KEYS[0]) or not (load(CHECK_KEYS[1]) or {}).get("passed"):
        raise ValueError(" Archival lack of pre-screened ")
    counts = dict.fromkeys(CONFIRM_COUNTS,0)
    natural, forks = [], []
    for w in study.worlds:
        before = len(natural)
        for t,c,role,suffix in natural_cells(w):
            out, q = load("natural/"+suffix), load("probe/"+suffix)
            counts["actor_trajectories" if role=="actor" else "direct_answers"] += out is not None
            counts[role+"_probes"] += q is not None
            if out is not None and q is not None:
                natural.append(natural_row(w,t,c,role,out,q))
        counts["natural_complete_worlds"] += len(natural)-before == 10
        before = len(forks)
        for t,c,lookup,key in fork_cells(w,study.fork_targets):
            out = load(key)
            if out is not None:
                row = fork_row(w,t,c,lookup,out)
                if row["forced_item"] != w["items"][t if lookup=="gold" else 1-t]:
                    raise ValueError(" Initial steps in archive  SKU  Unmatched with branch: " + key)
                forks.append(row)
        counts["complete_fork_quartets"] += len(forks)-before == 4
    counts["natural_scored"], counts["fork_trajectories"] = len(natural),len(forks)
    conditions, branches = [], []
    for role in ("actor","direct"):
        for c in (CONDITIONS if role=="actor" else FORK_CONTEXTS):
            group = [r for r in natural if (r["role"],r["context"])==(role,c)]
            metrics = SCORE_METRICS + ("on_path","decision_agrees_with_trace","first_valid","first_gold","first_other")
            conditions.append(group_summary(group,metrics+(OUTCOME_METRICS if role=="actor" else ()),role=role,context=c))
    for c in FORK_CONTEXTS:
        for lookup in FIRST_LOOKUPS:
            group = [r for r in forks if (r["lookup"],r["context"])==(lookup,c)]
            branches.append(group_summary(group,OUTCOME_METRICS,lookup=lookup,context=c))
    effects, world_values = natural_effects(study.worlds,natural)
    branch_effects, branch_values = fork_effects(study.worlds,study.fork_targets,forks)
    event_worlds = {}
    for r in natural:
        if (r["role"],r["context"]) == ("actor","C1"):
            event_worlds.setdefault(r["world"],[]).append(r["final_foil_sku"])
    valid = [any(events) for events in event_worlds.values() if len(events)==2]
    report = dict(status="COMPLETE" if counts==CONFIRM_COUNTS else "PARTIAL",scope="confirm96",
                  manifest_hash=study.signature,counts=counts,expected=CONFIRM_COUNTS,
                  primary=next(r for r in effects if (r["contrast"],r["metric"],r["subset"],r["template_group"])==
                               ("actor/Clex-C1","first_margin","all","all")),
                  key_behavior=next(r for r in branch_effects if (r["contrast"],r["metric"])==
                                    ("forced_foil_minus_gold/C1","final_foil_sku")),
                  label_interaction=next(r for r in branch_effects if (r["contrast"],r["metric"])==
                                         ("forced_foil_minus_gold/C1-minus-Clex","final_foil_sku")),
                  natural_C1_foil_event_worlds=dict(paired_worlds=len(valid),any_foil_order=sum(valid),
                    zero_event_upper95_one_sided=(1-.05**(1/len(valid))) if valid and not any(valid) else None),
                  natural_conditions=conditions,fork_conditions=branches,natural_effects=effects,fork_effects=branch_effects,
                  notes=["Only held-out worlds enter estimates. Averages of both targets are resampled by world (3000 draws).",
                         "Clex and C1 differ by one relation token. Direct has no orders. Both SKUs are in stock.",
                         "A first_margin is gold minus foil at the first divergent SKU token. On-path requires the observed and replayed prefix.",
                         "Raw natural behaviors without their paired probe can occur in PARTIAL. All failures remain in COMPLETE denominators.",
                         "B fixes only the first lookup. Its 192 continuations are separate from natural error rates.",
                         "Bootstrap CI absent for a constant sample; exact two-sided McNemar p absent without discordant pairs."])
    tables = dict(zip(REPORT_TABLES,(natural,forks,conditions,branches,effects,branch_effects,world_values+branch_values)))
    return report, tables


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as src:
        for buf in iter(lambda:src.read(1024*1024),b""):
            h.update(buf)
    return h.hexdigest()


def verify_confirm_zip(path, study, report, *, require_complete):
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        if len(names)!=len(set(names)) or z.testzip() is not None:
            raise IOError("ZIP  directory or directory of  CRC  Validation Failed ")
        if (json.loads(z.read("reports/report.json")) != report or report["manifest_hash"]!=study.signature
                or stable_hash(json.loads(z.read("manifest.json")))!=study.signature
                or hashlib.sha256(z.read("inputs/synthetic_confirm_inputs.json")).hexdigest()!=study.manifest["inputs_sha256"]
                or hashlib.sha256(z.read("inputs/protocol.md")).hexdigest()!=study.manifest["protocol_sha256"]):
            raise IOError("ZIP  Inconsistent reporting, mandate, agreement or operational identity ")
        if require_complete:
            expected = {k+".json" for k in checkpoint_keys(study)}
            expected.update("reports/"+name+".csv" for name in REPORT_TABLES)
            expected.update(("manifest.json","inputs/synthetic_confirm_inputs.json","inputs/protocol.md","reports/report.json"))
            if (report["status"]!="COMPLETE" or report["counts"]!=CONFIRM_COUNTS or set(names)!=expected):
                raise IOError("ZIP  Not all confirmed test sites or reports are included ")


def export_confirm(study, data_bytes, protocol_bytes):
    study.check()
    dest = study.root/"reports"
    dest.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        local = Path(tmp)/"snapshot.zip"
        with zipfile.ZipFile(local,"w",compression=zipfile.ZIP_DEFLATED) as z:
            z.write(study.root/"manifest.json","manifest.json")
            z.writestr("inputs/synthetic_confirm_inputs.json",data_bytes)
            z.writestr("inputs/protocol.md",protocol_bytes)
            for key in checkpoint_keys(study):
                path = study.root/(key+".json")
                if path.is_file():
                    z.write(path,key+".json")
        with zipfile.ZipFile(local) as z:
            report, tables = analyze_snapshot(study,z)
        with zipfile.ZipFile(local,"a",compression=zipfile.ZIP_DEFLATED) as z:
            for name, rows in tables.items():
                path = Path(tmp)/(name+".csv")
                write_csv(path,rows)
                z.write(path,"reports/"+path.name)
            z.writestr("reports/report.json",json.dumps(report,ensure_ascii=False,allow_nan=False))
        complete = report["status"]=="COMPLETE"
        verify_confirm_zip(local,study,report,require_complete=complete)
        archive = study.root.parent/("v6p3_confirm96_"+report["status"]+"_results.zip")
        pending = archive.with_suffix(".zip.pending")
        shutil.copyfile(local,pending)
        digest = file_sha(local)
        if file_sha(pending)!=digest:
            raise IOError("Drive  Copy  SHA-256  Inconsistencies, reservations  runtime")
        os.replace(pending,archive)
        if file_sha(archive)!=digest:
            raise IOError("Drive  Eventually.  ZIP  Reread check failed, keep  runtime")
        save_json(dest/"report.json",report)
    print("Drive ZIP",archive,"SHA-256",digest,flush=True)
    return report, archive, digest


def finish_confirm(study, result, auto_disconnect, mountpoint, is_mounted, flush_drive, release):
    if result is None:
        print(" No result yet generated  ZIP， Preserve  runtime。")
        return False
    report, path, digest = result
    if not auto_disconnect or report["status"]!="COMPLETE":
        print(" Current status ",report["status"]," Auto Disconnect ",auto_disconnect," Preserve  runtime。")
        return False
    mountpoint = Path(mountpoint).resolve(strict=True)
    target = Path(path).resolve(strict=True)
    if (not is_mounted(str(mountpoint)) or not target.is_relative_to((mountpoint/"MyDrive").resolve())
            or target.parent!=study.root.parent.resolve() or target.name!="v6p3_confirm96_COMPLETE_results.zip"):
        raise RuntimeError(" Not confirmed  ZIP  On Mounted Wheel  Drive  Contents, retention  runtime")
    if file_sha(target)!=digest:
        raise IOError(" Eventually.  ZIP  Unlike exported copies, keep  runtime")
    verify_confirm_zip(target,study,report,require_complete=True)
    print(" complete  ZIP  From  Drive  Read back. Synchronizing. Released after success.  GPU。",flush=True)
    flush_drive(timeout_ms=600_000)
    release()
    return True
'''


In [ ]:
V64_HEADS_AB_SOURCE = r'''"""Fixed v6.2 masks at two distinct free SKU decisions, with signed per-condition checkpoints."""

AB_MASKS = ("top80", "random80_1", "random80_2", "random80_3")
AB_ARMS = ("none", *AB_MASKS)
AB_CHECKS = ("checks/inputs", "checks/development")


def baseline_key(w, t, context, lookup=None):
    suffix = f"{t}-{context}" + (f"-{lookup}" if lookup else "")
    return f"{w['id']}/{suffix}"


def ab_cells(study, stage, w):
    if stage == "A":
        for t in (0, 1):
            for c in FORK_CONTEXTS:
                for arm in AB_MASKS:
                    yield t, c, None, arm, f"A/{baseline_key(w,t,c)}/{arm}"
    elif stage == "B":
        if w["id"] in study.fork_targets:
            t = study.fork_targets[w["id"]]
            for c in FORK_CONTEXTS:
                for lookup in FIRST_LOOKUPS:
                    for arm in AB_ARMS:
                        yield t, c, lookup, arm, f"B/{baseline_key(w,t,c,lookup)}/{arm}"
    else:
        raise ValueError("Unknown experiment stage: " + stage)


def lookup_event(w, t, lookup):
    case = case_for(w,t)
    if lookup not in FIRST_LOOKUPS:
        raise ValueError("Unknown fixed lookup")
    sku = w["items"][t if lookup == "gold" else 1-t]
    action = dict(tool="check_stock",item=sku)
    obs = OrderEnvironment(case["initial_env"]).step(action)
    if obs != dict(ok=True,item=sku,available=2):
        raise ValueError("The frozen lookup did not return the original actual stock")
    return dict(action=action,observation=obs,parse_error=None,origin="fixed_prefix",step=0)


def ab_prompt(w, t, context, lookup=None):
    history = [] if lookup is None else [lookup_event(w,t,lookup)]
    return actor_prompt(case_for(w,t),records_for(w,t,context),history)


def path_fields(event, q):
    text = (event or {}).get("generation",{}).get("text","")
    if any(not math.isfinite(q[k]) for k in SCORE_METRICS):
        raise ValueError("Non-finite probe score")
    observed = text.startswith(q["prefix_text"])
    on_path = observed and q["replay_prefix_match"]
    return dict(text_prefix_match=observed,on_path=on_path,
                decision_agrees_with_trace=on_path and text.startswith(q["greedy_text"]))


def event_signature(event):
    return (event["generation"]["text"],event["action"],event["observation"],event["parse_error"])


def audit_ab(study):
    ref, be = study.reference, study.backend
    m = ref["v63_manifest"]
    if (m["worlds"] != study.worlds or m["fork_target_by_world"] != study.fork_targets
            or set(ref["arms"]) != set(AB_MASKS)):
        raise ValueError("Archive/world/mask mismatch")
    tops = set(map(tuple,ref["arms"]["top80"]))
    for arm in AB_MASKS:
        heads = list(map(tuple,ref["arms"][arm]))
        if (len(heads)!=80 or len(set(heads))!=80
                or any(not (0<=l<32 and 0<=h<32) for l,h in heads)
                or Counter(l for l,h in heads)!=Counter(l for l,h in tops)
                or (arm!="top80" and tops.intersection(heads))):
            raise ValueError("Invalid frozen head set: "+arm)
    original = audit_confirm_inputs(be,study.inputs)
    keys_a, keys_b = set(), set()
    for w in study.worlds:
        for t in (0,1):
            for c in FORK_CONTEXTS:
                key = baseline_key(w,t,c)
                old = ref["A"][key]
                if old["event"]["prompt"]!=ab_prompt(w,t,c) or old["event"]["origin"]!="model":
                    raise ValueError("Different initial Actor prompt: "+key)
                keys_a.add(key)
        if w["id"] not in study.fork_targets:
            continue
        t = study.fork_targets[w["id"]]
        _, choices, _ = decision_tokens(be,w["items"][t],w["items"][1-t])
        for lookup in FIRST_LOOKUPS:
            tokens = []
            for c in FORK_CONTEXTS:
                key = baseline_key(w,t,c,lookup)
                old = ref["B"][key]
                p = ab_prompt(w,t,c,lookup)
                if (old["events"][0]!=lookup_event(w,t,lookup) or old["events"][1]["prompt"]!=p
                        or old["events"][1]["origin"]!="model"):
                    raise ValueError("Different forced history or post-lookup prompt: "+key)
                tokens.append(be.encode(p))
                keys_b.add(key)
            if (len(tokens[0])!=len(tokens[1]) or sum(a!=b for a,b in zip(*tokens))!=1
                    or len(tokens[0])+len(choices[0])>be.cfg["max_prompt_tokens"]):
                raise ValueError("Post-lookup decision is not a matched single-token pair")
    if (keys_a!=set(ref["A"]) or keys_b!=set(ref["B"]) or len(keys_a)!=384 or len(keys_b)!=192):
        raise ValueError("Incomplete archived no-mask baseline")
    return dict(original_inputs=original,reused_first_choices=len(keys_a),reused_forks=len(keys_b),
                frozen_masks={a:stable_hash(ref["arms"][a]) for a in AB_MASKS})


class HeadsABStudy(ConfirmStudy):
    def __init__(self, root, backend, inputs, reference, identity, verify, scorer=aligned_score):
        self.inputs, self.reference = inputs, reference
        self.fork_targets = inputs["fork_target_by_world"]
        Study.__init__(self,root,backend,inputs["worlds"],[],{"none":[],**reference["arms"]},
                       dict(identity,fork_target_by_world=self.fork_targets),verify,scorer)

    def masked_adapter(self, w, t, prompt, arm):
        _, ids, split = decision_tokens(self.backend,w["items"][t],w["items"][1-t])
        return FirstActionMask(self.backend,prompt,ids[0][:split],self.arms[arm])

    def compute(self, stage, w, t, context, lookup, arm):
        prompt = ab_prompt(w,t,context,lookup)
        gold, other = w["items"][t], w["items"][1-t]
        if arm == "none":
            if stage != "B" or lookup not in FIRST_LOOKUPS:
                raise ValueError("Only the missing no-mask post-lookup probe is recomputed")
            return dict(probe=self.scorer(self.backend,prompt,gold,other,[]),
                        origin="v6.3 behavior; new post-lookup probe",mask_fired=False)
        adapter = self.masked_adapter(w,t,prompt,arm)
        if stage == "A" and lookup is None:
            # Stop the original Actor loop immediately after its first real tool call.
            # The backend keeps its original 6-action configuration for B and later units.
            adapter.cfg = dict(self.backend.cfg,max_actions=1)
            out = run_actor(adapter,case_for(w,t),records_for(w,t,context),[])
            if len(out["events"])!=1:
                raise RuntimeError("Missing first free action in A")
            result = dict(event=out["events"][0])
        elif stage == "B" and lookup in FIRST_LOOKUPS:
            out = run_lookup_branch(adapter,w,t,context,lookup)
            if out["events"][0]!=lookup_event(w,t,lookup):
                raise RuntimeError("The prescribed lookup or its actual observation has changed")
            result = dict(output=out)
        else:
            raise ValueError("Unknown stage/lookup combination")
        if not adapter.used:
            raise RuntimeError("The first free Actor call was not reached")
        q = self.scorer(self.backend,prompt,gold,other,self.arms[arm])
        return dict(result,probe=q,mask_fired=adapter.fired,origin="v6.4 masked")

    def development(self):
        dev = self.reference["development"]
        checks = []
        for s in dev["A"]:
            w,t,c,arm = s["world"],s["target"],s["context"],s["arm"]
            new = self.compute("A",w,t,c,None,arm)
            if (event_signature(new["event"])!=event_signature(s["event"])
                    or new["mask_fired"]!=s["mask_fired"]
                    or any(abs(new["probe"][k]-s["probe"][k])>1e-4 for k in SCORE_METRICS)
                    or not path_fields(new["event"],new["probe"])["decision_agrees_with_trace"]):
                raise RuntimeError("v6.2 masked development replay failed; check the original Colab hardware and versions")
            checks.append([w["id"],c,arm])
        w,t,c = dev["world"],dev["target"],dev["context"]
        q0 = self.scorer(self.backend,ab_prompt(w,t,c),w["items"][t],w["items"][1-t],[])
        if any(not math.isfinite(q0[k]) or abs(q0[k]-dev["unmasked_probe"][k])>1e-4 for k in SCORE_METRICS):
            raise RuntimeError("v6.3 unmasked development score did not reproduce")
        for lookup in FIRST_LOOKUPS:
            old = dev["B"][lookup]
            replay = run_lookup_branch(self.backend,w,t,c,lookup)
            if ([event_signature(e) for e in old["events"][1:]]!=[event_signature(e) for e in replay["events"][1:]]
                    or old["events"][0]!=replay["events"][0] or old["final_state"]!=replay["final_state"]):
                raise RuntimeError("v6.3 post-lookup development replay failed")
            if lookup=="foil":
                new = self.compute("B",w,t,c,lookup,"top80")
                e = new["output"]["events"][1]
                if (not new["mask_fired"] or not path_fields(e,new["probe"])["decision_agrees_with_trace"]
                        or e["prompt"]!=old["events"][1]["prompt"]):
                    raise RuntimeError("Post-lookup mask/trace/probe alignment failed on the pilot")
        return dict(passed=True,masked_A=checks,unmasked_Both_lookups=True,masked_post_lookup=True)

    def prepare(self):
        self.check()
        audit = self.unit(AB_CHECKS[0],lambda:audit_ab(self))
        dev = self.unit(AB_CHECKS[1],self.development)
        if not dev["passed"]:
            raise RuntimeError("Development replay incomplete")
        print(" Checked history first step ",audit["reused_first_choices"]," trajectories, and historical controlled branches ",
              audit["reused_forks"]," Bars; intervention and tool replaying for the development of samples. ",flush=True)

    def run_stage(self, stage):
        self.check()
        if not self.unit(AB_CHECKS[0]) or not (self.unit(AB_CHECKS[1]) or {}).get("passed"):
            raise RuntimeError("Run the preflight cell first")
        selected = [w for w in self.worlds if stage=="A" or w["id"] in self.fork_targets]
        for index,w in enumerate(selected,1):
            before = self.new_units
            for t,c,lookup,arm,key in ab_cells(self,stage,w):
                self.unit(key,lambda:self.compute(stage,w,t,c,lookup,arm))
            if self.new_units!=before or index==len(selected):
                print(f"[{stage} {index}/{len(selected)}] {w['id']} |  Add  {self.new_units-before}  One condition. ",flush=True)
        self.check()
'''


In [ ]:
V69_PATCH68_SOURCE = r'''"""Single-position interchange on the frozen v6 Llama Actor and three-tool environment."""
from contextlib import contextmanager
from uuid import uuid4


SCOPE68 = "v6p8_identity_patch48"
DONORS68 = ("target", "history", "intent")
CHECKS68 = ("checks/inputs", "checks/smoke")


def heads_by_layer68(heads):
    layers = defaultdict(list)
    for layer, head in heads:
        layers[layer].append(head)
    return dict(layers)


def patch_hooks68(donor, heads):
    hooks = []
    for layer, indices in heads_by_layer68(heads).items():
        def replace(value, hook, layer=layer, indices=indices):
            source = donor[layer]
            if value.shape[:2] != (1, 1) or value.shape != source.shape or value.dtype != source.dtype or value.device != source.device:
                raise RuntimeError("Unmatched donor/recipient query-head activations")
            result = value.clone()
            result[:, :, indices, :] = source[:, :, indices, :]
            return result
        hooks.append((f"blocks.{layer}.attn.hook_z", replace))
    return hooks


def first_score68(logits, pair):
    scores = logits[0, -1, list(pair)].float().tolist()
    if not all(map(math.isfinite, scores)):
        raise ValueError("Non-finite logit")
    return dict(x=scores[0], y=scores[1], margin=scores[0]-scores[1],
                greedy=int(logits[0, -1].argmax().item()))


def prefix68(be, prompt, prefix, pair, *, capture=(), donor=None, heads=()):
    """Mirror the original cache and prefill, teacher-forcing only the public JSON stem."""
    if not prefix or bool(donor is not None) != bool(heads) or (capture and heads):
        raise ValueError("Invalid prefix/capture/patch design")
    tokens = be.tokens(prompt)
    if tokens.shape[1]+len(prefix) > be.cfg["max_prompt_tokens"]:
        raise ValueError("Probe prompt budget")
    activations, hits = {}, Counter()
    hooks = patch_hooks68(donor, heads) if heads else []
    for layer in sorted(capture):
        def record(value, hook, layer=layer):
            if value.shape[:2] != (1, 1) or layer in activations:
                raise RuntimeError("Donor capture escaped the specified position")
            activations[layer] = value.detach().clone()
            return value
        hooks.append((f"blocks.{layer}.attn.hook_z", record))
    def counted(name, fn):
        def call(value, hook):
            hits[name] += 1
            return fn(value, hook)
        return call
    tracked = [(name, counted(name, fn)) for name, fn in hooks]
    cache = tl_cache(be)
    with torch.inference_mode():
        logits = be.model(tokens, past_kv_cache=cache, return_type="logits")
        on_path = True
        for i, tok in enumerate(prefix):
            on_path = on_path and int(logits[0, -1].argmax().item()) == tok
            step = torch.tensor([[tok]], device=be.device, dtype=torch.long)
            with be.model.hooks(fwd_hooks=tracked if i == len(prefix)-1 else []):
                logits = be.model(step, past_kv_cache=cache, return_type="logits")
        score = first_score68(logits, pair)
    if any(hits[name] != 1 for name, _ in tracked):
        raise RuntimeError("Probe did not reach all declared layers exactly once")
    return score, on_path, activations


class LivePatch68:
    """Instrument the first free generation only. The original Actor owns every action."""
    def __init__(self, be, prompt, prefix, pair, donor, heads):
        self.be, self.prompt, self.prefix, self.pair = be, prompt, prefix, pair
        self.donor, self.heads = donor, heads
        self.cfg, self.encode = be.cfg, be.encode
        self.used, self.score = False, None

    def generate(self, prompt, role, heads, use_cache=True):
        if self.used:
            return self.be.generate(prompt, role=role, heads=heads, use_cache=use_cache)
        if prompt != self.prompt or role != "actor" or heads or not use_cache:
            raise ValueError("Expected first free Actor call after the fixed real lookup")
        self.used = True
        model = self.be.model
        initial = self.be.tokens(prompt)
        offset, matching, active = -1, True, False
        hits = Counter()

        def before(module, args):
            nonlocal offset, matching, active
            tokens = args[0]
            active = False
            if offset < 0:
                if not torch.equal(initial, tokens):
                    raise RuntimeError("Unexpected first prefill")
                offset = 0
            elif matching and offset < len(self.prefix):
                if tuple(tokens.shape) != (1, 1):
                    raise RuntimeError("Expected a single cached query")
                matching = int(tokens.item()) == self.prefix[offset]
                offset += 1
                active = matching and offset == len(self.prefix)

        def after(module, args, out):
            if active:
                if self.score is not None:
                    raise RuntimeError("Position fired twice")
                self.score = first_score68(out, self.pair)

        def gate(name, fn):
            def call(value, hook):
                if not active:
                    return value
                hits[name] += 1
                return fn(value, hook)
            return call
        hooks = [(name, gate(name, fn)) for name, fn in patch_hooks68(self.donor, self.heads)]
        pre = model.register_forward_pre_hook(before)
        try:
            post = model.register_forward_hook(after)
            try:
                with model.hooks(fwd_hooks=hooks):
                    out = self.be.generate(prompt, role=role, heads=(), use_cache=True)
            finally:
                post.remove()
        finally:
            pre.remove()
        if offset < 0 or any(hits[n] != int(self.score is not None) for n, _ in hooks):
            raise RuntimeError("Free-generation hook coverage mismatch")
        return out


def prompt68(w, t, h):
    if h is None:
        return ab_prompt(w, t, "C1")
    if h in (0, 1):
        return ab_prompt(w, t, "C1", "gold" if h == t else "foil")
    if h != "Z":
        raise ValueError("Unknown lookup identity")
    case = case_for(w, t)
    action = dict(tool="check_stock", item=w["fallback"])
    obs = OrderEnvironment(case["initial_env"]).step(action)
    if obs != dict(ok=True, item=w["fallback"], available=2):
        raise ValueError("Z history is not the actual equal-stock response")
    ev = dict(action=action, observation=obs, parse_error=None, origin="fixed_prefix", step=0)
    return actor_prompt(case, records_for(w, t, "C1"), [ev])


def donor_state68(t, h, kind):
    if kind in ("none", "self"):
        return t, h
    if kind == "target":
        return 1-t, h
    if kind == "history" and h in (0, 1):
        return t, 1-h
    if kind == "intent" and h in (0, 1):
        return t, None
    raise ValueError("Invalid donor intervention")


def cells68(w, t):
    for start in ("gold", "foil"):
        h = t if start == "gold" else 1-t
        for kind, arm in [("none","none"), ("self","top80"), *((d,a) for d in DONORS68 for a in AB_MASKS)]:
            yield h, kind, arm, f"branches/{w['id']}/{start}/{kind}-{arm}"
    for kind, arm in [("none","none"), *(("target",a) for a in AB_MASKS)]:
        yield "Z", kind, arm, f"neutral/{w['id']}/{kind}-{arm}"


class WorldPatch68:
    def __init__(self, study, w, t):
        self.study, self.w, self.t, self.be = study, w, t, study.backend
        stem, ids, split = decision_tokens(self.be, w["items"][t], w["items"][1-t])
        if split != len(stem):
            raise ValueError("Expected first numeric group immediately after public JSON stem")
        self.prefix, self.pair = stem, (ids[0][split], ids[1][split])
        self.layers = sorted(heads_by_layer68(study.arms["top80"]))
        self.donors = {}

    def clean(self, target, h):
        key = (target, h)
        if key not in self.donors:
            self.donors[key] = prefix68(self.be, prompt68(self.w, target, h), self.prefix, self.pair,
                                        capture=self.layers)
        return self.donors[key]

    def cell(self, h, kind, arm):
        w, t = self.w, self.t
        source = donor_state68(t, h, kind)
        clean_score, on_path, _ = self.clean(t, h)
        donor_score, donor_natural, vectors = self.clean(*source)
        donor = vectors if arm != "none" else None
        heads = self.study.arms[arm]
        prompt = prompt68(w, t, h)
        row = dict(world=w["id"], target=t, history=h, kind=kind, arm=arm,
                   donor_target=source[0], donor_history=source[1], prompt_hash=stable_hash(prompt),
                   donor_prompt_hash=stable_hash(prompt68(w, *source)), donor_clean_score=donor_score,
                   donor_prefix_natural=donor_natural, prefix_ids=self.prefix, xy_ids=list(self.pair))
        if h == "Z":
            score = clean_score
            if heads:
                score, prefix_natural, _ = prefix68(self.be, prompt, self.prefix, self.pair, donor=donor, heads=heads)
                if prefix_natural != on_path:
                    raise RuntimeError("Patch changed pre-intervention path")
            return dict(row, score=score, prefix_natural=on_path, measure="prefix_only")
        actor = LivePatch68(self.be, prompt, self.prefix, self.pair, donor, heads)
        output = run_lookup_branch(actor, w, t, "C1", "gold" if h == t else "foil")
        if not actor.used or (actor.score is not None) != on_path:
            raise RuntimeError("Replay and live Actor disagree on the public JSON prefix")
        if on_path:
            score = actor.score
        elif heads:
            score, _, _ = prefix68(self.be, prompt, self.prefix, self.pair, donor=donor, heads=heads)
        else:
            score = clean_score
        if kind in ("none", "self") and score != clean_score:
            raise RuntimeError("Self-copy or baseline changed original logits")
        return dict(row, score=score, prefix_natural=on_path, patch_fired_live=bool(heads and on_path),
                    measure="live" if on_path else "prefix_only_off_path", output=output)


def targets68(worlds):
    strata, targets = defaultdict(list), {}
    for w in worlds:
        strata[w["template"], w["reverse"]].append(w["id"])
    for group in strata.values():
        ordered = sorted(group, key=lambda wid: hashlib.sha256(("v68/target/v1|"+wid).encode()).digest())
        targets.update((wid, i%2) for i, wid in enumerate(ordered))
    return targets


def audit68(study):
    worlds, arms = study.worlds, study.arms
    if (worlds != make_worlds("confirm", 240)[192:] or study.smoke_worlds != make_worlds("confirm", 242)[240:]
            or study.targets != targets68(worlds) or Counter((w["template"],w["reverse"],study.targets[w["id"]]) for w in worlds) !=
            Counter({(i, r, t):6 for i in (0,1) for r in (False,True) for t in (0,1)})):
        raise ValueError("New worlds, smoke separation or stratification changed")
    top = set(map(tuple, arms["top80"]))
    if set(arms) != set(AB_ARMS) or arms["none"]:
        raise ValueError("Expected original five groups")
    for arm in AB_MASKS:
        values = list(map(tuple, arms[arm]))
        if (len(values) != 80 or len(set(values)) != 80 or any(not(0<=l<32 and 0<=i<32) for l,i in values)
                or (arm != "top80" and top.intersection(values)) or Counter(l for l,_ in values) != Counter(l for l,_ in top)
                or stable_hash(values) != study.reference["mask_hashes"][arm]):
            raise ValueError("Frozen head coordinates changed: "+arm)
    longest = 0
    for w in [*worlds, *study.smoke_worlds]:
        stem, items, split = decision_tokens(study.backend, *w["items"])
        if split != len(stem) or records_for(w,0,"C1") != records_for(w,1,"C1"):
            raise ValueError("Unmatched item token or changed memory")
        for h in (0,1,"Z",None):
            p, q = (prompt68(w,t,h) for t in (0,1))
            # Examples contain their own MEMORY_JSON markers. Split at the live request only.
            a, separator, ma = p.rpartition("\nMEMORY_JSON:")
            b, other_sep, mb = q.rpartition("\nMEMORY_JSON:")
            if not separator or separator != other_sep or ma != mb or a == b or a.replace(w["requests"][0],w["requests"][1]) != b:
                raise ValueError("Target donor differs beyond the live request: "+w["id"])
            lp, lq = len(study.backend.encode(p)), len(study.backend.encode(q))
            if lp != lq:
                raise ValueError("Target prompts must have aligned token positions")
            longest = max(longest, lp+len(items[0]))
        for t in (0,1):
            left, right = [prompt68(w,t,h) for h in (0,1)]
            a, sep, ah = left.rpartition("\nEXECUTED_TOOL_HISTORY_JSON:")
            b, sep2, bh = right.rpartition("\nEXECUTED_TOOL_HISTORY_JSON:")
            if not sep or sep != sep2 or a != b or ah == bh or ah.replace(w["items"][0],w["items"][1]) != bh or len(study.backend.encode(left)) != len(study.backend.encode(right)):
                raise ValueError("History donor differs beyond the actual stock lookup: "+w["id"])
    if longest > study.backend.cfg["max_prompt_tokens"]:
        raise ValueError("Frozen input exceeds original budget")
    return dict(passed=True, worlds=len(worlds), smoke_worlds=2, max_tokens=longest, mask_hashes=study.reference["mask_hashes"])


def trace_key68(run):
    return ([event_signature(e) for e in run["events"][1:]], run["terminal"], run["final_state"])


class PatchStudy68(Study):
    def __init__(self, root, backend, inputs, reference, identity, verify):
        self.reference, self.targets = reference, inputs["targets"]
        super().__init__(root, backend, inputs["worlds"], inputs["smoke"], reference["arms"],
                         dict(identity, targets=self.targets), verify)

    def smoke68(self):
        rows = []
        for t,w in enumerate(self.smoke_worlds):
            runner = WorldPatch68(self, w, t)
            for h in (t, 1-t):
                orig, sham = (runner.cell(h,kind,arm) for kind,arm in (("none","none"),("self","top80")))
                if orig["score"] != sham["score"] or trace_key68(orig["output"]) != trace_key68(sham["output"]):
                    raise RuntimeError("Identity interchange changed free actions or final environment")
                for kind, arm in (("target","top80"),("history","random80_1"),("intent","random80_2")):
                    observed = runner.cell(h, kind, arm)
                    # Independent replay also checks donor alignment on the actual free-generation path.
                    q, path, _ = prefix68(self.backend, prompt68(w,t,h), runner.prefix, runner.pair,
                                    donor=runner.clean(*donor_state68(t,h,kind))[2], heads=self.arms[arm])
                    if q != observed["score"] or path != observed["prefix_natural"]:
                        raise RuntimeError("Live vs controlled-prefix intervention mismatch")
                rows.append(dict(world=w["id"], target=t, history=h, natural=orig["prefix_natural"]))
            runner.cell("Z", "target", "random80_3")
        if not any(r["natural"] for r in rows):
            raise RuntimeError("No smoke case exercised a live patch. Check the runtime before formal trials.")
        return dict(passed=True, checks=rows)

    def prepare68(self):
        self.check()
        for name, fn in zip(CHECKS68, (lambda:audit68(self), self.smoke68)):
            if not self.unit(name, fn)["passed"]:
                raise RuntimeError("Failed technical check: "+name)
        print(" Enter, refill the original and generate the path by the technology pre-screening. ", flush=True)

    def run68(self):
        self.check()
        if not all((self.unit(key) or {}).get("passed") for key in CHECKS68):
            raise RuntimeError("Please run preflight first")
        for i,w in enumerate(self.worlds, 1):
            t = self.targets[w["id"]]
            runner, before = None, self.new_units
            for h,kind,arm,key in cells68(w,t):
                if self.unit(key) is None:
                    if runner is None:
                        runner = WorldPatch68(self,w,t)
                    self.unit(key, lambda:runner.cell(h,kind,arm))
            del runner
            if before != self.new_units or i == len(self.worlds):
                print(f"[{i}/{len(self.worlds)}] {w['id']}  Done, this one is new  {self.new_units-before}  Breakpoint ", flush=True)
        self.check()


@contextmanager
def writer68(root, takeover=""):
    root.mkdir(parents=True, exist_ok=True)
    path, token = root/".writer", uuid4().hex
    if takeover and path.is_file() and path.read_text("ascii").strip() == takeover:
        path.unlink()
    try:
        with path.open("x", encoding="ascii") as f:
            f.write(token)
            f.flush()
            os.fsync(f.fileno())
    except FileExistsError:
        owner = path.read_text("ascii").strip()
        raise RuntimeError(f" This tab still has a running lock. Confirms old  runtime  When stopped, lock number  {owner!r}  Enter  TAKEOVER_LOCK_ID  Keep going. ") from None
    try:
        yield
    finally:
        if path.exists() and path.read_text("ascii").strip() == token:
            path.unlink()
'''


In [ ]:
V69_ROUTES69_TASKS_SOURCE = r'''"""Frozen v6.9 worlds, source annotations and the unchanged executable tool loop.

This module is embedded after the frozen v6/v6.8 namespace. It performs no Hub
calls and contains no model credentials. Module A/C retain the original prompt;
module B explicitly versions its three-record and display-catalog extension.
"""

from collections import Counter, defaultdict
import copy
import hashlib
import random


ROUTES69_SLICES = {"development": (264, 288), "confirmation": (288, 384), "smoke": (384, 386)}
SOURCE_NAMES69 = ("current_record", "other_record", "history_action", "history_observation")
CATALOG_NOTE69 = "Non-operational listing only; not an executed action and not an update to request fields."


def frozen_inputs69():
    """New IDs follow the frozen original generator; no selection by model result."""
    base = make_worlds("confirm", 400)
    used = set()
    for w in base:
        for key in ("requests", "items", "lots", "events"):
            used.update(v.replace("-", "").split("demo")[-1] for v in w[key])
        used.update((w["fallback"].replace("-", ""), w["customer"].replace("-", "")))
    result, targets = {}, {}
    for split, (begin, end) in ROUTES69_SLICES.items():
        result[split] = copy.deepcopy(base[begin:end])
        strata = defaultdict(list)
        for w in result[split]:
            rng = random.Random("v69/third-record/v1/" + w["id"])
            item_groups = {sku.split("-")[1] for sku in (*w["items"], w["fallback"])}
            if len(item_groups) != 3:
                raise ValueError("Frozen original SKUs do not differ at the first numeric group")

            def identifier(kind, excluded_groups=()):
                while True:
                    digits = str(rng.randint(400000, 999999))
                    key = kind + digits
                    if key not in used and digits[:3] not in excluded_groups:
                        used.add(key)
                        return f"{kind}-{digits[:3]}-{digits[3:]}"

            w["third_item"] = identifier("sku", item_groups)
            w["third_request"] = identifier("req")
            w["third_event"] = identifier("evt")
            w["cohort69"] = split
            strata[w["template"], w["reverse"]].append(w["id"])
        for group in strata.values():
            ordered = sorted(group, key=lambda wid: hashlib.sha256(("v69/target/v1/" + wid).encode()).digest())
            targets.update((wid, i % 2) for i, wid in enumerate(ordered))
    # The two smoke worlds exercise the two target identities; they never enter estimates.
    targets.update((w["id"], i % 2) for i, w in enumerate(result["smoke"]))
    result["targets"] = targets
    result["survey_ids"] = {split: [w["id"] for w in result[split][:n]]
                            for split, n in (("development", 8), ("confirmation", 24))}
    result["source_ids"] = copy.deepcopy(result["survey_ids"])
    return result


def symbols69(w, t):
    if t not in (0, 1):
        raise ValueError("Invalid target identity")
    return dict(X=w["items"][t], Y=w["items"][1-t], Z=w["third_item"], W=w["fallback"])


def _catalog69(symbols, mention):
    if mention not in ("Y", "Z"):
        raise ValueError("The B catalog must end in Y or Z")
    first = "Z" if mention == "Y" else "Y"
    return {"item_ids": [symbols[first], symbols[mention]], "purpose": CATALOG_NOTE69}


def render_prompt69(case, records, events, catalog=None):
    text = actor_prompt(case, records, events)
    if catalog is None:
        return text
    marker = "\nNEXT_TOOL_CALL:\n"
    if not text.endswith(marker):
        raise RuntimeError("Frozen Actor completion boundary changed")
    return (text[:-len(marker)] + "\nDISPLAY_CATALOG_JSON:\n" +
            canonical_json(catalog) + marker)


def prompt_from_state69(state, events):
    return render_prompt69(state["case"], state["records"], events, state["catalog"])


def _unique_span69(text, needle, begin, end):
    i = text.find(needle, begin, end)
    if i < 0 or text.find(needle, i + 1, end) >= 0:
        raise ValueError("A source span is absent or ambiguous within its live block")
    return [i, i + len(needle)]


def source_spans69(state, prompt):
    """Character spans of live records and tool fields, never demonstration matches."""
    memory_marker, history_marker = "\nMEMORY_JSON:\n", "\nEXECUTED_TOOL_HISTORY_JSON:\n"
    memory_begin = prompt.rfind(memory_marker)
    history_begin = prompt.rfind(history_marker)
    answer_begin = prompt.rfind("\nNEXT_TOOL_CALL:\n")
    if not (0 <= memory_begin < history_begin < answer_begin):
        raise ValueError("Missing or reordered live prompt sections")
    memory_begin += len(memory_marker)
    history_begin += len(history_marker)
    catalog_begin = prompt.rfind("\nDISPLAY_CATALOG_JSON:\n")
    history_end = catalog_begin if catalog_begin > history_begin else answer_begin
    out = {k: [] for k in SOURCE_NAMES69}
    for record in state["records"]:
        group = "current_record" if record["entity"] == state["case"]["request_id"] else "other_record"
        out[group].append(_unique_span69(prompt, canonical_json(record), memory_begin, history_begin - len(history_marker)))
    for event in state["history"]:
        for key in ("action", "observation"):
            # Here all source states contain exactly one genuinely executed lookup.
            out["history_" + key].append(_unique_span69(prompt, canonical_json(event[key]), history_begin, history_end))
    if any(not v for v in out.values()):
        raise ValueError("Every prespecified source must exist")
    if state["module"] == "B":
        begin = catalog_begin + len("\nDISPLAY_CATALOG_JSON:\n")
        out["display_catalog"] = [_unique_span69(prompt, canonical_json(_catalog69(state["symbols"], state["mention"])), begin, answer_begin)]
    return out


def state69(w, t, module, context, start, mention="none"):
    if module not in ("A", "B", "C"):
        raise ValueError("Unknown v6.9 module")
    if context not in ("Clex", "C1"):
        raise ValueError("Unknown relation label")
    if module == "B":
        if context != "C1" or start not in ("X", "Y", "Z") or mention not in ("Y", "Z"):
            raise ValueError("B requires a stock lookup and independent catalog order")
        # X is a neutral competence preflight only; formal cells use Y/Z.
    elif start not in ("X", "Y") or mention != "none":
        raise ValueError("A/C retain original X/Y history without display catalog")
    symbols = symbols69(w, t)
    case = case_for(w, t)
    records = records_for(w, t, context)
    if module == "B":
        case["initial_env"]["stock"][symbols["Z"]] = 2
        extra = {"kind": "verified_profile", "entity": w["third_request"],
                 "attribute": "preferred_item", "value": symbols["Z"], "source_event": w["third_event"]}
        # Mirror the full sequence with the original record-order factor.
        if w["reverse"]:
            records.insert(0, extra)
        else:
            records.append(extra)
    env = OrderEnvironment(case["initial_env"])
    action = {"tool": "check_stock", "item": symbols[start]}
    observation = env.step(action)
    if observation != {"ok": True, "item": symbols[start], "available": 2}:
        raise RuntimeError("Fixed lookup differs from the actual equal-stock environment")
    history = [{"action": action, "observation": observation, "parse_error": None,
                "origin": "fixed_prefix", "step": 0}]
    visible_symbols = symbols if module == "B" else {key: symbols[key] for key in ("X", "Y", "W")}
    result = dict(world=w["id"], target=t, module=module, context=context, start=start, mention=mention,
                  case=case, records=records, history=history, symbols=visible_symbols,
                  catalog=_catalog69(symbols, mention) if module == "B" else None)
    result["prompt"] = prompt_from_state69(result, history)
    result["source_spans"] = source_spans69(result, result["prompt"])
    if module in ("A", "C") and result["prompt"] != ab_prompt(w, t, context, "gold" if start == "X" else "foil"):
        raise RuntimeError("Original A/C prompt or history was modified")
    return result


def run_branch69(backend, state):
    """The original six-action budget includes the externally executed first lookup."""
    case, records = state["case"], state["records"]
    env = OrderEnvironment(case["initial_env"])
    events = copy.deepcopy(state["history"])
    for event in events:
        if env.step(event["action"]) != event["observation"]:
            raise ValueError("Stored initial history is not a genuine tool execution")
    end = "action_budget"
    while len(events) < backend.cfg["max_actions"]:
        prompt = prompt_from_state69(state, events)
        if len(backend.encode(prompt)) > backend.cfg["max_prompt_tokens"]:
            end = "prompt_budget"
            break
        out = backend.generate(prompt, role="actor", heads=(), use_cache=True)
        action, error = parse_json_output(out["text"], dict)
        observation = None if error else env.step(action)
        events.append({"step": len(events), "origin": "model", "prompt": prompt,
                       "action": action, "observation": observation, "parse_error": error, "generation": out})
        if error:
            end = "format_failure"
            break
        if observation.get("error") in ("action_must_be_object", "unknown_tool", "invalid_arguments"):
            end = "protocol_failure"
            break
        if action.get("tool") == "create_order" and observation.get("ok"):
            end = "order_created"
            break
    metrics = gx_score_actor(case, events, env.state["orders"], case["initial_env"])
    first = events[0]["action"]
    metrics.update(first_lookup_other=first["item"] in (state["symbols"]["Y"], state["symbols"].get("Z")),
                   first_lookup_gold=first["item"] == state["symbols"]["X"], first_valid_lookup=True,
                   prompt_budget_hit=end == "prompt_budget", action_budget_hit=end == "action_budget")
    return {"metrics": metrics, "events": events, "initial_state": case["initial_env"],
            "final_state": env.snapshot(), "terminal": end, "forced_check": True}


def audit_input_structures69(inputs):
    if inputs != frozen_inputs69():
        raise ValueError("v6.9 frozen worlds or target allocation changed")
    counts, ids = {}, set()
    for split in ROUTES69_SLICES:
        worlds = inputs[split]
        counts[split] = len(worlds)
        if split != "smoke":
            strata = Counter((w["template"], w["reverse"], inputs["targets"][w["id"]]) for w in worlds)
            if len(strata) != 8 or len(set(strata.values())) != 1:
                raise ValueError("Unequal target/template/order allocation")
        for w in worlds:
            if w["id"] in ids:
                raise ValueError("World appears in two cohorts")
            ids.add(w["id"])
            t = inputs["targets"][w["id"]]
            symbols = symbols69(w, t)
            if len(set(symbols.values())) != 4 or len({x.split("-")[1] for x in symbols.values()}) != 4:
                raise ValueError("Four SKU identities require distinct initial numeric groups")
            for start in ("X", "Y"):
                a, b = [state69(w, t, "A", c, start) for c in ("Clex", "C1")]
                if a["prompt"].replace('"attribute":"preview_item"', '"attribute":"preferred_item"') != b["prompt"]:
                    raise ValueError("A label contrast changes more than the other relation")
                if a["history"] != b["history"] or a["case"] != b["case"]:
                    raise ValueError("A label contrast changed the actual task/history")
            for mention in ("Y", "Z"):
                y, z = [state69(w, t, "B", "C1", h, mention) for h in ("Y", "Z")]
                if y["case"] != z["case"] or y["records"] != z["records"]:
                    raise ValueError("B lookup changed request or memory")
                for s in (y, z):
                    if s["history"][0]["action"]["item"] == symbols["X"]:
                        raise ValueError("B must begin with an unchecked target")
                    for item in (symbols["Y"], symbols["Z"]):
                        # One record, one display entry; queried SKU appears in action + observation.
                        expected = 2 + 2 * int(item == s["history"][0]["action"]["item"])
                        if s["prompt"].count(item) != expected:
                            raise ValueError("Unexpected B SKU exposure")
    return dict(passed=True, cohorts=counts, disjoint_worlds=len(ids), actual_equal_stock_histories=True,
                original_A_C_prompt_unchanged=True, B_catalog_contains_both_competitors_once=True)
'''


In [ ]:
V69_ROUTES69_SOURCE = r'''"""Frozen, source-scoped causal interventions; executed in the audited v6 namespace."""

SCOPE69 = "v6p9_semantic_routes"
STAGES69 = ("development", "confirmation")
CHECKS69 = ("checks/inputs", "checks/smoke")
SOURCES69 = ("current_record", "other_record", "history_action", "history_observation")
TRAJECTORY_ARMS69 = ("top80", "random80_1", "random80_2", "random80_3")
SURVEY_ARMS69 = tuple(f"survey80_{i:02d}" for i in range(1, 33))


def cells69(study, stage, w):
    if stage not in STAGES69 or w not in study.inputs[stage]:
        raise ValueError("Unknown stage/world")
    def cell(module, context, start, mention, kind, arm, measure, source="none"):
        suffix = f"{context}-{start}-{mention}/{kind}-{arm}-{source}"
        return dict(key=f"{stage}/{module}/{w['id']}/{suffix}", stage=stage, world=w["id"],
                    module=module, context=context, start=start, mention=mention,
                    kind=kind, arm=arm, source=source, measure=measure)
    survey = w["id"] in study.inputs["survey_ids"][stage]
    for c in ("Clex", "C1"):
        for h in ("X", "Y"):
            for kind, arm in (("none", "none"), ("self", "top80"),
                              *(("label", a) for a in TRAJECTORY_ARMS69)):
                yield cell("A", c, h, "none", kind, arm, "trajectory")
            if survey:
                for arm in SURVEY_ARMS69:
                    yield cell("A", c, h, "none", "label", arm, "probe")
    for h in ("Y", "Z"):
        for mention in ("Y", "Z"):
            for kind, arm in (("none", "none"), ("self", "top80"),
                              *(("history", a) for a in TRAJECTORY_ARMS69)):
                yield cell("B", "C1", h, mention, kind, arm, "trajectory")
            if survey:
                for arm in SURVEY_ARMS69:
                    yield cell("B", "C1", h, mention, "history", arm, "probe")
            for arm in TRAJECTORY_ARMS69:
                yield cell("B", "C1", h, mention, "mention", arm, "probe")
    # Same extended schema, correctly checked target: normal task competence on every world.
    for mention in ("Y", "Z"):
        yield cell("B", "C1", "X", mention, "none", "none", "trajectory")
    if w["id"] in study.inputs["source_ids"][stage]:
        for c in ("Clex", "C1"):
            for source in SOURCES69:
                for arm in TRAJECTORY_ARMS69:
                    yield cell("C", c, "Y", "none", "source", arm, "trajectory", source)


def donor_cell69(cell):
    d = dict(cell)
    if cell["kind"] == "label":
        d["context"] = "C1" if cell["context"] == "Clex" else "Clex"
    elif cell["kind"] == "history":
        d["start"] = "Z" if cell["start"] == "Y" else "Y"
    elif cell["kind"] == "mention":
        d["mention"] = "Z" if cell["mention"] == "Y" else "Y"
    elif cell["kind"] not in ("none", "self", "source"):
        raise ValueError("Unknown interchange")
    return d


def expected69(study, stage):
    counts = Counter()
    for w in study.inputs[stage]:
        for cell in cells69(study, stage, w):
            counts[cell["measure"]] += 1
            counts[cell["module"] + "_" + cell["measure"]] += 1
    return dict(worlds=len(study.inputs[stage]), **counts)


def decision69(be, symbols):
    enc = lambda s: list(be.tokenizer.encode(s, add_special_tokens=False))
    prefix = enc(ITEM_PREFIX)
    sequences = {name: enc('{"item":"' + sku) for name, sku in symbols.items()}
    if (len(set(map(len, sequences.values()))) != 1 or
            any(seq[:len(prefix)] != prefix or len(seq) <= len(prefix) for seq in sequences.values())):
        raise ValueError("Unaligned item tokenization")
    ids = {name: seq[len(prefix)] for name, seq in sequences.items()}
    if len(set(ids.values())) != len(ids):
        raise ValueError("All SKU first numeric tokens must differ")
    return prefix, ids


def score69(logits, ids):
    values = {name: float(logits[0, -1, tok].float().item()) for name, tok in ids.items()}
    if not all(map(math.isfinite, values.values())):
        raise ValueError("Non-finite candidate logit")
    return dict(logits=values, mxy=values["X"]-values["Y"],
                myz=values["Y"]-values["Z"] if "Z" in values else None,
                greedy=int(logits[0, -1].argmax().item()))


def source_tokens69(be, state, source):
    """Absolute prompt key positions; validate against the exact original BOS tokenization."""
    spans = state["source_spans"][source]
    encoded = be.tokenizer(state["prompt"], add_special_tokens=True, return_offsets_mapping=True)
    ids, offsets = list(encoded["input_ids"]), list(encoded["offset_mapping"])
    if ids != list(be.encode(state["prompt"])) or len(ids) != len(offsets):
        raise ValueError("Source offsets do not match the actual Actor tokenizer")
    # Exclude tokens straddling JSON boundaries: adjacent records must not share a masked key.
    indices = sorted({i for a,b in spans for i,(lo,hi) in enumerate(offsets)
                      if hi > lo and lo >= a and hi <= b})
    if not indices or 0 in indices or any(not (0 <= i < len(ids)) for i in indices):
        raise ValueError("Empty or invalid live source token span")
    return indices


def source_hooks69(heads, positions):
    """Knock out source edges before softmax, only at the one instrumented query."""
    if not positions or len(set(positions)) != len(positions):
        raise ValueError("Expected nonempty unique key positions")
    hooks = []
    for layer, indices in heads_by_layer68(heads).items():
        def block(value, hook, indices=indices):
            if value.ndim != 4 or value.shape[0] != 1 or value.shape[2] != 1:
                raise RuntimeError("Source intervention escaped one cached query position")
            if min(positions) < 0 or max(positions) >= value.shape[3] or max(indices) >= value.shape[1]:
                raise RuntimeError("Source/head index outside the actual attention scores")
            result = value.clone()
            for head in indices:
                result[0, head, 0, positions] = -torch.inf
                if not torch.isfinite(result[0, head, 0]).any():
                    raise RuntimeError("All attention keys were blocked")
            return result
        hooks.append((f"blocks.{layer}.attn.hook_attn_scores", block))
    return hooks


def prefix69(be, prompt, prefix, ids, *, capture=(), hooks=()):
    if not prefix or (capture and hooks):
        raise ValueError("Invalid probe")
    initial = be.tokens(prompt)
    if initial.shape[1] + len(prefix) > be.cfg["max_prompt_tokens"]:
        raise ValueError("Probe exceeds frozen prompt budget")
    activations, hits = {}, Counter()
    requested = list(hooks)
    for layer in sorted(capture):
        def record(value, hook, layer=layer):
            if value.shape[:2] != (1,1) or layer in activations:
                raise RuntimeError("Invalid donor capture position")
            activations[layer] = value.detach().clone()
            return value
        requested.append((f"blocks.{layer}.attn.hook_z", record))
    def counted(name, fn):
        def call(value, hook):
            hits[name] += 1
            return fn(value, hook)
        return call
    tracked = [(name, counted(name, fn)) for name, fn in requested]
    cache = tl_cache(be)
    with torch.inference_mode():
        logits = be.model(initial, past_kv_cache=cache, return_type="logits")
        natural = True
        for offset, token in enumerate(prefix):
            natural = natural and int(logits[0, -1].argmax().item()) == token
            step = torch.tensor([[token]], device=be.device, dtype=torch.long)
            with be.model.hooks(fwd_hooks=tracked if offset == len(prefix)-1 else []):
                logits = be.model(step, past_kv_cache=cache, return_type="logits")
        result = score69(logits, ids)
    if any(hits[name] != 1 for name, _ in requested):
        raise RuntimeError("Probe hook coverage is not exactly one per selected layer")
    return result, natural, activations


class LiveRoute69:
    """Never insert a prefix or rewrite an action; only instrument the first free call."""
    def __init__(self, be, prompt, prefix, ids, hooks):
        self.be, self.prompt, self.prefix, self.ids, self.hooks = be, prompt, prefix, ids, hooks
        self.cfg, self.encode = be.cfg, be.encode
        self.used, self.score, self.hits = False, None, Counter()
        self.observed_prefix_ids = []

    def generate(self, prompt, role, heads, use_cache=True):
        if self.used:
            return self.be.generate(prompt, role=role, heads=heads, use_cache=use_cache)
        if prompt != self.prompt or role != "actor" or heads or not use_cache:
            raise ValueError("Unexpected first free Actor call")
        self.used = True
        initial, model = self.be.tokens(prompt), self.be.model
        offset, matching, active = -1, True, False
        def before(module, args):
            nonlocal offset, matching, active
            tokens = args[0]
            active = False
            if offset < 0:
                if not torch.equal(initial, tokens):
                    raise RuntimeError("Wrong initial recipient prefill")
                offset = 0
            else:
                if tuple(tokens.shape) != (1,1):
                    raise RuntimeError("Expected one cached generated token")
                if len(self.observed_prefix_ids) < len(self.prefix):
                    self.observed_prefix_ids.append(int(tokens.item()))
                if matching and offset < len(self.prefix):
                    matching = int(tokens.item()) == self.prefix[offset]
                    offset += 1
                    active = matching and offset == len(self.prefix)
        def after(module, args, out):
            if active:
                if self.score is not None:
                    raise RuntimeError("Decision hook fired twice")
                self.score = score69(out, self.ids)
        def gate(name, fn):
            def call(value, hook):
                if not active:
                    return value
                self.hits[name] += 1
                return fn(value, hook)
            return call
        pre = model.register_forward_pre_hook(before)
        try:
            post = model.register_forward_hook(after)
            try:
                with model.hooks(fwd_hooks=[(name, gate(name, fn)) for name, fn in self.hooks]):
                    result = self.be.generate(prompt, role=role, heads=(), use_cache=True)
            finally:
                post.remove()
        finally:
            pre.remove()
        if offset < 0 or any(self.hits[name] != int(self.score is not None) for name,_ in self.hooks):
            raise RuntimeError("Live hook coverage mismatch")
        return result


def state_from_cell69(w, t, cell):
    return state69(w, t, cell["module"], cell["context"], cell["start"], cell["mention"])


class WorldRoutes69:
    def __init__(self, study, w, t):
        self.study, self.w, self.t, self.be = study, w, t, study.backend
        self.layers = sorted(heads_by_layer68(study.arms["top80"]))
        self.cache = {}

    def clean(self, state):
        prefix, ids = decision69(self.be, state["symbols"])
        key = stable_hash([state["prompt"], ids])
        if key not in self.cache:
            self.cache[key] = prefix69(self.be, state["prompt"], prefix, ids, capture=self.layers)
        return prefix, ids, self.cache[key]

    def cell(self, cell):
        start_time = time.monotonic()
        recipient = state_from_cell69(self.w, self.t, cell)
        donor_state = state_from_cell69(self.w, self.t, donor_cell69(cell))
        prefix, ids, (clean_score, natural, _) = self.clean(recipient)
        _, donor_ids, (donor_score, donor_natural, vectors) = self.clean(donor_state)
        if donor_ids != ids:
            raise ValueError("Donor and recipient candidate identities differ")
        positions = []
        if cell["kind"] == "source":
            positions = source_tokens69(self.be, recipient, cell["source"])
            hooks = source_hooks69(self.study.arms[cell["arm"]], positions)
        elif cell["kind"] == "none":
            hooks = []
        else:
            hooks = patch_hooks68(vectors, self.study.arms[cell["arm"]])
        row = dict(cell=cell, world=self.w["id"], target=self.t,
                   prompt_hash=stable_hash(recipient["prompt"]), donor_prompt_hash=stable_hash(donor_state["prompt"]),
                   clean_score=clean_score, donor_score=donor_score, prefix_natural=natural,
                   donor_prefix_natural=donor_natural, prefix_ids=prefix, sku_ids=ids,
                   source_token_indices=positions,
                   source_char_spans=recipient["source_spans"].get(cell["source"], []),
                   prompt_token_count=len(self.be.encode(recipient["prompt"])), patch_fired_live=False)
        if cell["measure"] == "probe":
            score, match, _ = prefix69(self.be, recipient["prompt"], prefix, ids, hooks=hooks)
            if match != natural:
                raise RuntimeError("Intervention altered tokens before its declared position")
            row["score"] = score
        else:
            actor = LiveRoute69(self.be, recipient["prompt"], prefix, ids, hooks)
            output = run_branch69(actor, recipient)
            if not actor.used or (actor.score is not None) != natural:
                raise RuntimeError("Live trajectory and replay disagree on the decision prefix")
            if actor.score is None:
                score, _, _ = prefix69(self.be, recipient["prompt"], prefix, ids, hooks=hooks)
            else:
                score = actor.score
            row.update(score=score, output=output, patch_fired_live=bool(hooks and natural),
                       live_hook_hits=dict(actor.hits), live_prefix_ids=actor.observed_prefix_ids)
        if cell["kind"] in ("none", "self") and row["score"] != clean_score:
            raise RuntimeError("Baseline/self-copy changed candidate logits")
        row["seconds"] = time.monotonic()-start_time
        return row


def audit69(study):
    if study.inputs != frozen_inputs69():
        raise ValueError("Frozen scenarios or cohort assignment changed")
    audit_input_structures69(study.inputs)
    top = set(map(tuple, study.arms["top80"]))
    if set(study.arms) != {"none", *TRAJECTORY_ARMS69, *SURVEY_ARMS69} or study.arms["none"]:
        raise ValueError("Wrong candidate/control inventory")
    for name, heads in study.arms.items():
        if name == "none":
            continue
        points = list(map(tuple, heads))
        if (len(points) != 80 or len(set(points)) != 80 or
                Counter(l for l,h in points) != Counter(l for l,h in top) or
                any(not(0<=l<32 and 0<=h<32) for l,h in points)):
            raise ValueError("Wrong head set: "+name)
        if name.startswith("random80_") and top.intersection(points):
            raise ValueError("Historical control now overlaps candidate")
    be, maximum, pairs = study.backend, 0, 0
    for w in [*study.worlds, *study.smoke_worlds]:
        t = study.inputs["targets"][w["id"]]
        for h in ("X", "Y"):
            left, right = [state69(w,t,"A",c,h,"none") for c in ("Clex", "C1")]
            a, b = (be.encode(s["prompt"]) for s in (left,right))
            if len(a) != len(b) or sum(x!=y for x,y in zip(a,b)) != 1:
                raise ValueError("A must differ by exactly one label token")
            for s in (left,right):
                prefix, _ = decision69(be, s["symbols"])
                maximum = max(maximum, len(be.encode(s["prompt"]))+len(prefix))
                for source in SOURCES69:
                    source_tokens69(be, s, source)
            pairs += 1
        states = [state69(w,t,"B","C1",h,m) for h in ("Y","Z") for m in ("Y","Z")]
        if len({len(be.encode(s["prompt"])) for s in states}) != 1:
            raise ValueError("B histories/mentions must have equal token lengths")
        for s in states:
            prefix, _ = decision69(be, s["symbols"])
            maximum = max(maximum,len(be.encode(s["prompt"]))+len(prefix))
    if maximum > be.cfg["max_prompt_tokens"]:
        raise ValueError("Frozen prompt budget exceeded; do not truncate")
    return dict(passed=True, worlds=len(study.worlds), smoke_worlds=len(study.smoke_worlds),
                label_token_pairs=pairs, max_prompt_with_stem=maximum,
                expected={s:expected69(study,s) for s in STAGES69},
                heads={a:stable_hash(h) for a,h in study.arms.items()})


def trace69(output):
    return ([event_signature(e) for e in output["events"] if e["origin"] == "model"],
            output["terminal"],output["final_state"])


class RoutesStudy69(Study):
    def __init__(self, root, backend, inputs, reference, identity, verify):
        self.inputs, self.reference = inputs, reference
        self.targets = inputs["targets"]
        super().__init__(root,backend,inputs["development"]+inputs["confirmation"],inputs["smoke"],
                         reference["arms"],dict(identity, inputs=inputs, reference=reference),verify)

    def smoke69(self):
        checks, timings = [], []
        for w in self.smoke_worlds:
            t = self.targets[w["id"]]
            runner = WorldRoutes69(self,w,t)
            for module, context, start, mention in (("A","Clex","Y","none"),("A","C1","X","none"),
                                                  ("B","C1","Y","Z"),("B","C1","Z","Y")):
                base = dict(key="technical",stage="smoke",world=w["id"],module=module,context=context,
                            start=start,mention=mention,kind="none",arm="none",source="none",measure="trajectory")
                original = runner.cell(base)
                sham = runner.cell(dict(base,kind="self",arm="top80"))
                if original["score"] != sham["score"] or trace69(original["output"]) != trace69(sham["output"]):
                    raise RuntimeError("Self-copy failed on free actions or final state")
                if module == "A" and start == "X" and not original["output"]["metrics"]["strict_task_success"]:
                    raise RuntimeError("Basic correct-history task failed; inspect smoke before confirmation")
                kinds = ("label",) if module == "A" else ("history","mention")
                for kind in kinds:
                    c = dict(base,kind=kind,arm="top80")
                    observed = runner.cell(c)
                    recipient = state_from_cell69(w,t,c)
                    ds = state_from_cell69(w,t,donor_cell69(c))
                    prefix, ids, (_,natural,_) = runner.clean(recipient)
                    _,_,(_,_,vectors) = runner.clean(ds)
                    replay, match, _ = prefix69(self.backend,recipient["prompt"],prefix,ids,
                                               hooks=patch_hooks68(vectors,self.arms["top80"]))
                    if observed["score"] != replay or match != natural:
                        raise RuntimeError("Independent live/replay interchange mismatch")
                    timings.append(observed["seconds"])
                checks.append(dict(world=w["id"],module=module,start=start,natural=original["prefix_natural"],
                                   strict=original["output"]["metrics"]["strict_task_success"]))
                print(f"[smoke] {w['id']} {module}/{context}/{start}: self-copy + live/replay passed",flush=True)
            # Every source family must fire in a real free generation and agree with replay.
            for source in SOURCES69:
                c=dict(base,module="C",context="C1",start="Y",mention="none",kind="source",arm="top80",source=source)
                observed=runner.cell(c)
                recipient=state_from_cell69(w,t,c)
                prefix,ids,(_,natural,_)=runner.clean(recipient)
                positions=source_tokens69(self.backend,recipient,source)
                replay,match,_=prefix69(self.backend,recipient["prompt"],prefix,ids,
                                      hooks=source_hooks69(self.arms["top80"],positions))
                if observed["score"]!=replay or match!=natural:
                    raise RuntimeError("Source-edge live/replay mismatch")
                if not observed["patch_fired_live"]:
                    raise RuntimeError("Source smoke did not exercise its actual free-generation hook")
                print(f"[smoke] {w['id']} source/{source}: live/replay passed",flush=True)
            # B neutral competence: an actual first check X in the extended catalog prompt.
            neutral=state69(w,t,"B","C1","X","Y")
            out=run_branch69(self.backend,neutral)
            if not out["metrics"]["strict_task_success"]:
                raise RuntimeError("B extended task fails after a correct lookup; do not interpret as interference")
        if not all(c["natural"] for c in checks):
            raise RuntimeError("Smoke lacks natural prefix coverage in at least one task")
        return dict(passed=True,checks=checks,mean_trajectory_seconds=sum(timings)/len(timings),
                    note="Technical tests only; no effect direction or error-rate success gate.")

    def prepare69(self):
        self.check()
        if self.backend is None:
            raise RuntimeError("Read-only export cannot execute inference")
        for key,fn in ((CHECKS69[0],lambda:audit69(self)),(CHECKS69[1],self.smoke69)):
            if not self.unit(key,fn)["passed"]:
                raise RuntimeError("Technical preflight failed: "+key)
        print(" Input, self-patching, actual generation positions, and tool-workflow preflight passed. ",flush=True)

    def run69(self, stage, max_wall_hours=0):
        self.check()
        if stage not in STAGES69 or max_wall_hours < 0 or not math.isfinite(float(max_wall_hours)):
            raise ValueError("Invalid stage or wall-clock limit")
        if self.backend is None or not all((self.unit(k) or {}).get("passed") for k in CHECKS69):
            raise RuntimeError("Run the technical preflight before inference")
        started=time.monotonic()
        for i,w in enumerate(self.inputs[stage],1):
            runner,before=None,self.new_units
            for cell in cells69(self,stage,w):
                if self.unit(cell["key"]) is not None:
                    continue
                if max_wall_hours and time.monotonic()-started >= max_wall_hours*3600:
                    print(" Reaching the maximum time limit; breakpoint saved and exported  PARTIAL。",flush=True)
                    return False
                if runner is None:
                    runner=WorldRoutes69(self,w,self.targets[w["id"]])
                self.unit(cell["key"],lambda cell=cell:runner.cell(cell))
                if (self.new_units-before) % 16 == 0:
                    print(f"[{stage} {i}/{len(self.inputs[stage])}] {w['id']} {cell['module']} "
                          f" Add a new scenarios  {self.new_units-before}  bars;at present used  {(time.monotonic()-started)/60:.1f}  min ",flush=True)
            del runner
            if before!=self.new_units or i==len(self.inputs[stage]):
                print(f"[{stage} {i}/{len(self.inputs[stage])}] {w['id']}  finished; adding  {self.new_units-before}  Article ",flush=True)
            self.check()
        return True


def open_saved69(root, inputs, reference, identity_checks):
    """Read-only model-free reopening; never creates or repairs a checkpoint directory."""
    root=Path(root)
    if not (root/"manifest.json").is_file():
        raise RuntimeError(" No same one found  RUN_LABEL  checkpoints; export  No model will be loaded or new experiments created. ")
    manifest=read_json(root/"manifest.json")
    if (manifest.get("inputs")!=inputs or manifest.get("reference")!=reference or
            any(manifest.get(k)!=v for k,v in identity_checks.items())):
        raise RuntimeError(" Export Code / Input does not match the original run, please use the original  notebook  Version. ")
    study=object.__new__(RoutesStudy69)
    study.root,study.backend=root,None
    study.inputs,study.reference,study.targets=inputs,reference,inputs["targets"]
    study.worlds=inputs["development"]+inputs["confirmation"]
    study.smoke_worlds,study.arms=inputs["smoke"],reference["arms"]
    study.manifest,study.signature,study.new_units=manifest,stable_hash(manifest),0
    study.verify=lambda:None
    study.check()
    return study
'''


In [ ]:
V69_ROUTES69_REPORTS_SOURCE = r'''"""Read-only replay, stage-complete world-paired inference and verified Drive export.

Executed in the frozen notebook namespace after routes69.py. No model or Hub call
is made here; export can therefore run from saved checkpoints after a restart.
"""

TABLES69 = ("branches", "probes", "conditions", "effects", "world-effects", "survey")
MAIN69 = ("label_toward_donor", "history_toward_donor")
SEED69, DRAWS69 = 76902, 4000
CHECKS_REPORT69 = ("checks/inputs", "checks/smoke")
OLD_ARMS69 = ("top80", "random80_1", "random80_2", "random80_3")
SOURCE_REPORT69 = ("current_record", "other_record", "history_action", "history_observation")
OUTCOMES69 = ("final_X", "final_Y", "final_Z", "final_W", "final_other", "final_no_order")
BOOL_REPORT69 = ("prefix_natural", "donor_prefix_natural", "patch_fired_live", "first_recheck_X",
                 "first_free_X", "first_free_Y", "first_free_Z", "first_free_W",
                 "first_reserve_wrong_attempt", "first_reserve_wrong_accepted", "strict_task_success",
                 "format_failure", "tool_error", "prompt_budget_hit", "action_budget_hit", *OUTCOMES69)


def _state_for_cell69(study, w, cell):
    return state69(w, study.inputs["targets"][w["id"]], cell["module"], cell["context"],
                   cell["start"], cell["mention"])


def _donor_for_cell69(study, w, cell):
    donor = dict(cell)
    if cell["kind"] == "label":
        donor["context"] = "C1" if cell["context"] == "Clex" else "Clex"
    elif cell["kind"] == "history":
        donor["start"] = "Z" if cell["start"] == "Y" else "Y"
    elif cell["kind"] == "mention":
        donor["mention"] = "Z" if cell["mention"] == "Y" else "Y"
    return _state_for_cell69(study, w, donor)


def _score_valid69(score):
    values = score["logits"]
    if set(values) not in ({"X", "Y", "Z", "W"}, {"X", "Y", "W"}) or not all(type(v) in (int, float) and math.isfinite(v) for v in values.values()):
        raise ValueError("Invalid four-SKU score")
    if score["mxy"] != values["X"] - values["Y"] or score["myz"] != (values["Y"] - values["Z"] if "Z" in values else None):
        raise ValueError("Saved margin disagrees with its raw logits")
    if type(score["greedy"]) is not int or score["greedy"] < 0:
        raise ValueError("Invalid vocabulary argmax")


def evaluated69(study, w, cell, saved):
    if saved["cell"] != cell or saved["world"] != w["id"] or saved["target"] != study.inputs["targets"][w["id"]]:
        raise ValueError("Checkpoint belongs to another cell")
    state, donor = _state_for_cell69(study, w, cell), _donor_for_cell69(study, w, cell)
    if saved["prompt_hash"] != stable_hash(state["prompt"]) or saved["donor_prompt_hash"] != stable_hash(donor["prompt"]):
        raise ValueError("Donor or recipient prompt provenance changed")
    for key in ("score", "clean_score", "donor_score"):
        _score_valid69(saved[key])
    if any(type(saved[k]) is not bool for k in ("prefix_natural", "donor_prefix_natural", "patch_fired_live")):
        raise ValueError("Coverage flags must be genuine booleans")
    if not saved["prefix_ids"] or not all(type(v) is int and v >= 0 for v in saved["prefix_ids"]):
        raise ValueError("Invalid output prefix")
    if type(saved["prompt_token_count"]) is not int or saved["prompt_token_count"] <= 0:
        raise ValueError("Invalid prompt token count")
    if type(saved.get("seconds")) not in (int,float) or not math.isfinite(saved["seconds"]) or saved["seconds"] < 0:
        raise ValueError("Invalid measured cell duration")
    if (set(saved["sku_ids"]) != set(saved["score"]["logits"]) or len(set(saved["sku_ids"].values())) != len(saved["sku_ids"])
            or any(type(v) is not int or v < 0 for v in saved["sku_ids"].values())):
        raise ValueError("Missing distinguishing SKU token identities")
    if cell["kind"] in ("none", "self") and saved["score"] != saved["clean_score"]:
        raise ValueError("Baseline or identity patch changed its score")
    row = {k: cell[k] for k in ("key", "stage", "world", "module", "context", "start", "mention", "kind", "arm", "source", "measure")}
    row.update(target=study.inputs["targets"][w["id"]], template=w["template"], reverse=w["reverse"],
               mxy=saved["score"]["mxy"], myz=saved["score"]["myz"],
               clean_mxy=saved["clean_score"]["mxy"], clean_myz=saved["clean_score"]["myz"],
               donor_mxy=saved["donor_score"]["mxy"], donor_myz=saved["donor_score"]["myz"],
               prefix_natural=saved["prefix_natural"], donor_prefix_natural=saved["donor_prefix_natural"],
               patch_fired_live=saved["patch_fired_live"],seconds=saved["seconds"],prompt_token_count=saved["prompt_token_count"])
    row.update(("logit_"+k, v) for k, v in saved["score"]["logits"].items())
    row.update(("greedy_"+k, saved["score"]["greedy"] == v) for k, v in saved["sku_ids"].items())
    if cell["kind"] == "source":
        positions = saved.get("source_token_indices", [])
        if (not positions or positions != sorted(set(positions)) or any(type(v) is not int or not 0 < v < saved["prompt_token_count"] for v in positions)
                or saved["source_char_spans"] != state["source_spans"][cell["source"]]):
            raise ValueError("Missing or invalid source-token coverage")
        row["source_token_count"] = len(positions)
    elif saved.get("source_token_indices") or saved.get("source_char_spans"):
        raise ValueError("Unexpected source intervention metadata")
    if cell["measure"] == "probe":
        if "output" in saved or saved["patch_fired_live"]:
            raise ValueError("A fixed-prefix probe cannot contain an executed trajectory")
        return row
    if cell["measure"] != "trajectory":
        raise ValueError("Unknown measurement type")
    out, history = saved["output"], state["history"]
    events = out["events"]
    if not out["forced_check"] or out["initial_state"] != state["case"]["initial_env"] or events[:len(history)] != history:
        raise ValueError("Initial tool state or actual imposed history changed")
    if not len(history) < len(events) <= 6:
        raise ValueError("Missing free action or changed six-action budget")
    env = OrderEnvironment(out["initial_state"])
    for i, event in enumerate(events):
        if event["step"] != i:
            raise ValueError("Non-sequential tool steps")
        if i >= len(history):
            if event["origin"] != "model" or event["prompt"] != prompt_from_state69(state, events[:i]):
                raise ValueError("Free prompt was not derived from the preceding real history")
            parsed, error = parse_json_output(event["generation"]["text"], dict)
            if parsed != event["action"] or error != event["parse_error"]:
                raise ValueError("Raw generation does not match parsed tool action")
        if event.get("parse_error"):
            if event["observation"] is not None or i != len(events)-1:
                raise ValueError("Malformed generation executed a tool or continued")
        elif env.step(event["action"]) != event["observation"]:
            raise ValueError("Tool observation differs from deterministic replay")
    if env.snapshot() != out["final_state"]:
        raise ValueError("Final database state differs from replayed actual tool actions")
    orders = env.state["orders"]
    if len(orders) > 1 or bool(orders) != (out["terminal"] == "order_created"):
        raise ValueError("Invalid order stopping condition")
    metrics = gx_score_actor(state["case"], events, orders, state["case"]["initial_env"])
    if any(out["metrics"].get(k) != v for k, v in metrics.items()):
        raise ValueError("Stored scoring disagrees with reconstructed orders")
    first = events[len(history)]
    if stable_hash(first["prompt"]) != saved["prompt_hash"] or saved["patch_fired_live"] != bool(cell["arm"] != "none" and saved["prefix_natural"]):
        raise ValueError("Live-prefix coverage or first prompt disagrees")
    live_ids=saved["live_prefix_ids"]
    if (not isinstance(live_ids,list) or len(live_ids)>len(saved["prefix_ids"])
            or any(type(v) is not int or v < 0 for v in live_ids)
            or (live_ids==saved["prefix_ids"]) != saved["prefix_natural"]
            or (saved["prefix_natural"] and not first["generation"]["text"].startswith(ITEM_PREFIX))):
        raise ValueError("Consumed free-generation tokens disagree with natural prefix coverage")
    hook_kind="hook_attn_scores" if cell["kind"]=="source" else "hook_z"
    expected_hits={f"blocks.{layer}.attn.{hook_kind}":1 for layer,_ in study.arms[cell["arm"]]} if saved["patch_fired_live"] else {}
    if saved["live_hook_hits"] != expected_hits:
        raise ValueError("Recorded hook sites/counts do not match the assigned heads")
    action, obs = first["action"] or {}, first["observation"] or {}
    item, tool = action.get("item"), action.get("tool")
    symbols = state["symbols"]
    wrong = item is not None and item != symbols["X"]
    row.update(first_free_tool=tool, first_free_item=item,
               first_recheck_X=tool == "check_stock" and item == symbols["X"] and bool(obs.get("ok")),
               first_reserve_wrong_attempt=tool == "reserve_item" and wrong,
               first_reserve_wrong_accepted=tool == "reserve_item" and wrong and bool(obs.get("ok")),
               strict_task_success=bool(metrics["strict_task_success"]),
               format_failure=bool(metrics["format_failure"]),
               tool_error=any(e.get("observation") and not e["observation"].get("ok") for e in events),
               prompt_budget_hit=out["terminal"] == "prompt_budget",
               action_budget_hit=out["terminal"] == "action_budget", terminal=out["terminal"])
    row.update(("first_free_"+k, k in symbols and item == symbols[k]) for k in ("X","Y","Z","W"))
    final_item = orders[0].get("item") if orders else None
    row.update(("final_"+k, bool(orders) and k in symbols and final_item == symbols[k]) for k in ("X","Y","Z","W"))
    row.update(final_other=bool(orders) and final_item not in symbols.values(), final_no_order=not orders,
               final_item=final_item)
    if sum(row[k] for k in OUTCOMES69) != 1:
        raise ValueError("Terminal categories are not mutually exclusive and exhaustive")
    return row


def _key_row69(r):
    return r["module"], r["context"], r["start"], r["mention"], r["kind"], r["arm"], r["source"]


def _row69(index, module, context, start, mention, kind, arm, source="none"):
    return index[module, context, start, mention, kind, arm, source]


def _paired_summary69(endpoint, comparison, subset, worlds, values, unit, co_primary=False):
    arr = np.asarray(values, dtype=np.float64)
    if not len(arr) or len(arr) != len(worlds) or len(set(worlds)) != len(worlds) or not np.isfinite(arr).all():
        raise ValueError("Invalid independent-world contrast")
    draws = np.random.default_rng(SEED69).integers(0, len(arr), size=(DRAWS69, len(arr)))
    boot = arr[draws].mean(axis=1)
    lo, hi = np.quantile(boot, [.025, .975])
    result = dict(endpoint=endpoint, comparison=comparison, subset=subset, n_worlds=len(arr), unit=unit,
                  estimate=float(arr.mean()), ci95_low=float(lo), ci95_high=float(hi), co_primary=co_primary,
                  positive=int((arr > 0).sum()), negative=int((arr < 0).sum()), tied=int((arr == 0).sum()))
    if co_primary:
        lo, hi = np.quantile(boot, [.0125, .9875])
        result.update(ci97p5_low=float(lo), ci97p5_high=float(hi))
    return result


def effects69(study, stage, rows):
    """All signs are defined before data; complete world blocks are paired first."""
    by_world, details = defaultdict(dict), []
    for r in rows:
        key = _key_row69(r)
        if key in by_world[r["world"]]:
            raise ValueError("Duplicate measured condition")
        by_world[r["world"]][key] = r
    def add(w, endpoint, comparison, value, unit="logit", subset="all"):
        details.append(dict(world=w["id"], template=w["template"], endpoint=endpoint,
                            comparison=comparison, subset=subset, value=float(value), unit=unit))
    for w in study.inputs[stage]:
        idx = by_world[w["id"]]
        def fetch(module,c,s,m,kind,arm,source="none"):
            return _row69(idx,module,c,s,m,kind,arm,source)
        def delta(module,c,s,m,kind,arm,field,source="none"):
            return float(fetch(module,c,s,m,kind,arm,source)[field])-float(fetch(module,c,s,m,"none","none")[field])
        measures = defaultdict(dict)
        for arm in OLD_ARMS69:
            measures["label_toward_donor"][arm] = sum((1 if c == "C1" else -1)*delta("A",c,s,"none","label",arm,"mxy") for c in ("Clex","C1") for s in ("X","Y"))/4
            measures["history_toward_donor"][arm] = sum((1 if s == "Z" else -1)*delta("B","C1",s,m,"history",arm,"myz") for s in ("Y","Z") for m in ("Y","Z"))/4
            measures["mention_toward_donor"][arm] = sum((1 if m == "Z" else -1)*delta("B","C1",s,m,"mention",arm,"myz") for s in ("Y","Z") for m in ("Y","Z"))/4
            measures["history_minus_mention_transfer"][arm] = measures["history_toward_donor"][arm]-measures["mention_toward_donor"][arm]
            measures["label/donor_buy_Y_change"][arm] = 100*sum((1 if c == "Clex" else -1)*delta("A",c,s,"none","label",arm,"final_Y") for c in ("Clex","C1") for s in ("X","Y"))/4
            measures["history/donor_order_change"][arm] = 100*sum(delta("B","C1",s,m,"history",arm,"final_Z" if s == "Y" else "final_Y") for s in ("Y","Z") for m in ("Y","Z"))/4
            for c in ("Clex", "C1"):
                for s in ("X", "Y"):
                    for field in ("mxy", "final_X", "final_Y", "final_W", "final_no_order", "strict_task_success", "first_recheck_X", "first_reserve_wrong_attempt"):
                        measures[f"label/{c}/{s}/{field}"][arm] = delta("A",c,s,"none","label",arm,field)*(1 if field == "mxy" else 100)
            for s in ("Y", "Z"):
                for field in ("mxy", "myz", *OUTCOMES69, "strict_task_success", "first_recheck_X", "first_reserve_wrong_attempt"):
                    measures[f"history/{s}/{field}"][arm] = sum(delta("B","C1",s,m,"history",arm,field) for m in ("Y","Z"))/2*(1 if field in ("mxy","myz") else 100)
        for endpoint, values in measures.items():
            avg = sum(values[a] for a in OLD_ARMS69[1:])/3
            unit = "logit" if endpoint in (*MAIN69,"mention_toward_donor","history_minus_mention_transfer") or endpoint.endswith(("/mxy","/myz")) else "percentage_points"
            comparisons = {**values,"random_mean":avg,"top_minus_random":values["top80"]-avg}
            comparisons.update(("top_minus_"+a,values["top80"]-values[a]) for a in OLD_ARMS69[1:])
            for cmp, value in comparisons.items():
                add(w,endpoint,cmp,value,unit)
                if endpoint in MAIN69 and cmp == "top_minus_random":
                    add(w,endpoint,cmp,value,unit,"template_"+str(w["template"]))
        # Unpatched total-label contrast and fully changed donor-prompt reference.
        for s in ("X","Y"):
            for field in ("mxy","final_Y","strict_task_success"):
                value = float(fetch("A","C1",s,"none","none","none")[field])-float(fetch("A","Clex",s,"none","none","none")[field])
                add(w,f"total_label/C1-minus-Clex/{s}/{field}","intact",value*(1 if field == "mxy" else 100),"logit" if field == "mxy" else "percentage_points")
        for endpoint,module,kind,score in ((MAIN69[0],"A","label","mxy"),(MAIN69[1],"B","history","myz")):
            values=[]
            states=[(c,s,"none") for c in ("Clex","C1") for s in ("X","Y")] if module=="A" else [("C1",s,m) for s in ("Y","Z") for m in ("Y","Z")]
            for c,s,m in states:
                row=fetch(module,c,s,m,kind,"top80")
                sign=(1 if c=="C1" else -1) if module=="A" else (1 if s=="Z" else -1)
                values.append(sign*(row["donor_"+score]-row["clean_"+score]))
            add(w,endpoint,"whole_donor_prompt",sum(values)/4)
        if w["id"] in study.inputs["source_ids"][stage]:
            for source in SOURCE_REPORT69:
                for field in ("mxy",*OUTCOMES69,"strict_task_success","first_recheck_X","first_reserve_wrong_attempt"):
                    unit="logit" if field=="mxy" else "percentage_points"
                    factor=1 if field=="mxy" else 100
                    per_arm={}
                    for arm in OLD_ARMS69:
                        contrasts={c:factor*(float(fetch("C",c,"Y","none","source",arm,source)[field])-float(fetch("A",c,"Y","none","none","none")[field])) for c in ("Clex","C1")}
                        for c,value in contrasts.items():
                            add(w,f"source/{source}/{c}/{field}",arm,value,unit)
                        per_arm[arm]=sum(contrasts.values())/2
                        add(w,f"source/{source}/label_interaction/{field}",arm,contrasts["C1"]-contrasts["Clex"],unit)
                    mean=sum(per_arm[a] for a in OLD_ARMS69[1:])/3
                    for cmp,value in {**per_arm,"random_mean":mean,"top_minus_random":per_arm["top80"]-mean}.items():
                        add(w,f"source/{source}/label_average/{field}",cmp,value,unit)
    groups=defaultdict(list)
    for r in details:
        groups[r["endpoint"],r["comparison"],r["subset"],r["unit"]].append(r)
    effects=[]
    for (endpoint,cmp,subset,unit),rr in sorted(groups.items()):
        main=stage=="confirmation" and endpoint in MAIN69 and cmp=="top_minus_random" and subset=="all"
        effects.append(_paired_summary69(endpoint,cmp,subset,[r["world"] for r in rr],[r["value"] for r in rr],unit,main))
    return effects,details,survey69(study,stage,by_world)


def survey69(study, stage, by_world):
    """Reference distribution is separate from the three historical controls."""
    names=sorted(a for a in study.arms if a.startswith("survey80_"))
    worlds=study.inputs["survey_ids"][stage]
    rows=[]
    for endpoint,module,kind,field in ((MAIN69[0],"A","label","mxy"),(MAIN69[1],"B","history","myz")):
        matrix=[]
        for wid in worlds:
            idx=by_world[wid]
            values=[]
            states=[(c,s,"none") for c in ("Clex","C1") for s in ("X","Y")] if module=="A" else [("C1",s,m) for s in ("Y","Z") for m in ("Y","Z")]
            for arm in ("top80",*names):
                contrasts=[]
                for c,s,m in states:
                    changed=_row69(idx,module,c,s,m,kind,arm)
                    orig=_row69(idx,module,c,s,m,"none","none")
                    sign=(1 if c=="C1" else -1) if module=="A" else (1 if s=="Z" else -1)
                    contrasts.append(sign*(changed[field]-orig[field]))
                values.append(sum(contrasts)/4)
            matrix.append(values)
        matrix=np.asarray(matrix,dtype=np.float64)
        means=matrix.mean(axis=0)
        rank=1+int((means[1:]>=means[0]).sum())
        for j,arm in enumerate(("top80",*names)):
            rows.append(dict(endpoint=endpoint,arm=arm,n_worlds=len(worlds),mean_transfer=float(means[j]),
                             candidate_upper_tail_rank=rank,reference_set_count=len(names),
                             candidate_minus_set=float(means[0]-means[j]),inference="descriptive_fixed_sample_reference"))
        fixed=matrix[:,0]-matrix[:,1:].mean(axis=1)
        row=_paired_summary69(endpoint,"top_minus_32_random_mean","survey_worlds",worlds,fixed,"logit")
        # Independent resampling of worlds and random-set draws: distinguishes the
        # sampled-set uncertainty from the ordinary fixed-control world interval.
        rng=np.random.default_rng(SEED69)
        boot=[]
        for _ in range(DRAWS69):
            wi=rng.integers(0,len(worlds),size=len(worlds))
            si=rng.integers(1,len(names)+1,size=len(names))
            boot.append(float(matrix[wi,0].mean()-matrix[wi][:,si].mean()))
        lo,hi=np.quantile(boot,[.025,.975])
        row.update(arm="top_minus_survey_mean",candidate_upper_tail_rank=rank,reference_set_count=len(names),
                   crossed_ci95_low=float(lo),crossed_ci95_high=float(hi),
                   inference="secondary; world CI conditions on fixed sets; crossed CI resamples worlds and sampled head sets")
        rows.append(row)
    return rows


def expected_report69(study,stage):
    cells=[c for w in study.inputs[stage] for c in cells69(study,stage,w)]
    counts=dict(worlds=len(study.inputs[stage]),cells=len(cells),
                trajectories=sum(c["measure"]=="trajectory" for c in cells),probes=sum(c["measure"]=="probe" for c in cells))
    counts.update(("module_"+m,sum(c["module"]==m for c in cells)) for m in ("A","B","C"))
    return counts


def keys69(study,stage):
    yield from CHECKS_REPORT69
    for w in study.inputs[stage]:
        for cell in cells69(study,stage,w):
            yield cell["key"]


def snapshot69(study,z,stage):
    names=set(z.namelist())
    preflight=True
    for key in CHECKS_REPORT69:
        check=read_checkpoint(z,names,study,key)
        preflight &= bool(check and check.get("passed"))
        if check is not None and not check.get("passed"):
            raise ValueError("Failed preflight saved in archive")
    rows,completed=[],0
    for w in study.inputs[stage]:
        saved_by_key={}
        expected=list(cells69(study,stage,w))
        for cell in expected:
            saved=read_checkpoint(z,names,study,cell["key"])
            if saved is None:
                continue
            rows.append(evaluated69(study,w,cell,saved))
            saved_by_key[_key_row69(cell)]=saved
        for key,saved in saved_by_key.items():
            if key[4]=="self":
                basekey=(*key[:4],"none","none",key[-1])
                base=saved_by_key.get(basekey)
                if base is not None and (saved["score"]!=base["score"] or trace_key68(saved["output"])!=trace_key68(base["output"])):
                    raise ValueError("Identity patch changed the original trace")
        completed+=len(saved_by_key)==len(expected)
    expected=expected_report69(study,stage)
    counts=dict(worlds=completed,cells=len(rows),trajectories=sum(r["measure"]=="trajectory" for r in rows),probes=sum(r["measure"]=="probe" for r in rows))
    counts.update(("module_"+m,sum(r["module"]==m for r in rows)) for m in ("A","B","C"))
    complete=preflight and counts==expected
    conditions=[]
    groups=defaultdict(list)
    for r in rows:
        groups[tuple(r[k] for k in ("module","context","start","mention","kind","arm","source","measure"))].append(r)
    for key,rr in sorted(groups.items()):
        row=dict(zip(("module","context","start","mention","kind","arm","source","measure"),key))
        row.update(n=len(rr),independent_worlds=len({r["world"] for r in rr}),mean_mxy=sum(r["mxy"] for r in rr)/len(rr),mean_myz=sum(r["myz"] for r in rr)/len(rr) if all(r["myz"] is not None for r in rr) else None)
        row.update(total_cell_seconds=sum(r["seconds"] for r in rr),mean_cell_seconds=sum(r["seconds"] for r in rr)/len(rr))
        for metric in BOOL_REPORT69:
            if metric in rr[0]:
                row[metric+"_count"]=sum(bool(r[metric]) for r in rr)
        conditions.append(row)
    effects,details,survey=effects69(study,stage,rows) if complete else ([],[],[])
    primary=sorted((r for r in effects if r["co_primary"]),key=lambda r:MAIN69.index(r["endpoint"]))
    report=dict(scope="v6p9_semantic_routes",stage=stage,status="COMPLETE" if complete else "PARTIAL",counts=counts,expected=expected,
                manifest_hash=study.signature,preflight_complete=preflight,primary=primary,analysis_role="confirmatory" if stage=="confirmation" else "development_only",
                inference_unit="independent world",bootstrap_draws=DRAWS69,bootstrap_seed=SEED69,
                primary_intervals="Two 97.5% marginal world-paired percentile intervals, Bonferroni family nominal 95%; controls are the three frozen historical sets.",
                survey_worlds=len(study.inputs["survey_ids"][stage]),source_worlds=len(study.inputs["source_ids"][stage]),
                observed_cell_seconds=sum(r["seconds"] for r in rows),timing_note="Measured per completed cell; includes its uncached donor capture, excludes technical preflight/export and abandoned attempts.",
                source_analysis="exploratory attention-source knockout; renormalization changes remaining routes; not isolated path mediation",
                task_risk="Externally imposed true first lookup; not autonomous error incidence. Probes without output are not executed actions.")
    tables=dict(zip(TABLES69,([r for r in rows if r["measure"]=="trajectory"],[r for r in rows if r["measure"]=="probe"],conditions,effects,details,survey)))
    return report,tables


def validate_zip69(path,study,report):
    stage=report["stage"]
    assets=study.manifest["assets_sha256"]
    fixed={"manifest.json","reports/report.json","reports/table-sha256.json",*assets,*("reports/"+n+".csv" for n in TABLES69)}
    with zipfile.ZipFile(path) as z:
        names=set(z.namelist())
        if z.testzip() is not None or len(names)!=len(z.infolist()) or not fixed.issubset(names):
            raise IOError("ZIP CRC or inventory failure")
        if stable_hash(json.loads(z.read("manifest.json")))!=study.signature or report["manifest_hash"]!=study.signature or json.loads(z.read("reports/report.json"))!=report:
            raise IOError("Archive/report identity mismatch")
        if report["scope"]!="v6p9_semantic_routes" or stage not in ("development","confirmation"):
            raise IOError("Wrong result scope or cohort")
        for name,digest in assets.items():
            if hashlib.sha256(z.read(name)).hexdigest()!=digest:
                raise IOError("Frozen input/code digest mismatch: "+name)
        table_hashes=json.loads(z.read("reports/table-sha256.json"))
        expected_tables={"reports/"+name+".csv" for name in TABLES69}
        if set(table_hashes)!=expected_tables or any(hashlib.sha256(z.read(name)).hexdigest()!=digest for name,digest in table_hashes.items()):
            raise IOError("Derived CSV inventory/digest mismatch")
        allowed={k+".json" for k in keys69(study,stage)}
        present=names-fixed
        if not present.issubset(allowed):
            raise IOError("Unexpected or cross-cohort checkpoint")
        for name in present:
            read_checkpoint(z,names,study,name[:-5])
        if len(present)-sum(k+".json" in present for k in CHECKS_REPORT69)!=report["counts"]["cells"]:
            raise IOError("Checkpoint inventory does not match report")
        if report["status"]=="COMPLETE":
            if present!=allowed or report["counts"]!=expected_report69(study,stage) or not report["preflight_complete"]:
                raise IOError("Incomplete archive marked COMPLETE")
            if stage=="confirmation" and ([r["endpoint"] for r in report["primary"]]!=list(MAIN69) or any(r["n_worlds"]!=len(study.inputs[stage]) for r in report["primary"])):
                raise IOError("Missing or incomplete primary confirmation estimates")
        elif report["status"]!="PARTIAL" or report["primary"]:
            raise IOError("Invalid partial report")


def export69(study,assets,stage):
    study.check()
    if stage not in ("development","confirmation"):
        raise ValueError("Unknown export cohort")
    if {k:hashlib.sha256(v).hexdigest() for k,v in assets.items()}!=study.manifest["assets_sha256"]:
        raise ValueError("Frozen assets changed")
    with tempfile.TemporaryDirectory() as scratch:
        folder=Path(scratch)
        local=folder/"snapshot.zip"
        with zipfile.ZipFile(local,"w",compression=zipfile.ZIP_DEFLATED) as z:
            z.write(study.root/"manifest.json","manifest.json")
            for name,raw in assets.items():
                z.writestr(name,raw)
            for key in keys69(study,stage):
                p=study.root/(key+".json")
                if p.is_file():
                    z.write(p,key+".json")
        with zipfile.ZipFile(local) as z:
            report,tables=snapshot69(study,z,stage)
        with zipfile.ZipFile(local,"a",compression=zipfile.ZIP_DEFLATED) as z:
            z.writestr("reports/report.json",canonical_json(report))
            table_hashes={}
            for name,entries in tables.items():
                write_csv(folder/(name+".csv"),entries)
                table_name="reports/"+name+".csv"
                table_hashes[table_name]=file_sha(folder/(name+".csv"))
                z.write(folder/(name+".csv"),table_name)
            z.writestr("reports/table-sha256.json",canonical_json(table_hashes))
        validate_zip69(local,study,report)
        path=study.root.parent/("v6p9_semantic_routes_"+stage+"_"+report["status"]+"_results.zip")
        digest=file_sha(local)
        fd,pending=tempfile.mkstemp(dir=path.parent,prefix="v69-",suffix=".zip.pending")
        os.close(fd)
        pending=Path(pending)
        try:
            shutil.copyfile(local,pending)
            if file_sha(pending)!=digest:
                raise IOError("Drive temporary copy readback mismatch")
            os.replace(pending,path)
        finally:
            pending.unlink(missing_ok=True)
        if file_sha(path)!=digest:
            raise IOError("Drive final readback mismatch")
    save_json(study.root/"reports"/(stage+".json"),report)
    print(stage,report["status"],report["counts"],"Drive ZIP",path,"SHA-256",digest,flush=True)
    return report,path,digest


def finish69(study,result,enabled,mountpoint,is_mounted,flush,release):
    if not enabled or not result or result[0]["status"]!="COMPLETE":
        print(" The current stage is not fully archived or auto-release is not enabled, keep  runtime。",flush=True)
        return False
    report,path,digest=result
    mountpoint=Path(mountpoint).resolve(strict=True)
    target=Path(path).resolve(strict=True)
    expected="v6p9_semantic_routes_"+report["stage"]+"_COMPLETE_results.zip"
    if (not is_mounted(str(mountpoint)) or not target.is_relative_to((mountpoint/"MyDrive").resolve())
            or target.parent!=study.root.parent.resolve() or target.name!=expected
            or target.with_name(".writer").exists() or file_sha(target)!=digest):
        raise IOError("Drive mount/path/write lock/digest check failed; runtime retained")
    validate_zip69(target,study,report)
    print(" Completeness of the current phase  ZIP  Verifyed and rereading, synchronizing  Drive； Release after sync succeeded  runtime。",flush=True)
    flush(timeout_ms=600_000)
    release()
    return True
'''


In [ ]:
#@title  Verify built-in world, candidate, random contrast and study source 
V69_INPUT_BYTES = zlib.decompress(base64.b64decode((
'eNrtfUnPXUdy5X/hWg/IOSK17a172BcKBkuibaIksZqkCm0U6r93nJxefjcj6bx8WnphuEhevSkyYzhx4sQ/3v384e8ffvn0t18//Pb13Y9/+se7jz+/+/Hd'
'T59++7ePn399GONSfPfDu4+//fzh/7370aXww7svf/vl49fnM/Kvnz/8398/fPn6Rf57/O9HpPAwMdZ/eeTsHpnduz/Ly3z98Gt56stff39QtA8iI0/hTzn4'
'R3blqV8+1ZeS///gYB/R4qXwJ+L8yIbwkHzq3+pjH/4ujyX3cOTf/VD+FFN6eMN47N/e//LLX97/9Ff5vHgXzvkRjJPnfvr9y9dPv374LP/w/qef5BW8fJqU'
'8S8fv/6n/O3//vL+r+/lj/KR//bL+68f3v1o8H3+/uHzF/nf8rpfPsg//sfHzz//K75We4PoE17qXf+n9tPIv5YfxptHcGH8a/kS8m/40Em+KeeED/DpPz59'
'/iqf5cc3xvnnD4px0myceGKcHOQncL4Zh4J7eEercUIQo1EzDkcWU6WrcaJNj5h9M052Qb5gXo2TvLyjj904chxggtU4IcqxcUYxTs7+4YJ5Gud//v7rX95/'
'fGMde2AdjiTHJG+sQ97LeSPVOlF+j2T8TevQbJ10ZB0nnyGGZp0o5z1GXqyDc0TlKfwpGflWYtGLdYj44Sw36yS5bTnH1TrRym9bfnX8KbC8MgXFOjHjDETF'
'OkGujmH3tM7/ev/x86e/fNxenq+ff9esA4fgXNpYJ1GS/7O6dXJ42GxvWodn69CJdYitnNLU745J8pOmxTrB8iNZ1x1bzMUZXu8Oy0c2qTu2FPEFV+uQMQ+f'
'+91hua2JsmKdHMTvBM06ciXFIYandf4HrLO9OjvbmPBg4u3NiXLU8sav+Yd1d/1anm3DJ7ZJLj98+QzlE4nHssEutslOzliy3a+5KH9jlpuT7MN7arYJXhyW'
'tYpfy6k9hj9l4+UuWcU2KTr5/bSgQ3L3QvlVD2/OzrHh/MTot1fHi0PXzZNJfjY298xTovYwTz7KCSTyuuIk8CcEFKfkBBlB0HEzTyD3sDFfzRNMxK3qVycb'
'pBhKToDXKpe1uAjJSSiSZh5xsuQ180SScF1C7DfMcxJ3suMHG7Mzj0s4Pbp5AuMG3TSPncxD5sQ8OORUPGL5RBKJQ1izAvhgNiMrkPBv/Rp35MvGYunyJyvB'
'IfrVPMhDkC4084g7hP9SzCOejThpKZuTi+nTN7OCg7AT5Utl77cpm7x90lM2TnKK/c2wQ242jj1KCtiJCzfdOC2uLymbM+KAelIQJZj6RFfjpCxprhthxxr5'
'1Y2SFIhXCMzNOEncVLKacaKkK4lIc21y32ycws7/+fz+33//cDfuZHnrzDvjsNwcT0aPOwm53l3H5mfjuKOcQJxr4n5zIpH8onnNCTg+LOduHBwq4qtxMklw'
'smnK2MRAq3GCRHQbRj7tSeKelrGJ85NnWXNscnNccMc3Z5tPS6ZDOWysE1gif2L96nj4tbvWCbN1/FkpKmGHej7NpTqLi3VYThl+6qd1PC1hJ0p54N2odiQx'
'y1bJ2EI0ksW6UYrKeTBey9gQAmvBea120sWvfW/YITl0Pu3CTiCYR88KxEU8TD0gN8wzIwV0hBQgoTXBdqTAkoQds2YFci8kieoJtbiwnNaEWspP8qZfHsnN'
'WCt3UKy44iWLeSSVkNJOLUYzDqlqHkIB9q2E+iDqBHG99XKqjk18vPGbhNohO72bEsxAAR0BBSg9nO0oTixp/Jqx4Uq5kHrGJk+RWRwbbkHyeXJsLTe/JtTy'
'rWO3DaWEXEK9OlKMswYUkJSvpqTkZ0DBxjhJ/GPc4gTMkjBEvRINKRYY655xZpyAjnCCgNM+/Bo5RAS/pgSSOFTUovg1OXJGuTgZhfWA2CQbsi4oKUH2QEg6'
'TiDvSGq1g6xMT6eRPeYSYV+C2EhuMHPc5WsSdKpH0ZJpgCt3b84ME9ARTBAltWXqpSgcfVKS6RgMkpSeTOdarVzzNfyaxc416ARJ7BTjyOvLLRylaJK7ajSY'
'ABHBsdVAHGR8yR3nazvrBG9Qiu9KHUbJoENsKNlNvJtNz0ABHQEFnADIdOuEiGO9AgUSSgo00GKOpCvW2cU6EpgidaAAjjknBZ2OAbm4G35NbKBWolIAPWzW'
'sulg48Nk8zLExnIDUtwlbJLPyDdNetRh+XL+pmPjGSegI5wgSQ5gfccJpOqQGLRaB27WpVHreGQRS9QhcQX5mRFIqeJIuTvZyP104+6UWkuzTpZk11gNYgME'
'wYlfjToxytdg2kUdYOCkowT4pr64xDvGmVECPkIJgBUTh1GISs3OYU3X5JMG3x1bNOKQ04oSePmPkxmNHclU2WkoAcvh71EH7RvLWu8AObs8q6VrIcgRcqfp'
'2tavyQdxW79G4vE96Tcnys2J6WbU4Rkl4COUgK2kVRyHXyPJxNaUIJARD9RTAmYWA61dNx/lwg8AVA5+0PwaS0WE9kRv7EgubbWbQ+L/cjHicnPEF7vSGjoC'
'p/ddN/kgW+OEFHBWN3Uo4mu+aZwZJeAzlEB+0RB55GuE8K/UoSQ/VA86UlnL3yzGQf0jRWZ3a5LU+aihBJIEhDD6Oigaghp05F9qx3JBCcQV5hxeLHSSBFKz'
'DTmxtX3VkCPFWcp3Q86MEfARRpDk8uaRrjEaZW71akG8WipHtfxJ8jJLC7zGKP5txwhSEkuRcnEIfsyFEXIi/kO15yZWy9rFScXj0zlGsOvroFXu3c6tSXJU'
'LaC7Nedv9nV4hgj4DCJA39HEqdKJJisxB50xN2JOEFcYlkrHSF5uexkaCZFe6VfD1TOPSkdeK7Dq1uSTBaeCn0kupXEvkwmMa25YzaVTaumN1nTDV7iZS/OM'
'EfAZRuBpYnqQZGsavJYlFuXQwc/sQEBYcuksl6XWjeUiya/uVGRaclQ/2tWlnnFaQpBJIptKJkAnKswYwb98/PKXT7/dR9ec1DJ2h+BEiqXfrtahEnUT3c0I'
'ZpCAj0ACuRAtL62dDHEmbm1Xo1OQ4wg66Ln5tefGpp31ClSTRBOtqSPvYQeCkz2q0qw6NsnyS0hemzry0tG/GHRQ5zBtMYKMprpehQaHeHo36MwYAdNZtiYZ'
'7xPAMWiGrT0dljseeADT6A8oDbfQSU3FUhE0HK0fKu+Rc69Cg9wjo3KkQAnioAUd5JSB46tUArR0fNoma5IobagE2cs3sN+yzZ9/6D/5+68f5U5rhDaeIQM+'
'ggwAKpHvvA8J6cjn1u4onGGgAVOj17B2EcRrUO7GylLyyYsrgE45EKPuQZM+q93RhBik8j4Akp7D1NsY5IDn8BYxQId+U5RKap2uPYQ31tGuUp4hAz6DDKSU'
'NGb0eOQHdmaF27JcOOLB/PCj8/kmCAFHyz23jiZJ+qNhoejy2U4tAB7s1cKHCS+n9q7RY8r+1cIHLfi4bY9K4SwRSs8QghzPSkW8ZZ0ZM8iHzAJ+mOxHi8cV'
'FtkShCTeZ9fbCEgkEqeV0WZwfwYvRzxo1LJrCUJS9XfrGHqkqDk6NNJJbSMA7HI5vwy3gVXh9tQCSYQymw3fEDc+3TXPjBrkM9QAVELbXRvipk8KGir3wrie'
'wREaILy4NooZba1+eRzqT60DF+EX+uVBm56M2r5ODp0e7fJIKWbDH1D8IEe0bluamjf53ZvbA4Q/2bvmmXGD7I4RN+NG5JHzFJUEOwGc7eZJDHIVrb4tTimc'
'+DXwm5Tb4+Qm5p5gy88jCbZOmwq9DarQpiiY11ltoCFJkbWrf0BSihvvRjh8t73bjB3kM+wAUX4wQuGSjMKmDg5E6TiuD2K7XfkF8lMnmqpTk1m5PmBdxZ4Z'
'gF3KSUuxJfdAGailcSH1i/VSqwfXh/fXx8fWQNeIU+CX3b4+M3iQj8AD/FhP8zCogrxWQLgLLnZMFIQzkPWu3i33cq8EHzS6iTWyO0tpNPgFAFAqS+SaZYNS'
'H1TwgKVaptd7PdFa1Me7zI0C5iT0zC0jqzN3zTPDB/kIPpASTX6EZ4Hq5QIoXWw0fNNA3kALsGuj1IfGQ6ioqDhEk7RWnPzsdrBzUKSTWgSBWaYXQeJr4+zd'
'VmznhNcmn8PbHSrKBp9ST6tRxqX7d2dGD/LZKILlQoNqiRuSbK/wP+SEBxsHtgP+xYrthNRwxJonID/TEjcJKWak1YCGY9KoU2TzLnFDWl3B7FeKniRRL4Zt'
'swcVtg2bSYTc0Mpb1pnxg3yGH4g9UqTeigO90CgNBfEyfnAMAnpu69UBCSuH0VAosyUaO8eDGdL7pAEoGql0d7mgyQY1rZaXZn6922MeFazU7w41CpDi2Exs'
'PvyWdWbAIB8BBnASYXAM8IsYNauWUnXw2pA2tU7Om7sTUQv1lgLgKRc0BkjuUH3Ff6XMUlsK+Hks86YkjTMu+l3IG/DD2nbSc2oxDm1GrEDrtzfxAm9mvCCf'
'jSI4KRxdjzoMCMMrjdKEAN3xgpABJy8dhZA7m7A4NuOApmqwaIAdu3EyAd9Re3FyQbKG5jDw129zpw6iTijkqC1cIPcqbm5ORkC9NrH/a+NMcIFY6ow7hRKn'
'c92jlQsblAG4KDfK+WEc0yPFm5RALOvDGIADsGtUOrVvdL76RYHPaVEngf8RVDAHCbX/dkJ9RKdG9Ax77hTABL0ZF0Aoz/GuedxsHnuGtTl0L3tSgBmOxEo3'
'LheyaB/jkYzFrHdHrqEb06MY6mGv3Z0EJu8gT4k3jE5zbDkhIKnmIeoY6UsNH3D4zTbugFicjG6e4sPjbdfmZ/OcwQVwU2Fwdg1gAaWrIEWjzeP2gBqwDvcW'
'+m25FrWxnTpud8nZMCPlOgcEXTejJtSBbTsSS1YQQd6Ix7dnxzMAW8BvqW3BNgKdcnkiOEbhrnXCbJ0jsABFozOj3JFInBXmYRRXbwdDB+y0mBfrZIO5gsGo'
'DnKV9NFrgiF7zoZRkaRaB+xTQ+rlia3De3Z5dvUOofW+LUYlo87f4LZl4rvWibN1DmcRkBmNwXhj5GddaSAcQLwe5DbrChN9SQuA+8fRRkAxqrk2oAM0pqxA'
'xUhqziZutjKF144cWn/55XoHNKy8IxoA+ahdM41SjUGx23lBmq0TT0evOXQgFF0prQUX5Kf2YzCewZ8ICxAaSsk6+qUFJ1Myai6HsidtYC1Eq1IPwe92XmUa'
'yBvl+LJsQUZmnLdpG3V4RKO3oZi7HXhoNs8RWMDy85g8XFsGXXrNqSNGF2kAbY5RtizmSbaxFioqauE3tcvT6Y7VSyBb19I2qdTLCyjmwZi39a/OjxKmlrYd'
'UjRo46YHR8ZLYk93rcOzdc4GElCmmAEWyAdOStaWTJpwtpRrtnNNC1BCD5wNbbvolctTIBpKwzoWZ0JLqo1r/N51Np46p/kVnM2BtJ23wyJobpvNYHzsPcA7'
'xsmzcc6wAtAJRtxBxA6qZ6PWHaizI1LChoUainlecoNDBRJMVnC2nCHyMVoIKchlVadFwOZRcbYyWx/C6y0elitAW82PiDZOztv53tqDv2Mfa2b75DMcNDSV'
'm2otMM6UrI0qV+CHgb1zWksei+lgHtM8rlhrBaklJNXR0OLDMbiQ1OnrAl9rSBtOi8np5ZIH/KNKmlChNs94Qk+qbe4E1TvmmQEDa86IVFQGeFrkiQCgvcKs'
'Tig0elItJm3R4m3kye3blusDqkLQZhIwk2XNILlJ5GOtgY0RYWtUmFr8cZrN821+6Dantq2Rrmdt8u6k16OEtPU2XGBnuMCewQUFahtojrM94F8Ec6Q4Iztw'
'UMy+hhWlRt8nD35oKomeIpgTHm6Qd1MwBfBSjMNSkLA2fo35MGvDi4GnCJfluAWpfXt3HWqz8W5KbWewwLqznA0V8yh40C7URq0AFpjOzMHMT2MXvp3msU0J'
'Y4zFsaY1FWfjgGkQOW0GfNug5jJ8nUFD/KZjOxrnQdDdjsZD/SxsdCXg2MjdNs+MFtgz6YIUWoe4G8tpYwkF2BiKOZint3btXSPvtoO960vBsiKhINGP/hvh'
'7qjSBRhmoajmBQ5yHXx6d/ZAKEEnbjsJ5xpAomQF0bXK+5Z1ZrTAHqEFmAjlMR1PlOG3FKG20GhctTcCNm9eNVkYkxb98gAoo6yFnYwyq3s2I2HHq0ColLc1'
'kC4FTzkd5g8QauNWi6tgDlgW27kEOPHbWcEMF9gzuACIhhtwATgz7BU9I4g8PId8URfFtQHnC4t9Gkwwmm9DGR7HYEJRc8wq6xBe16rNa6Qf7lWoLYI3t788'
'GGfZWCc4akKNt6wzowU2namy2DYcUaswKAasMz3I8IMZaUEyZVJv6SLY0jXtkafOFShjI0aiz9DMwZCDKi3BAT2wpMLUCf/dy5o5OHPZbJX0rGscSo3+Lp/N'
'mLvmmeECeyZzKD+CT3EMxLk+OP02pZaIRENvisB78opEqJsapOjlWG2qB53Tp7gEQZYvsKo3BVhB1S9AxUUvR57k+qiSCrWhCN/Nw0HXNNzFC+yMF1g+UwND'
'rjXo7nKVjKIGliI1aayeYZOCF0gQcU/mR4aCYtK6CH4aRgCnzaqaOcjcKxK2DsT5hl68SqjmIgC8q0epFVxak0dcfbzr29wMF9hDuMC2vlm9POj8rZRDOcRS'
'Hvspq7ZuVWYJctpzZ4SW7rBVsTY7aYQim2gU7muTBwldDqq+rmst8FdKHqC+gEY2dwcaNS5ugFDXbt0t48xggTsCC+CAgjFjGAHdSkWZGgTqpxxYgEtePVvs'
'U6I1pwY9TmO7S+BxQ/sDdOEcVPFjyZoMqXfHQU3b/RF06j6cp2Jt0GnY4gUMwtld+8x4gbNniQFjMmW0EeSXT0ZN2+gpayTBHOXvNTEgA0R7dEhtJ/VeqTnc'
'/GT9ExSCtMsDsV9jVPugXvf8epfHUiOW6nmbaWM0SlYtJWnku1ComxED587YH77Rbsr1QV9TCT0geeWhdQjMM/FKzgnVMbY+Avs+tHXJ2xhN2U46BKc9sDop'
'D2aTSjqM4HHF14ueMpgYt+O+kHPLdiOmR02c95Z5ZsTAnSEGEA9PQ54aKL9iHpx45wZjF2ibWds8DLLTyAxKI5U1UbCuklcrVC70SqXoAT6ko21OXnrmF3zn'
'LA+4pRy3gA6CjwmbaWzM+di75pkhA3dGMMDcxxA7TBE40ppWg51IIUyJWxN7fTtq5RrZqQ4AuyKcvwYfZ9ugVL1LUOJQBfU4NA1TjXfYePAvabYVMHqvZABA'
'ahN6INidb4eeGTFwh4hBKKz0lrfhI22gajtqUoRzt1J2YUI2o+gBZS1poyIGV49G0VPnepTUoAQGrSYt3c3/yrcdjfJgVnerRgmxE0+bUR4JAibcvj0zZuDO'
'xAxy19TogksxOiU1oKL72WdFHA7X2ib1hfLR5SiRE6oz2BhHcGPSiguspY/yxKRR2wqUT+FVhgFGl6tinYq3SXhxcaM9BSdz//bMkIE7hAyolYg19KCra5Uu'
'XNdorYcu9c7ZWz1K31TX+6BVExm7qrSA8zZuDwRPvDqDjVZLVq2TkTbxy6GntJKs2cKhATQJ3blBQe2+eWbMwJ1xDCScV7dVAXSwB4zSSzCFmN4yA0zmrur7'
'ALEr+DsQBM08IPfk+BzCtqXdojdJky6iUxijr7d6kOHYLW0Xbp93EwlSMEVzN6/2M2bg8pmGOCRCRzMB+yKSMkVauDI8xR6y6+0h25C0Cum4ouq5mifmqRNX'
'1MnVYR4IEdemw9omlWvK7vWdPAbvsCW3Jb81D7Tb8u2q1M+ogTdnXWw3DWFD9S4rEgZFPTMPlWrIDpLaKK1zHLWnTT1/uO6zwrjFuD1UPbwWeggZtBp6CIDe'
'8e3ZVT0xvdEouHbiaKush5n0aO62evyMGfgzBQNJBUwYeCgwHYXznqAbZe2Q1nOYN9IwHTeskww6Dho/pyyeGBwDdkXmXuvEweup1pGrzM6+2uoBGdlvIQMg'
'bmkzRQrEL7i7vQQ/Qwb+jGQQfZvp7ArviRV2m2TMdqjtRiw9Ir+K6+XG+Kx9UqAMWmKAObY0GnFQ5FOFDwuqrSYGkBmtGsYvsapBICYXt1tf+rCrAlaDVunu'
'ct79jBh4f6bxjumCQQzFniCjpG1ETcKx5gUJWPRiHeuajmTNEiB57DXN0IxXmBpxUZc+LJr/SVWqhn3tq1OkwHPSdicPhvDNJqnGREJNuG9ZZwYM/Blg4HvA'
'7xTr5NY2KSYVKiWmKrrG4veXeRFuK6AqHyT23OEqGmqxDKNbBzmcum4M1GmjMkPR8DPm9WVwUGKibRcbBJ1oNvwpTLia22nBjBj4Q/FDdC16zQOcMSqujSC5'
'PRYkwB0q4zxo2jjPwzy5DJgrnZ4wbRxDuz7prg2Taip/isA8NPnVTg8qN3LbSTi5HxuKQYY7vj0v4me4wJ8tSEgGE83j7riHZ82z8cMP2alc8v2ogW12sKdQ'
'8WQtZyt6/mPxSyplp1U9m28rTTXWbiTzKtgGvCgb+w3VqbiRzQHa526DOX6GC/yZegEoH3E0EqDfrvThQuqj/F0MF/O3y9Ik32roepGAw2sq79SnTWs3GH2e'
'vOG21fmkxTpSjyYbXwfbQA63W2pokrjkdxwD7I+1t/OCGS/wfAaGQgNnVDzBdimvpeJxY8yXCkrl1oK0DsX3aSsqhY1Cn6rKY30WTgriFDbrRRLpe3lsA85f'
'3JGAwRO3VTBwvsD2atrGvoGPd8wTZrzAnykeYqvVc0kCwBWblTFfVPUDLzBYp7Ni1RZ7P8YgqXitnL1G0MkFSf1hLLQwaps0YM1iUsmH2AFkXi558AntVsKg'
'7LLZdBKwAZJvD5KGGS4IhxIGHYKpdB0qWx3WgjS3LV+1kA9lrv6aVAO3y89RRbMZwrZ9grkvw/aqJljhLNikNrFzWz33GtgGlYW4n0lAqrprYkPt/q7ioQ8z'
'XhCOFQ/r6HRf3xfyenmwQ3yihsrPk9fYAwVJ85xJAOKhKu9CSTDYgeYkYAbqEovYtY8XSb3wVtfo+9htGNfzW/oU5gJpg1TDxwS6y9AJM14QDhUMwOAcgnro'
'aFtNOYcndpuEf005J5FpurB1PXldYKMITOQiqdP1P0JH966RB3h01JBq6MTauUn6nVgbQL8t9zBnbL31m61joelV3bLOjBeEM4YBZl2fukYY51WWLEPww+ah'
'FooOoVk3kVJusuNdCNlrmmBlq/zQl0Dy6lVmaAlQutQuFoHOSOj3bSKFsNF+UwJ2NqYdXlBGOu9m1WHGC0I4a5H2jkllDKWij7mKhaay96C1SBkjOXHdO4bF'
'iCPwtPOlTJIS/FnP2krvQUNzMJffWB7LUji0EeyrNU+Zdd3vTgqmyUBqno2bJNUt68xwQYhnvN0uo9AXkzbd28vdiS3zHqwWv64ZQYs0PukfqQi7rmlB6CBW'
'vzshJBUucG11+tpFcNDrfV0qFNLoe51qSIXmnbJRAFp+1zwzYBDO5A4x82VG1gYhHWV9Ejrbybh5t5Vf9/hKRMlubFQMoa8EvgLVNEu2xVAWD2ikd1MI1xpQ'
'DWoovaxgEGJzxWpOHS0YIBs4ByMat5O2GTAIdDYkH6b95BEBPziFO4VdYzz4BdSVDN9cHsdYkjCwNokO2jAcMuJKMa6SqKFPZS2AgWuA9nJ5ilZVeHUtXJmn'
'Sn4rRtkxIwXOgfrtbTgnzHBBONuPAKH8sRcO89FZsQ6mEGuzYQjpKNtIo28z7bXiqWq6CtjWdbFr4InIXlWpUFMaTto8z5DwfqniwdJGk/ZwgW0QiRJ50PW/'
'3SGNM1wQzkYSoEvIY8sy+mOKph4QsjguD5id2Sq8UAeq+iDnmML6WV0bz/If2DVg1EHfgC555J12TgivVjxgCSXaz/Nw25yiZG2YlrjN2o0zXBDP4ILkW/St'
'3jYUbY512grL3MMgf/gCs1w57zDtWFEOQfGY1b19oW387aq75HXNttSmbZbAU2ZM06uabSjvaNu/xiycj7y5OxIT/d3+dZzRgmhPhY3qUGb9SFQUIpa7g3SZ'
'Rh8BlK818GBEjsdEAhW5d9YmEvoIcy3tCAmeWo9il2tW61F6O6r4fRUP4N20tw52Sye7lXKtw9G3rDOjBdGdrcE2bQld32ifFfl9TNBUKaHOLggrcQo7diK7'
'sbpvzAJf0gJs7BmBpyzFTk4NPKl3Ia+BBzNIwb6sFlrks7bUHIy/mB01J3TV2lvmmeGCeCh4yEXisCOhUsworg30Yh44NaZc8zovgqlmN8AcrBLH1P7q2iAx'
'nee8wKlgDsqKOmergTkuvhx40EExfit4WFp0GzAnmkclSN+yzgwXxHBKCjWuZ21oX7qsrIdDjTHEQqk1L5f1cKElwV1gj502LiLJqhuM9zKUqoqFgrKZPamX'
'p+67eE05J6HW22M5EBcMbjupmG6TC+KMFsQztEDqkqrR1TmHSUGpi/iQHy2eFEvjYVl7Se1A1cErr0/5SqnSRELKn1LdkacXPCbrgnriWQO9uB8O+ek8SnUx'
'jgmNOafpF3QW2S3jzFhBPJtFSDMMWvYja2qHjpqQT68TlLATC9fMTRtjo6Z2CKZ4HpJtpTHurDrJY3BHVHKBb+j+a+QCdGZN2Faj6Y0yyFuQGljh7aszYwXx'
'bBYBuzvS8GsudSLGZUDeNrn5Gq5ZVTtEFB07eaC0mbVyJ5bNf2NShIag8jLkS5uNsQBI3yx9+d6lpAlLM7bmgYaG4+1S0nxb/CPOYEHkM8G21Ih3tQTDsVzD'
'Dlan2THjC6hM2XwZSutmmKesDtWANkC8gcflqcQ/vf+22TjGkN7gV+sdjCGG7SAPlkfbzQxpQLZ9W3UqzVhBPBtFAGd5CLkSpNMVqVBMuFl+SoWGbsO31ILU'
'lsZVfmgqg2JKDyE1DePKpg5dOfGKUgNOyzq1oF7x1/fBhaLitIs8rg3bacuwuc/o3zHPDBaks1EEZC6jxVPoml5ZaOXBJrBjFEEVoywzOTwWWoGYp3ELMOVd'
'hxkrZo1yWPVtWE+QojqlCG1g+7JyDqTOiLY7k9i2RUQKEIreKd/1bWlGC9KZfgHitxvrEdgWv7Puww4ljvzwhEWV1SJSOyU716NGW3ZZdOHCE6aG7qdO/aCi'
'F6bx2nLpMb26lUcceZVd1ydFbMPCtQ5cgIO9a50ZLUhnswhFldWOVfK+LxK7FDxmYrujyjbJrbJTHheoYzlMoOpoaIFpg9VVpwF1UdhcHqNqiIMH/WYA+7u2'
'8oCcnLYyu1x0TzeLX9BqMXdZU2nGCtLZJkUp6eu0beWiIJFMCmE3NRWZ6rBRKq9Xx6CnPHoIMI4lTe6QQeyZ5PdJlzs0uenFr7vg6mKnVzs8o6jVR+BCm2/S'
'QGpq+8FvWWfGClI424Cdm4x5l5zXSFOQn3T2qbKLO0/rdLykuk+QGpxrDSsoYk4+D5Hq2BORZd2YwwyhGndMI9W/tgHbxKJxvpMQt/vVFZKxMN3OCmawIB2B'
'BdhgEgcMCsZhTMokgqEy3tOXI2BQZO1dS6i0uSM5mECxaj0Kbz/4oJjMMuqqPmgqRr3iSbaN5b4Wdmx+gwdcCh7qSgxK2ClrJW/fnRktSOl0jCebYR2wM1Wc'
'LT9sGnxQrIZbl/KAHVILuJoU+K58dE2p/ay+b80mpQ7yXCNXKO03m14ueDAkTLwvR+uGR73gyd8hOZVmtCDRGZbTkeB6k7CblxW9ttwOUv3dQEpz2gLs59oX'
'wNqBrbZh2bdUrHeynV6OEhgpavsNA5LMr8Yd7PHeM93BZvdx00GQuxNvr4NLM1iQzgYRIElKPLICxHtle7wU7vE53IvWS1rXXIIhONTdi5yoxvvAb2vjAAua'
'gK+2uSK1UbkFo5b3D2T/AMG20LkQqn2K0KBOagPY6G+zpmiGC9IZtQC7vu3o8GCFESWFWtCH+aqno74N5Nq89gPMwYporw1fY4mzfSoXYBFgVlVZOLeNXytr'
'CvuY8h+hmpOay9bhnNwoSxol1GPi5a59ZryADqUL6OHYTJMiGpxT+Z0DzgGqxorwBzV0v1IQDXb2KbEHa3vHLEJO3LXGr/cnhDYXuQ74hi5g/RKd2oXWbldj'
'j/dvhrPfZgb8HWNwNMMFZM86cLksVX5uTdIWwsFqdhB2C6PJrWAbVvz4DhfkAtNoHTjL4GcN6kcdldVqHkhgRjUz4AL2vbgeoaiUbpFQOUN2k1Rjg3fbv33H'
'ODNaQGdih3BmnkYbwZdYqV0dS88Ny0kVnAKpbSxNQkEarObakCjmkbZFdIbUkgdsE7V5jUKgLnJ7dS8PdGu3rq2s1fB2s5cHvdW77WuaAQM60y4Ax2xsUwSm'
'ZrQN2J7bUtuhXL/u64PsaByuDbwjjpp9XG+RdD41q0UPi+epsntLauBSy0FeDD3gJ9ttJwER1ptNasAg9t1NrGmGDOgMMoDm4kgNGL0Vo8yQYmZhrBeJqJid'
'1egFdW1ATQ1Sf+giaiSByw/IoKTy6pxVsIi+apMUezADvxp6IBDj/ZYTCl5y3oQeKLrcZh3SDBlQPN1FWkuI6nBd0QdbrBNiE/Xo9UJel7/ABfqBVENcr60T'
'u8rpAckccBu0QtXNPGjfRhM2fZ6QX+e14WBupSWgvWI21gHnrS4qumWdGTKgM4KBqzrR3Tqx5CpL7IEQyWAdEsYEeIHbsKEzDcZuljBmsyZGabEFu5c95HPp'
'vmoDvpKyGK0LhyBnv72w7whug1T8Nm1DLuS3HexU9kjctM4MGRCd6oFV1k1V4McEgVL0AHc2zxlFsDVpzQy4dU3qzmVX5nOVCVJsVwgDqgaxRIUMLDdjr3t8'
'GVfzdbk2DmXfwOby4OvsMAOODQa7ZZ4ZM6AzzMBj6RcNpVDq09Bvs2rchCFegO1iaR2DK6vLx6wIM/VNfFeCgZvWLEM72UadFIpM3mykPzK9jLcBCElpK1KN'
'TsJuUyx2s93u8/CMGFA+m47PrcyvE+FStiSF7l5ku9KcVq8VadHW55G2oWzUtA4h3Z6GCjIUBSOrSqFjg/uKGKCFxK8S2yDgtK15sOQ30A4uyG1T2y3jzHAB'
'mzMhytwmhrqfQ+9Py9lorPGFTqhdCbvox3N+9q8x6KHN9xqMLPT+dSijjKpqDrbQ6IM8mCFx/g+A24BlbRXey46M5DaXx/fzccc+M2DA9nR/Rc1P+5Cii6xc'
'HjCs/KTLEtZZEQA48bmjHOI2GpzDht7sfgl9pGSZ8BV3qoLVEI+nHF+mtmHYqAr9qJCBzU03QJPN8a1cvWWeGTLgM4JB6Zo85+AgzKbtIuWmXNy3XFmzom1Y'
'ex/zGMB2oBUqvs3blih2VUcbNPpHLuReddAKnzLkP2Q1D3+zTeo3Oq6ZMad0tyLlGTFgf0qesmM1T6F5e1Lytkrn7RsV63j4NauGIIgZsjkQFM9WI4amNtpQ'
'tQxot5oHbdKsTmBLWPDnbdJdp8fX2LhxbYDnzSYvwOqX8E3jyNf58uunv8q7/0kx0wwc8NlcAqQfx7bySq1TJEBMH2Lry8iwmWvJEGJp1D3Fp1i7RRmtLk7z'
'Gguj3iLEABWzZozhmj+i5yNnxNC3ZrHJb4QM5BDWUdPJUtUw6v2ZMQOOZxSdTiDoMIbV1r+APzUa2RirV4SNICwQx5gvDqC6zTeV1xrNOCADVk2ssQ5AlZ0C'
'N9XNWq7fu50HS23cVo/SVJaYzqDybZvUahf5Jl/ff/73D/jG/5isAxFiHJf5b6h80OlvJJO/PCOX7fpMXp6x12fktF2fccszvHyesLxXWl4nLa+zfuble/Hy'
'3dPyjNzi6+fJyzNueS9e3issz5jldZbvla6vAxbG22cgHHZ5ZrVF9tdnFrtjveblmbi+1/W7o0l+eSZdzwYUZi7PxOW9ePmdeX2vvLzX8t1xet8+Y5bP4+zy'
'zPJefvnufjkbeX2v5fOwW757WN7rei+AFV3fa3nGpOV7Le9ll/dKy1nNyxmjsLyOWc7G8nn88l7hep6hgHr97rx8L7ec+cUn5OW94vI6YXmdxf+AAnd9ZvEb'
'eTkbYT0byxlb/AYm9q/fa3kmLPfC0PJ5lveyy/0yy323y93x6/1azrNf79d6B5f3isszcb2Dy2+YVp+w+Oe8+rrFpmE5G3Y9h+sdXO5OWp4xyxlb/XNY7/IS'
'm/L6edb7tTyzxAtv1ju4nOewfGa73Au3PBOXuJyX34eW11linDfLb8jLWS2x+5+SxP3++e8f/vNfP/5cUpSfJbP55dPffi3pzZ8u2cg1G7jG8GsMvWYq16zk'
'GvGv0f3PP7ytRN5+IL68YL68YL68YG0svQnOV+d2vfDX43MNQ1eHfnVg12NzDVpXB3x1JlejXgPsNTBdXdj1Gl2D7zXYXN3k9QD/Gefl0++ff/rw3+flv8/L'
'yXn55/8HceGmcg=='
)))
V69_REF_BYTES = zlib.decompress(base64.b64decode((
'eNqNXdtuHEly/RWhX03JGZH38dMaBowFbL8YMAwMBkJdsiR6KVIgqVnLxvy7ozojszKC1VxjF1vbza5b1slzIk5Elv738u1pLQ+XX/738jh9K5dfLt/K6/Tx4WH6Nv39v+z/+9F+go/pHy93l+fy+/3L/dMj/Wg1rviM8+y2ME152UrJULCgmbYpRgghFljdTHutrz+/78edt4en6RUCffV9Wv4yfSkv+1lfn56Xr/Rn/ATwyfzd8gOsoZ+8Pk+PL9vT87fy/PGhPL7UX8RP6m/79+6Tj5+Avv/648uX+8cv27SUj19/zPQn88mG6y6PP759/1mP8cler+BxnepB8frFOr1OL+V1/8p+qvt8/zk9Pz/9df+R+WSuX70s99fDwCfw193oi7/cv9IVTs+P1+/D9Uq+Pzy9Puw/pI+f8PLHfrDXr9eRo9Gk/4b9u+cfj6/31zHfIJqUik9LnNZiyxZScZAWD9uyxik7DEtJdgGbg7VLSpuZVjel7P3q47rSKb98/0EH+rf/+PM//flPH/4Exnz89//8V/cxmX/eH930/O062o9Pj3S+X3+jQXz6ngz9318x3MFvd79auANLW/po60fcP+U7MPsW7izULWTagrlz+6/MHaT9a3O3/wpo3+uv8fpXwDvA+uPrlnbGfWe011/TTuj5a8PnCvUKrnulemrwd4F/BXzQXM9RT03fp7o3Xn8X633Q765HsXf8V4j71t2h47sFvmK+QlN3ut4AuOte+7nqp+uF0ZW026uDgfV+gE9JN5T5XDySdQvXg9FYBT5mrIPg676Zx/V6SH/9CMdI1UPY+udYx2B/GvVmgG+m3nri6zNt+K/jCrlu8/VbOoOt41KPHOsxoH6d6vAE3jPyWAdGAR3R8qNx9e+I9YTAg12fRX1kdMd0GMIazdf16Vsyn2EHXKgHiXfxekK4Xtp1P3oMDS+OP5o6GuDrDbWHVDd8lzyssW4yDzLfpambzJfHT/f6CGhbT+vrvr4PxXU4CRupbipU4l3bIG8Dj0GuY1LxyQ+BRr2CxFwPvg8Rf4T2mWcOMgAaDni+Nexcr2nfNrS3qdRmRWRE8BPiTeI5YnkbGbiG4cQbx9tQt3Uiug4vX59m3+a6Rca3rxuetYknL39bTxUrjzDMU5+sbcbXDc+bxgPXQxB8eBPr5npAAnubwMjYb3MgCawhY61OGwa3vYItMMjatmEv8tRDATaFK0Zi+xHyn68PmxDmeHIFBlxmwDXgISOsbjx/wgo0Xzd1ktUJsuOMiaJ9fR0dwhnDrh4yd3x5hpdXMGucy7hhImEGDgpsjZgZhOgk6BrYkNHFjIzt63SKtqTAxozMmAsVW4ahZhlqjgmobZmQ6qkCT4dwkD/DDRrejMBbZNhpwDFDY900bWPA9W1ipB3IGxFnK+Kux47XgU13FXAMMFbOyhMD3kDgTQENWImwcXoDYBYE5waec022kPEFvDGS3lDSW6M1o+AWBdwqZeaDxurGStTVx3aQmzvnuAa3+D7ckNmrqRCgZDfkz8jKyqzWcMbwq0d1FbQ8OCOpIcujEawWGXfAOGtbxziLzHKJ8caMFhho/BHSwGwD4JxguCbnmfEWGWe8JY4hvL38eP69/CS8GZbT65izFKfroTIzSiWDEXKJITdESHjnR8nk2XxCcXVny8i0DBDLcHL1MCwaB+hwiJxauMXTlcBlFOb4a8tSykxXEVrnEUc3tOFP4BTBNe76G1gDpacNc36MKmussUMNFOTalkHWgroONo7b6t8df3Y9QOJN3c13WIUx3AoM9IHlckWbHSJd5EAZswx1B3ZzdVsFtMbqA8sFKastG+CvaTwE6FhXr+fooKs0x2CDFlJnCTp7qqfMegAKfI3uwhHFt/iSthZGPXV8Tn+AjwM7bCGcY9xx3NriV2gam0bSq7NoB16SoRxIyHVtZfbDJLS1sV1QEZxRSIsj4qAnF5rlKrcObMcpAkp19fxtZsCZEYC+JQEQJNuZI/MaSM9J2BkZylkhrZwv8pdeRnI4KqrpsVtQoVw+J7qmrJwD1cQhXDHXwCbSuBbbvYnlvAJdAxuDD61kOoU9xhpv4pE2QZ3PO96Y5uqRA3PnkSt0nLHmVlo8YjkmOlSpg3s/dUARxMUj6Rz5DyT/WaPQZhTR+bPYzvK5bf+MSeKtwY1VtaW1yLjDcJgMI9Gp5KEelaHWojqOdQ91zaNFkXmY2AiwpqXdwB8tCNBlGdTRfQrMuYq56zAx03Lamw9bI41pKkvPIa83QNeYzinwBZGqQpIpa2Uux8xX4+QDfi2DAKtwh0Mev+PODHk7pM539eRHEsFAa+l9Jzgvic6KJCJLpHWes4rnskSeyliTwp0M5lrG2uEXhuzhCOpQkh0Idms2E0SVS7QcYgztUlNVZrQGwS6uVoKOCS8z73Ho1jII3rhzbfWcs8YRcxVykREHg6EGnM0NBklWoVzDXlDJRDwlOjBKXXlrjcBcSyesSCMwnUGspascIfd8NfHuuWOLkRdEcNfzCMaOEYCLitpQZa9GCasRIZ1TxMdJBnqpp04FeEFYJC2JkKraaCzL5KEBriexUbBc3Ss1JTU3ctUmrVlIazyN5hTXKV0NzHGC4qohB/6weYeYriIDOpmZwSbtoV1HWz5HG7YEIkqTJKoEouEsnHOcZQ1NKqQTVNdwxs4m5M5hRtlyOFKb6foIN+w4HdRZFdSF0zQiq+zBjYZJJ7nm/eIIQXcnIjpO509COSfMuSSQ2HQVmeOacYxBkhwrKfjRmOdMYgBbC+u8pDeraE7BLgobGKtTct3wGaCywchzyMoaznCHkvyyckrCYHiPytpMYCPN4O4CZ5ZYEDzHkZwbnP/ukhwBXZBuSR7qAxylvskbDsxJ6FlhofgbYZ2whw+g4Q3AMcmBUXIaZTTnhqIJclo1eCXdnbNKUWEkuCOB8Iw6ziC0SeJk/tpMkxq4NXukwY1djdGfs11WB0f4wF1i3LnKdxV3V7QcIV28EdFFUXlQjknPG8JIdIdHB0JlG+7q3g1/XPnx/SOyG9wyCS+KECqPbfFcy1szh3NOiCqcquoRzpn38giv/BLt1THRcdoQb9gmYwGwG8N2YL9WdTiskqgyCRTBnPaHwyix3SnhUM4JZbWnWURDW0tVrcwe4KiODpkrKKjlCjVGGlSkccbavDg3QuywSZp6cnqQzl3hoDw6L5Am3eCeuLKwulFPDzu4eyRGQgykVSJLgIdH0vw3I60SlKlrLyOLzNXKBEJVIbrW5hF6HWrdJ3Eyg0AU/oh/K7EHv0VRfECQdS4nzTlUUZyVpYe6d+q2G7ObU05wC+dQwm3w5oSCilqXNQJuYCrcOG9IVVBTL/qCaSWI5gs3XmtaKUupN2sPaSySH04wROWQsBWMQeYNLaBD4QXbc7w1q67nDayoUXjBVuSr+azm1XnNyqxBK6hXBnGQaWoQwtrNODt2BAzpQlI+MCOttyH4oxiPvqUR8SxmAy9N4FZYdarS5WSC+sYFttIF5nwVYGjkGDLVhkDfrTmBuFbKTyPDVcQ139ec2cFwF86qq4BDb0TvXHibOjCxNby1/JThls+Kq3EI247aai+q6mzVjgr6Bm54XsvvPokK0nRimhSRwY2afpaMdp4uDPVVvGGOxPPEodEbC6yV9khW1hyznwVRz28s14I2JxKIPLbkHOWuqGxgxhybJElWHjTJ4Uhy1yPXVLmbIyioruoTHGmAiODeOCRZAa8lC61zxI25a7U5u0PSJbWXHRraUJYdGHRhZLYjZ8CxjyS3Qn5QkHMyhrMyaOuspryQDjm8YcolUdG3wpPrWUSS3SOgCvqtbUThLRydTb2y/6aw2tXUn8KNrREQKIvSEbEqgmvpKXII1ztFZPEhnsONiw5YO0iqpkKtrmaRo9qxxnpYI3CWMWhnJMjuEXBKUM3Yf+YOfIlGpZaqHhV9Kyr6VqWqLKjK+vWnJYeeKngJN3eupXiD2G64IqypEJTne54t6Jw1CieuBW1JNZA0GpOs1jPUII0RduTqz976ImZs8HtbzgelqZLYehSHvWtPII5LDmje2r/urLzVcgiOUQ5RRVlxkJw2GCMsql7acs39NUMvydGqBDI7tRJxXnQupbH4le6MShiczBREg9JhiqgWEv8u8LpN55UtkpXJ68aOkbeK2qCWR9P3gFxz5KzqVpK9JF1RZRrRa1z5tMZ1o4Okfd0Q2UjOjlDjUTs4LqsOEsVxreDAHSR5JLkWxrXwLUrIhVPrzaguuXAjcQDVJpcE5FBhzksjzo/ieuQNqCK5rEpcDXZJwi7eIDo4yx3eL7Eqb/hWnhoUFq3oWuoF/GaOcEW1AsEdHpw/j+KC0lVUFl0rNjhVdDgvrVplAMNpPR85a2jJghnakt8acdDqDrWeD3loXOoMh8olkZ1LPV9wquQQlS+inDircOdFSb/VufDoO+89JeGgPKMMEdEH/KZ5qWUQzeyV/sgttntDaEFmp53Ybihty1rjaXFVNmrabgd3nssSew2CKB25JCX21C3BJNmuhXXsyHUFVQWHdJK8HmXV5gSjqquCxJ+muzi6cnGs6FuRQfhRbY8mEtl2fqvYZSTV4WiNHIizpx2azftt3XJJdS1Z1U0Sx26SGisMHcFG5qxGNZN4WVq9VUz9f/ZotuJqz1mNiuWSjOnceXn1DdtFmTw0wVWFB9mWeRQelFcSx2USN/tHIMmSflBkl9835zTbJSGyUBNXrjokAbg4OiZHsUumEPGcAXW1C41KXGVrsBNMB6oRPcsKa1TOHEivpG0tyMDullmS31PYW4FduNEaLPNWL+kORqXtTrDszTyKXTCuqjmK+klCzp5VWUEtekiyhaStekgSd/GkOdjKpVVD2mreVLdOSY5LD3CSRbSifhD96NASV6c0VdcadFCXVVAnm0ga4FRbsFV5hFMNc0GUuZLygv1Yyz+W2iS59KGVVUEBTtZVnSI8Hb2Z9wpeR/sIqOpDVIYwYwxHH7i1zQ3tmVmCDWS7nFPBnO5cshJ2KDqXVEiXVeOSP21c8tKTM4rjogAdcgECYeQ44JK+ILek3JL4jluikXgjmkOUWURf3RVEb2ZbewPsCINslVPa2qK8Fs21JLatqWsyalSdy505w5rknMwfdGMwnLdo2vfqXEenHChXuOW0FhTknKg9NMhpf86cL8BpTAdtmZc5K7EmpbAqjZANJKY7IyDSidhtE4G5tvyBi6zVD5bdS61Pkz9aIy26oLJWp5IIUGu8GtHheYsmM1046Qg+pBVBaitKiVXJa1J9wWrpg31fWWWRVS71SqLkqnvRezSneuTs2D5ykJxXdYgsi17Mda1RqbskLX/IwqJzI6cdIV0zhK1ccBPOSvoKdhgFxSXZtaSa0PvCB+nOYVtQWJd34ZkhnFWTZlv4YCSuWln1vD8O+++bK6xLEUxzYmkhj+qgrC2HwCGVOEoPujkzyma5JqwNWV6xW1amSTpjuVsd6EGlrlZiThW/uikMp8seml3SCq5W5KpHESJKfU3SFUYZxoFql2sdwUFwHJ5iTlnCfQVEUtmDV9lDOC20oh3XsVpmuTyuY3WytN9JLgiSSzfW2aD0hhvmEFT6EIRTAjJ/6PWIKFtJVGk/nS2G6IWIdN6fedp6fhgl9swWzu/VVd8Kqyrtg/TperqgyC2qSM5KWTWjOXJEcsqlO22Y6yttWqNcc0m88oKtXOCFYbRFhsJXOi182XEZ64E4J1bbAI7Cqtbpt09ZJqzxhp4aiTnNc0b1aUbZFuzEWv0kg7kku+TS8W6FEXD5dFlhgxbIJpIgs4gGGuX9ap8unfUqvW2TA2WXaFO4pw9RlfS9RF1Wy23E8gcIZ45cPltJfeQOvKbQyralrJDnZBN6D+ScXPcAIoLrC6hlyor+ZOU0xnHpNKq2zCzrXp3V8EbigLKRSSHOj+1MR+HLyeUOIDnOj/HcsXraqYxVQU6RnFPZQ1Zkd0py4XR54c1mEllrjZLrzPkqm6i4LklRba5cc0p6P7A/W03YX+uiGpiAXTkLEnJ9HdcZvQ3Sqs3gIFszrYrnlLAGsZCVU1YvbDlUq6ed9IPD+4ts3qwuzKKHKYgeTScWFUKUutpiua6roAxhr5w5VtaGvu7MSUtYvSqi/zX+jV6Sv1F70OttjNJT834vei+AoVxx01vnsnwrCQZZAJNvJenVVlXoN6rQ706DOU13XjRrolzN2tfrW9WtKdc/YJTLvIZGJitbSloRzCqzRL0pAm7ILUpHWC1/aGV+VfuCk2b00F7v1FlNkZ0yhO34/qYjiQiyw0Tpq3JK/FnB9U1ncDxbhXOzM7inEChqEJjOjZIs01b1Mhy0ciWrVZacO1/6YOV7SfQSiDBkrAPVBdXRBOcvioAOQAE4rkBAGlvnoC7aRys84dat2RqE/Vn7Ujov+otu4bevwonqVThONAejlfKaFNOJ/pJWiXhjzvGawvYerCjsErX6BmTSqt8XoQM5cyOXAPkqHFTLp6MU2Kxa5/LIdUPZS7+cxMsswtzIV5UfDLyi1TLLSZdEYy9LkoO3vvDgklgV0ilDOIsSP0MuiXeTpPOlhdICTiqw84rj7CnHxZHqurg2NXUyg2jLH4JaOi1fTQKiFHHUWrNUVo0xq8jOnb19KZ0v0X+js/Y0osvnyGsZgxFcl+WLvpQV3Ar8vbIPqnkknDdrtrQ1yLTVn3ek93oXyrQVxKLpI23tvejhXV21Rujq9VdcDGdCHSw6I5vS4bSBLqpeYXeWx/bCV5JOiZEtm07qajNKWsemVyhT/lxrEQbVVOIV3OC8GqEWQsTztzHhDcazsodOrqLuVQgv7bmgCM6MXU28bvp4K0lW+USUNGffX+QFbVWrrHvdkFiUDXSiynpkEVZ6Jem0tm/h7C0RjDgjhdVJxMku4VtvwzldSX0krKKN7gjlpDcXZPeck3krnDolXVhxFNjcQjiU2SvI/EGVH87Xr6r2kjdmMJxlrT2FUC/8QlV/sKqw315rCApmQcLNK3gl9fYlP5a6jliuC+d5r7Af5PXtGohWaHXnuqpeS2JRrJ2GODSTyDWrplNeM4RRvSMCFbuFGy/DiVJTvSx6JVlrtWqplxerb1p5S7YJg1o7PS6HyBpkndqSXMNqT1cUqvfOqQWFb0r7XpT2w/mbcVCuJIzSDm7JQzhd59WLrFG94MucdcsdRa8o3rwUz95pKHLUcRXE29pX7yUJKmENv/32x93l2/Tyl89fp5evZXhp8MVt3i7rBGnBeZrMYqx38+y3PNmyzMWXFXCa3ZoKwDylaBfcACAAGpyz85f+0uELGueWbXWJvl7BItB+bnITHa1sdMRtxlw2yPt/VretyZTsjJvB0ZGXxV3kO2UvG4YVUoxrdDiFqfiYypQTlpxLWNIyuS3OdoXVOZfoutK6TgbjWmi/bOxyke8NvQR00ZYZi3PzumXjvI2bBWc3t+XiQ0puCmg2M2H2ha41rQHTvMWyreu62It8KyTdb7Jh2XLalgxhjnQcUxINVF6tnbO1NluTkkkuLCtARL/5uCa7X22qQyfe+ndBl5zfltXTaWnsJ7q14Dy4JYQ4LXH1MK3B0lMyq08BbS4Wg6dRhpDn7XqB4o1ul0CX4axfc6BhWRyN8poQk5n9PAdnljTN9DC8nTHihCXadZ5cmmcwEfLm1ot6XddlicbYeUYbke7IAd3M6nyKdLPbtG0WS9isLcHQkZNffZnnGNbi6FLnnJK8Qrc/YxrlhXAVFhvc5rJZcYt0qxN9aWmsIg0eHcg6mPwUCAs2zz6sjh72lD2ki3rRzmUxZYEJ0aPJYfNmSyHbZYZgt4SRnhFheSpl9tNiZ+vnAvO2+o1OSqBAdBf1JpWLJ3SWNfppjWUiuPg50octGRenGGyYCUdhijGbuJU0p22eQploRPKKywLzRb0j4wI0QjSlFkOTKtD0KjmUDWkIaGgnOkQxS47WL4HQhescZ5c3GoGypmUNC5aLevnBxcXgwcxIAN5Wi6bQTstGwwV0XVtZPH0IdOd+cWvY8rKk1XtnJnsdGycfSt4P6BFSovk64zxvGxQ/ex8J7oSilaZIoKcTzRbA0KwJkeYyzZqlENo2wm+KF7WI+RInAm3e0BaCLh1hCvSJLgqWnL01xBvrPNPTntHnGdYlxoTz/nwwTQlMuKg1qhc/r8t+O/uEMAVsMBPhg3YNYZ+9AelK8HoXmzU+bDSmNAA0E2GJdPXiKe8rEC/gsBRcV4y50P+jWRXpyWyEc/BTwVQ8Yd4R2mdiF7fQUMze2UR0ZGOhS7yoNWYEm2XeIEFBIrfs82IBM2whIxjiDcIG8Z7fZrq0FB3RjysFPNBEpBkARky9fQnRhbhhJSrcUqLBiduWPA17IfKhpxxoMMtUTF5nnzfCe56XJeyTDudEp4HZCNjAdaa4rVhvbFxIAMoSZ0NPhIC0kgg4AqXd7EK0Q1c8m1hoxmV6YHQqUgZYIE0X1ft/iXTqdfLRRRpAkpE0AU1RdM7OQMcg9G6OYLBsk5kSMTA98iU7oJNGQzcv5jJcZwqNelgTkOx4uiiiWmNDmpCe6rzQOJl5DR6dJ22JSKREQJyQGDJOxCV5hYvq171g2ec6EMfGhdRqDYUuz4dMcmGyK7CsNpmQ97fwQ1hJkgwRCIHJerCkN2Km7B2ZF/CkFDNJoUMaqjlsNP558gbzTMRPKkUDEk2c54WEzxRDZw50554EbWcTcct41VAbo5to1AgYNECkpeu8kqjkmcafRJguumQaXuLZbVtJXxeagsTFdEJSiLhcVCvVhfQXECOxvYs0a/NGAAuzNTn5BGjRh2VeNmKJhdRwzZ4wlOj55W3daIgWQV97s8xlZ7VMPJjoccyEGNoBN9IrT5Q1lUhbcKRLEzG3Xf1MNEMiSPAiQp73/15UG8Ql0+QCIqdUiFNpwtO+uzwGmvnECxQj4OYhh12waD4lEx2QOG00gRINQBJzea9yX6aya2iJRFsz3RPiWiZP1FgoYvEeiQsCQYbGcFoQCPszMRjxDAl5KDh5MVP2IualBCLnsmyRSHOlqCe5shFAHMSZhp7GNphlghJzSIaU09IEMTkSfGhm03SSB9xnSl5mUglbcKOLCMT0aRfNuHNDmEukqZNnCpWIyxbi6GWZ8n6hJs27SgcjCHavO1xCIBmbCIxIM2PCPQ6hIG0PQbY9RKGjEE+sOa0bqRSFEXR8XwibqyUF2IwIRXZf+TLTtzvbB588Tb8CISY6B4kJDeoun4ufk4uGwE48DHSNpPgrbEQqgea24MPdNbxMYU6GVIom2UxhZkqToydJ6JvcQlEMTQ9SG9xpgRh6pt+tRMH0q5SIvb3PF2UJ0RjimtcAeQGPhItdSVbirSXNfiWqpIv1meSAiC0QVxNsicMQM9HIsiv4RSX8F7fSTaWF2HLZdhUlQFiIpJUZKfhb4wyLiXSm4g3Qs90liiCesRC+gZ7fRSV0FG/ONHy79BSipZ1sSIbjzqWBwo2VIt89risUZJaFHhXNS7oNOleC1RiK0vZ/TWR7fvqf8vj55enH87LH7b9eaKbGuFF0lZHQQZF7oJDaOqLDSIej+UzasO0QpzmNNHfoZ/TYKEyjoG+DaaPLpFG6BkNIMdLqIkmH3f/FkSmvSyDOczQLiegthSgUFcWd0RMhMtLMQaDJuD8LO7kyk35tdl4n4glbrF3NtBFWZ2IGM1Fgbsw+kBRqEEvsYQIJ6JbcZunCdgST3s7EcQvNKU+nBEo3VlL/eaLYFCkM3RzsSj4RgxJQiJ2LpZDTEA3uIWfeIWbzfiyKlukpJU+wjGU1FF/5eZ8P9AR36lgpLNmQTk1R/J7iYCDuBNLjeSe3RKAtlER4T8egjIACPE/iVSh5oXiF4no/TYnYnQYXVxKDHCn1oUBlpViZ6Gcn3D3syiRnZd0iUMBP/DynjCTDxWwr5BgpNAazMxI9MmJw+nnZd6FQmlC8RzdAF0WnpdFdiWPStJOX22ne0XRbCdN7EO0NXV6ggaAg2eyPj2RqihT70wT7bf/XYJav97+Xzy9fJ0LBLvAUthU7kewY0uuZ5ipFnSR1FB5T5LomSlFwoaBmD0A2yqdo5JKnIU+2EPP6fXS+Pz/N5XP9d2Z+FdmKTDVkniCDfBmhy/BaxsYysJVRqQwpZTwogzkZickwSsZAMoCR0YcMHaTuS9GWiivlUmqdFCqpMlIiJL9LcpbMKmlRcpokpN+u/8bSf5Xl9en5Z3+mNXkXmbdIm0XO2/9dhM/PZSvP5XEpn9fycv/lcXcUlqcfj6+XXyzSeUtZP7+8Ts/0OYYMhuD58PD0188L7X6/Tq/l89Pv5flh+n755fX5R6E9pm/fH+4fvxBu7x/X8r3Q/zy+fvjxeL//m1Af/nr/+vX+8cPD9LM8/8OHb9Pr8vVDP1b9+sP1/C/7FT897NbGy/L0XD4+PT78/LDd/3dZP/SL/rD/o1B39L/fp+d9f2LZbx9ev5YPT8/3X+4fpwf68FzKh2O86OCPr3Tcl8sff/wf06xOJg=='
)))
V69_PROTOCOL_BYTES = zlib.decompress(base64.b64decode((
'eNqdW9uOHNd1fddXHMgwQCLdPTMU74QeZihKYkSRNIcyL4bBOd11urs01VWtOlUzbBoBbD8EDuCHAHkIkvxBvkFv+RP7A/QLWWvvfU5VD2UjCGBYw+6qc9nXtdfe/Qt3cXt276cf/9Odho2vu3LhKj8PVZy4sgj4d7dzP/ShLUN0vi5cbPp2gb+bpSvrZdNufFc2tfvkE3fa+a6P9114v8XjG7zrK7etfO2WbfMh1G4e8HzA92HR86WZe9q4Oly6TVOEyrUh9lWHXfBMWS+qvgiFW4c2zLD6L37h3IsQg28Xa1fG2NtxwgVPuQjOX/gSB68Cj/IlzhXaaudiqMKiwzrY2S2azbYKclycvlsHhwsXvsMG7u6hc18HX8zcmxDxbwjlrnPrMnZNu5sWTY3346XfRrfAhRZrX69wyi5sproFF40LXE+PtSxrXL5pi9BitReQbFmX9YqSjHxWb3m5DjhFqwKflnXRL3BUj9UuKHa/XGLlKB/wlW7dNv1qzZPH4GoIDUpKa9hR7ZBd6+u4xMc84qBIXMLbWTu/ChM56+W6pEzrbd9BXisqDpfso6+q4Qy4dVFSVvl41Ir7IsRyVbsqxMgTut+cdg2EE2lGD9d+A9ng859+/NPxwyfuxuGNWz/9+C+/vbbuum28f3DgF7CObt1UzWo3a9rVAZ+Y4dNp1dSr2dHNW3cPrjs335nVXTE6PT72TIZ7wkN2cbRB+768kJXX3aY6uHH78O7s6PadwzsXR1g3hq1vfbYHH88hlrZfdD10A5vduAo660u5zaKnXbyE0SzFuJzfbgO2ew+7mR25dMJ51SzOqek21DxnVX6ASbTZAHzXURdN/UAMsOK/oaXotnAZ35YQ+Tys/UXZtDCgZN0z2GYLddGKHRQHDUEQePNCF1Nzgivh9UJUQjuCcOTjpnNF2UKNOMe2KhdeXxEfaGLQlWivm0DDLuMmJpf7wnd+4k7xVifrvehr94K2H/HAFN+Hi1A1W9rMfVhXuRK7X5bvsT+OGSBeHvls0dTLst1MDw9v3L515lzX7H949+7ZTz/++cZN545rnKcIlC2vedm0FZySez3Ux+kHHrvtvX7v4zU/u3uTa967/ffWfIkb12J/0OCmOQ8IhHtr3Dr76+//sPfJba7qHtfLqodqqFP4UidGEtOiiDNwFXgnpI4gAEv7gQ9bcFj0bctzdL5dhc7BYvygxGWyGiivc8HDNdMDDhFs3bTQ/dzDcWBtl2W3LunMkLOfuS/bED6IHegdzfGXJaLQREIB/qO7RnUehIkCdo77dW1TxRSiyzpuGdRwub3YLPc7hkROnJgCtI+QY3vVYtE54kqEUY+puGVrmyFgdhoqXexhwDv14ofOnaoLPR4ZtvsO9vm8DUVARMWip/0cr993J3rOkfkhhDv3rIaT2B1MZ14zjRPb0qPS9eBx+B9dw+MYXJT+sFg3JbU09xGbYfsUXWHEfBZ/47qBobtlhOKpeUHctS6mHlELf3tmGVf3m7l49mzG9HJw4ybN5mlDPdjWcqqDe7f/+vs/XvElt0UMDc4dTW7euu1S4gow8aZibP8eIm5ySnY3Jjc/u8HrbRv48rSpEQa2bTMPD/akYKtic3drcuv2XbpUWjitadq4M7nNjHisy4idQVDL1rIHd5iZxzhGFVqlhdNQ6RrhvaRwDaSashZNX9OIvoUtlRQSToeALbZvpuzNmlLgoi2PfVela472RVh6LAW3f/Hd03enL4+/evT5p8Ugxk/hvH8UBe2JQSyTYvBFs6WpIsGI6SJfOe4a5k1zDit3pzgVHBCRJX/f+Tn2+3S84N/aJoiiA+ygAJSZuS8auVJPHXQx4x2sDrtjIF4DgNBD1/gKWZF/w/C2TTQZFX1LlxzdcOYeK5LRqG/ABDdbMq2MHpxAk1AB/Mph4QFrMRy1QfK6yBoOHXl41fh+3Lx6u5QkvmWIwFkf1Rcl3ILbxZQtkAKYKujDurs724TOT6vKb/zBE/7/9LPZ0fTuyZk7OT59RCd5gbQnh3BnxeHNcOvejfn85vK29/cWyxDuHYUb4cahX/o7d45u374Tjoqbc8bk+bJqfHcEd/rDCoGw2HEtHoPy0ZQUm2V36SU4bOZIVQlXIdIm/NT2NayLYYev5ZTmjo8OD4kSvzqxTLTqWwtv+SEqTp1UYznS0Toszt9h5cW5JBKTwjvGYs0sIvx3ghRxif+SBcXNo8BpuQD9ClKDRi+QGgpVQNtXGVk4HzU2eUXLGpTK99PYha2b9wXzjGFqJP6aIFxzPpxV8bgm7B5RFvmmOe+3Dih+FBdm7riqNFQylxEiVnZNS/TqSO54ISmfuSh9w+PC88uK4LGHoOTWhrsUJE808XWScw7k/J5alGi4BLSPdAWqJN0GnvV+DZja0bVeO34gqRCi5h84nRPRDOUI9HqMtapq7vkMkXHRBM0CcyTqDe2iFaAEgBcEihEIC+4UX0hfqxottUDczxhxJdDBNoDmUUvhivAC2LJHOD37+st3L5998+ipYIe//Md/I8lhxZ386b7uEVKeNLChnLtwyJL55SmOpsgSoaDvANGR3yC8c4n0FIl7+/i5c6/aslOLLZpFr5EhRzKXIg+XFQwJN8LZvnr+0h0/f8z0Qz8+ZhH46L3VNhIAustmeul3VhFSDMlNcOlvAmyLWyY80wYpbybu9cEbp18nzEGDtlcnqiPFH9nSAwxEd0Z6tiNIHhMvlBxsy8OVAyLYPHSXITBGILBdlOFSfQqlxsMqvEeZoYmM36IOAoIYvj/itxbabRlkcnoq1m4jRfea0V8/QrqG5PM3uNkJzjM2AbErLaAibE4yxl5YwCWnSKSbLdBGjEgIuBcC7Rxvb5gnA75YinnDiOV6s4RwcrE84RvlEu76Az2oCvVKInRBAANbQTbFRueUyAugfL/QQPBIENpLmOSBXmFq3j1KvbBUOgoMCu453nEo+oDqltMtr4atJhaU8C+YUcrvpiUxFYgzvypJzYl35UdFZQ2iG5L9Oc+9FbRhsfpKPF2h6kVJq+gRkrt7mNItzNMqmwyfN8EDVTLUZzRy3z08olAWoWT+oXk4+B9jkBQ/Q8HNMCKRp7hAQelXISaiANoDLowP7O1hMaz8yjCiLSOoDCfBIaSc56J5QYIAUgmCvcOi3JZ0HPIrO/UV1ahGyWY+mJiyIgLn4cVWey8oHiRU9zli9qrsrr2+7qb255vrWqd00fIfdA9hoUypFwaJT7/5LpmMvhQp2CdU4EAg0C6lCqEeBdN4mJTIpttTAWV+6bdaS/jYCZDB2vnx5oLLyTnUBhGCJFI8EouG0tfEDfr4fdzw7OW7Y1wNKq2RQn3b/e43m2sPj/7yz/9KLVyf8h/Xf+v+wfHj9An+ywfwxT8d3DiT2JYqR8sHEMGGhkmnRd503OaamCClx92u8RO1t7uH745ms8+uXxeM90KR7pwBYDC8iX6wd7WJCk1hh5VXI5ucZGCiXjfVMkucBysvIXmt+y1yFyWZgARUu4bUmj4chKAhiDQY7nGFojTAH5A5kuFJCYUz0PN9u9PEC7HH89GOEfCUKTjj3azcpFezXQkZfh6bCtghI3tJJCdMJL9Sm07Mk++GJccUlKxWcVmJ00LlUWP7JTJLltc055c5ztDwLCPkwnnYAtlb/WZdbhm0Ec7f8v3TcaHWb7dV8lb3KgUVLdnF85kuNQblnaT48z0tG544c28O3sKRuKVlNd2+qcjWiEFYIZg87QsTdTf7m/nlWZv/hcWPBcoTo04YSp7i6A/U4JQolUuL8+RKIRtXDZjnBAOWVmdwR8IfzXF/dF+PaMOtKb7V1FM3jjQcNk9VBZfEY22zbUt6kCxv2pTYDtlxB1mn+QglHRckpQjIqnJRdgjldVNPNd8Ic4vwhMy1Q96BfTcrMTnjzaiwZqFIQ4lA8S+KS/L8WyVlmE0VPNAcBgAhpm6xHJYDNTUK+zU2MtkM9mflHlIWvJLGS/PDRoK2KDQGVXhyl1ZMtLQZO+v6//l3FORbXzIVaW2GUo3lbCnWu/TzlkQchUoIBFFtYa0CRgd57yGemIETDZ2QQuBc8rzTq8jcXDS7xEiomvhnxpSrJCXAJjZL3TK9IdKb/D9hgvLRdCZNaprrEoy8XJfQ+zmgYgo6adMMB5362BIo2KWlDRPg1gPnoMlnfGSpB4RoEhYj2m6srNNue4fSYA3tG4gYgEHOhoBvzaKUWkcA8kZpqlCMlK+RD4GrHyph2wci7Fskc6Qgqk4ttWtWAiDEKN2JsyjEAj0ZmGra11qpWBonM73eQzpa+iEwNkZaGY7S5LLwWz8vK0UoW03tG3wrhMt9i1bb9S4SYVio0KirSH1sUYOzmA8FPJK9oe40eWWncU4rZTFElp611Pcls401ZwT7kujyyEis/SNOsPHcfu3pFgpAxqz3LOXjVJbtl5JSNkKNcVRqpI/GZBVuRlQehfYrF4NlpwuRF2F3QDw4V4y8ZWvG+dh4VuNps7k0l6h/M+LYR20/ZNT25vpU/3irKONK6hx6OdFQ0UlCRWZ+v/vND9e+/vwtcM/Xn2Mx+YcCI/5pn7yRr9/+H3DRScZFJx/hopOfxUXPJfEAEF/4ip25EgGB0W0kiaW5YobCkPulbwsrWhn2tM1YCBCeGPtH1K1tkS2qYOEvRStQLx8jKxEz9GF+PNUmnAKACwnmtKogsS2F9JJGMSRDq/deDPkq6XXj5v2Ob5ZSFR7g04NXB+6Z+NcBrTL1+L4QnIRzIr5qFwlIzBKAhJcpqbU+Tk1nGU89cMwOYjz0p7KWXJejNA7QJe7Z/Howwci8iSiArQYIl0tjPLwxSPZQGrxCJEwzotvrIE0kLzfSsJHy/pg9KTwtyUsinrDYLJ3kLBI26GtV1VzOBE2/ClUKHCM4Y4UxfVBgS2ovj1xw49vznAAym0AcNsnFv/6Lq+TkoZTxKI6nhCldOJegiRSW9hKONCqrEuaTuFQLirMGqK0b3iubH9dNXxXGFLlNGeUSFZs0ce/UStbMDO6VHxgivkViYpDRMyYgmMAaIeOF+Ay0Mf1A5fOSUu2ngDJqekrP4VUwenzVSFOx3l0StjnIlXu//rgdOcC6n7GeRA0jl/QLepyiOW1gKjKQ3ic5N3q44Bbbgc2pqvxgDVltqBQBlrQTSAfPhA8MnKS+ZMAuJoTWCN0qIRsWRIsU+m2kYLg+mw+S1Gm1rIFVkiRNH1mvQT5AGmmbKFWML773LICm1ocj9aXsEVATiqAy8ReDnKCni6akXsUtx6WQ+8fTZ09RKpcb0vdJj8fR+q2Ejci9NiAxA4y39JvPbu0ae/EUapcSI259HQel70bIlKAnyQxSX1nTpxUM8D05wAepihx6ZoouMt1rd1dSVXnDkWBVHgO2WFS+3Nhtmh7AMINoQHMCqtRsh/8XCuu7EcOAHKJ0ad8hpm92yjIklFCEhTL7g/Grq/p0S0IB9h7ZtCyMO9KCM7fThx66UgpDc51tAxpd7piMeajEMn0pKM8an8pySokXydfigUT9i8qbrfHXs4GYNy5IjLise8XsG5uWoIXShfdxykOX52YkgBmVYoXrz0/HPICnij/ROdlTReDc0YY0dwydmEJ6v5988tAlZ7RSDYoqt70OxZTWr1blJGmrWtIwyQPWojk8FCmhWRN5R2YBcmZIWJTtoi+Ff5Y2X2SmzZaBbZpC6Gvr6K7LAmtoZUGGFEtVLWSwS4Fgb5RIHFKjflakBv7RcAUhcFsiN1Kk2R4ug5BIM2vGWXOTZcM61w7I9L/65uDXzj0JdMCCxP9e4ewLzbxCsUDi0/yt1nnkSKSZMQ59wyBSeI+j6REiWe3cXsm4NyZ3IwNB3DJOICPW/tHQuciXKVLT7etMx9UjbqaVikJGM4aqTIabKBFF0Aa6kbh85qfgy2yHBYvgHePSLI+aHPNRrUhYBk4jIWwd0gzF4PxjvhZ2lqYL0kms1WZ2TeetkBpxOlahy77a721PBJdCZoqLUnUg5ozoJHJWY1C96GmOiyLmhq129YeRBJk3QOhmt/wrHtOd7lBntk0tAxcv9Lx3D3/68U94i5Dnzu17R4d/+f2/4b832TdgPWId/h6gDUY75CnzTxaQ0gQYNKBF+oRSbS6FzrC7S/wZ5JOsJs9jpNaaRmEl8IXP2atrx5M+FONUy9+0igom1UqDMvyKOdiGTERriTgqZVjj3OUc8T5LTwVKA39siThm6u0KcZ8jXAazJLVry/f89oFoQjJD8ueh7rJn8pwGyR0yD9uGQU1DX4Apltv0ko6jpOg6yz0Cs8NctEaUlHT9RMWpRqkXy+1JoFpXXK2xp3JqLq5CM8FAGYwB9BUBINNoc4FWRupgg4ywpZ76Igz5YOUZ8nRGbDRuAP+HIyIgdJzy2Uo5Hveweo9lWkZR0qwQMQp4aMfab/nkGlblDDogArdJt76iH1O3KPmVTn7EjU/d3y5NsAgT+F5rRt4iZ45Vjx1hHKn/m2MkwRWW2RKd0YtzHkxBbW/0J489af7H2UbtZHWg3f5ogQ5tamfXGmjWexTKzgPidz3OMx061lBgJ2N4krdn8hySPtsjkcuc/e5TLvfp/U/jeT89c9KxTY0q4a2lhWuQ7nId6r12ixmYMkZ5JIim8r2GJH0z1SGX64bBTSguaQVKiTH0haXUydlOQ+K8TMN+4oLSJN7bA1c5WyOhvPvAAQOgBCKcYRUmx6gUCjnIwlCwlti1NB4yRTbLGjLswH1SdqRVTTSikToWkbbaitSGNGeEadPbHmXHIlVOQsPvTN407+MBCJfLNC2livs40gi1hZd14sOxlIH5zpDcOyOqUpMth9IlHq12D2zxbLW2hXiGtxbuqPjVhScpeSFwwumcYqA2SI00ss9Z6o5TSsNslcVTDc/w/2bDrDv0A4nTGX/EA6e6ZeJN8ATqTnxX+dVePLLAPs4Ik1RsDieafDSXvLfI6Zi5dZteGjJYjD3MUZ/KqMTcLvxDHk4WAwztRjZQpPeAhPVWIorSboo5Os5cCseIShyLslVCq2EXJJYVdEUWhdyommKFG0jQhpyKttE2RURZh/qtyaHjNA9hTlIVi1TIC9OibdKJZNJoeAYG+fjqXKhmLDWdHCCR6lFoGUOKelYTxApGt6JFjVuBid240g38iE/PE29ibDaS+nCM693xgZHBzGEbxBqpIKVG2Gr0gJ/CHoWbujk5PATMe6oxhikHmSNeyRwZ0xze0EErsRySTCRmomQvd+/O7NYvnXuee4e55HbXhobidXeCw4aW2AlFld+QoxMKxd279UsgJcY05ckUt3XGhrdhSNsX6ek/s7Hu20QEPVA0qCQLFZD8vrg6IylR5YUS05r+x6z2TH4JsEcFW7vl+6ywzOHZWIM6HV1tGK9Svy3F02UwVgqH7L5sRAC9LLrMn5CIKtsQR/0l2dhaKR9NzCzgcM1Gvio76w5rCaS9nwFUSgdLyOhvATeE/BkNJmlkYZ69MugkA6ypLRKtrlwwi+5GNDpRqsj6o67wwd59VVKDgQirAAGOSMiytk+12rCIQU2nYSRNXYb0RkguKlt0dTKb6ECFT9842WMbr/RQXzsy0t1U+fzRzz7YAqGAH7CXkTgERWd+DmFMR/0cS+RqTTo5az/A4FYDoEtDI7mBphRLKThw6GfYNn+nJzOT3u6oS+m63VZHUOSWNuDC1DATAShHnfnLPN2bp5cTpR8HRknPrB0V8j3AK2lQxQ9VnnnVMOzQDSPtc51+Mj5NHedqgphY12roC3YT+7kOXdTwuGB0LcPmOyEhM/Eznk7y2u9HQrBKbi/+O2U+9/fbo8OUgoz5dxWQc7gUAaYbipnI7z+0VZCBuTWSYOnyFz++0jgSrrCTcUa2JRXv6lrwSjM3ijazI3K2PN/xCAG0TfO6UuLkuUyF4eM8ksLRGK/TU/eFZNQbx6sYnlnKSWlpFOzGA7P3iAhbW6HoDVNL8YO0VFVabkr2Tkwk4/GciXidyFMLU+758YuXj4+fOKYliJPhJPYbisPYNbNPnVdpQ8/AaoOsoRh+WnA6AgvMbRkmMCRXJHYmVxvMJAsFoDG252EZXxRXZpV1aNzl1roRi2m8Uye8+PubXZqnkRhZD50ZnbQep5/Yr1YyEjJAFQvNE4GYH66OVAvG6+u/N1WdwMwjLhikGpxIIp+yq6RC1doUApDozxE/AojFz0zyHyR+oLVxtRGcRxjfMDYysFictiF/HjELNbNLPGlR8sIz92vT3UDBqGGE/PsRKtkksqE9fK9Ds33M4W1kEbloWthYteFVjZCpDTpJPGkhP0lqBl4cj6WJiVmCAmkmcnrJwdjWKcmMu9SMQOIilPhCHK3zyC5CJXv4gDCZMaYBiRca6TL/pqmv8jWTBHEhf+VDPnBjP6f7CHlPhX6UXOttMEXDIC4IGDpL6S1VlR9hnEzZjUYTWHrNOML7XZoXIC2LpDf8rEXjR50BIfJWKzPS0oJl8DlTjz3TEX51+jzbpBSRDHoy96jSLgaHFeRUC6DEKVAKjMaYcUUbaGt8kSOPWm1upiXu1/SqbULhyBJ8KsY6nihdMHCVkb9paLpm0VTpx0yXgyUeuOdWCg2t1ETDaRJkVsilgTtTrice6H9n3yNpS2/7TfDrmY1ZW1nqHr54yNrnSUnsRpp9CcfgJ/orygMA+YKV7WYjYVIPNZ3vphZBs3/ZwYMQXV3qMKQ6aoaVrowWCPtmjE7qmsogHaotC8Qyuqwuy9RcdpW6cYZRo6KXNV7gwIaBFQkhIwzwLDV/FuTsRj+dyQ9L3df2IYno1yoi3N25L1pB+DIDL1B0KMwSVUAs5R29VOi35Kbp1WfKVFUyqisBqrDf/EBinWpPUmdMPK4VeEQ4gm5GtWDyU2r1eG7dTft9RVfqeI5hhtGjTzMfL4cR0n/0OwyLU9LuSBIW0syQ456LwqaM2mk05MGtgUfsRwP8vQyk/r8n2Ddw'
)))
V69_ASSETS = {'inputs/worlds.json':V69_INPUT_BYTES,'inputs/lineage.json':V69_REF_BYTES,'inputs/protocol.md':V69_PROTOCOL_BYTES,'code/patch68.py':V69_PATCH68_SOURCE.encode(),'code/routes69_tasks.py':V69_ROUTES69_TASKS_SOURCE.encode(),'code/routes69.py':V69_ROUTES69_SOURCE.encode(),'code/routes69_reports.py':V69_ROUTES69_REPORTS_SOURCE.encode(),'code/legacy.py':V6_LEGACY_SOURCE.encode(),'code/tasks.py':V6_TASKS_SOURCE.encode(),'code/study.py':V6_STUDY_SOURCE.encode(),'code/reports.py':V6_REPORTS_SOURCE.encode(),'code/aligned.py':V61_ALIGNED_SOURCE.encode(),'code/confirm.py':V63_CONFIRM_SOURCE.encode(),'code/confirm_reports.py':V63_CONFIRM_REPORTS_SOURCE.encode(),'code/heads_ab.py':V64_HEADS_AB_SOURCE.encode(),'code/core.py':V6_CORE_SOURCE.encode(),'code/runtime.py':V6_RUNTIME_SOURCE.encode()}
V69_DIGESTS = {'inputs/worlds.json': '01dabfe718dfb509f4651287e54c04fcdaf7b4d5bb361ce3abc6749babb598c8', 'inputs/lineage.json': '23418a8e0707568c022ac92d5be35655180b33c664f9c32f318cb06d2a742759', 'inputs/protocol.md': '7fa5accb2c50c308671cbbdd4b65c1e52390f25c361b1d3e0ba360a1b0a33ca9', 'code/patch68.py': '9f899125f3d0292a7fcbeb99ed66943f64188ebc20c13909750b93946ecfdbd2', 'code/routes69_tasks.py': 'd5d7dbaffbf81ef15cb762bf7dd4ed62e8ec342a0c6f0a60cdd8ff5e10fdf5de', 'code/routes69.py': '45115caffc78c1fd2f52a8321c1268de89f5f2529d678eab6cfc0cfd8a8fcd60', 'code/routes69_reports.py': 'c527fdb1335a490633029a4376e69356f437bdb13bfb1e4ef78c62d45d203a72', 'code/legacy.py': 'ecc77fb8592acfda16f19342d276672567fff0c65fec292f92abec4d4b5bf1af', 'code/tasks.py': '4e9f82772209d475ea35d57a9dc6c4044281a63bc07c47a574848972e0212f59', 'code/study.py': '3a4eb45ff3bda20c3e33d0afccab8cf0ab3d00cf32554f56667d374f84f3fec7', 'code/reports.py': 'ee2befbc0ad54421ecbd9b1babb62a48f412962ada77454e1e3114089dd45890', 'code/aligned.py': '39fec76288d585eeb7ed01f35b61bfe8f2f51d3bff24f8fddbaa0266ff1c31bb', 'code/confirm.py': '8d1fe3d1558d54533155635e1abf239165aa84277662d0f6976123b2d3dbdb4c', 'code/confirm_reports.py': '88fd9dfbedf710889eeb89293ce0fd1977aff105e7e2ac04dfd9efd9d03b9c26', 'code/heads_ab.py': '126656312fd40a8ae7e54ee6e45bbd9c86de45053364e10d202772d4aa716929', 'code/core.py': 'dfee0014eb1746e959d1b652d152093f00ea90782932900bd2123ca5afee7c53', 'code/runtime.py': '02759ddf3a5c525a0f22d558ff2b4aedf25f1afc3fa09987a3f8bc2ec51b1bb7'}
if {k:hashlib.sha256(v).hexdigest() for k,v in V69_ASSETS.items()} != V69_DIGESTS:
    raise RuntimeError(" Inconsistent summary of built-in materials or codes ")
V69_INPUTS, V69_REF = json.loads(V69_INPUT_BYTES), json.loads(V69_REF_BYTES)
print(" Enter verified: 24  Development, 96  Confirming, 2  Technical pre-examination; there is no overlap between the two formal phases. ")


In [ ]:
#@title  Load verified original tools and new experimental modules 
V69_NS = dict(V6_RUNTIME_NAMESPACE)
V69_SOURCES = [V6_LEGACY_SOURCE,V6_TASKS_SOURCE,V6_STUDY_SOURCE,V6_REPORTS_SOURCE,V61_ALIGNED_SOURCE,V63_CONFIRM_SOURCE,V63_CONFIRM_REPORTS_SOURCE,V64_HEADS_AB_SOURCE,V69_PATCH68_SOURCE,V69_ROUTES69_TASKS_SOURCE,V69_ROUTES69_SOURCE,V69_ROUTES69_REPORTS_SOURCE]

V69_HASHES = [hashlib.sha256(s.encode()).hexdigest() for s in V69_SOURCES]
if V69_HASHES[:8] != V69_REF["frozen_sources"]:
    raise RuntimeError(" Model tool flow is not consistent with historical frozen source code ")
for index, source in enumerate(V69_SOURCES):
    exec(compile(source, f"<v69-{index}>", "exec"), V69_NS)


In [ ]:
if RUN_MODE != "export":
    #@title  Load fixed models and connect to this breakpoint 
    archived = V69_REF
    if not torch.cuda.is_available() or torch.cuda.get_device_name(0)!=archived["gpu"]:
        raise RuntimeError(" Please.  Colab  Select the original experiment.  A100 80 GB GPU， And then it is all over again.  Run all。")
    stack = V69_NS["stable_hash"]([V6_PACKAGE_VERSIONS,sys.version.split()[0],V6_CFG,V6_CORE_SOURCE,V6_RUNTIME_SOURCE])[:16]
    candidates=[]
    for name in ("V69_PARENT","V68_PARENT","V65_PARENT","V64_PARENT","V63_PARENT","V61_PARENT","V6_PARENT"):
        previous=globals().get(name)
        if isinstance(previous,dict) and "backend" in previous and "MODEL_IDENTITY" in previous:
            candidates.append(dict(label=name,backend=previous["backend"],model_identity=previous["MODEL_IDENTITY"]))
    try:
        V69_PARENT=V6_RUNTIME_NAMESPACE["v4_load_runtime"](V6_CFG,BASE_DIR/"runtime"/stack,V6_PACKAGE_VERSIONS,
                                                           V6_CORE_SOURCE,V6_RUNTIME_SOURCE,candidates=candidates)
    except Exception as error:
        cause=error
        seen=set()
        while cause is not None and id(cause) not in seen:
            seen.add(id(cause))
            if (type(cause).__name__=="GatedRepoError" or
                    getattr(getattr(cause,"response",None),"status_code",None) in (401,403)):
                raise RuntimeError(" Please.  Colab Secrets  Setup Yes  Llama  Weighted powers  HF_TOKEN， ♪ And open this book ♪  notebook  Access rights. ") from None
            cause=cause.__cause__ or cause.__context__
        raise
    model_id=V69_PARENT["MODEL_IDENTITY"]
    if any(model_id[k]!=archived["model"][k] for k in ("name","revision","dtype")):
        raise RuntimeError(" Models, weight revisions or precisions  v6.4  Unmatched ")

    V69_ID = dict(scope="v6p9_semantic_routes", version=1, model=model_id,
                 runtime=V69_PARENT["RUN_FINGERPRINT"], gpu=torch.cuda.get_device_name(V69_PARENT["backend"].device),
                 source_sha256=V69_HASHES,
                 engine_sha256=[hashlib.sha256(s.encode()).hexdigest() for s in (V6_CORE_SOURCE,V6_RUNTIME_SOURCE)],
                 assets_sha256=V69_DIGESTS, prior_archive=V69_REF["archive_sha256"])
    with V69_NS["writer68"](V69_RUN, TAKEOVER_LOCK_ID):
        V69_STUDY = V69_NS["RoutesStudy69"](V69_RUN/"experiment", V69_PARENT["backend"],
            V69_INPUTS, V69_REF, V69_ID, V69_PARENT["verify_live_run"])
    print(" model ", model_id["name"], " output ", V69_RUN, flush=True)

else:
    with V69_NS["writer68"](V69_RUN, TAKEOVER_LOCK_ID):
        V69_EXPECT_ID = dict(scope="v6p9_semantic_routes", version=1,
            source_sha256=V69_HASHES,
            engine_sha256=[hashlib.sha256(s.encode()).hexdigest() for s in (V6_CORE_SOURCE,V6_RUNTIME_SOURCE)],
            assets_sha256=V69_DIGESTS, prior_archive=V69_REF["archive_sha256"])
        V69_STUDY = V69_NS["open_saved69"](V69_RUN/"experiment", V69_INPUTS, V69_REF, V69_EXPECT_ID)
    print(" Breakpoints have been opened and verified; no models have been loaded. ")


##  Start the selected phase 

 The first run defaults to development. Freeze both primary confirmation estimands and random controls before execution; label development results exploratory. Do not stop confirmation early based on effect direction or significance, and do not silently drop scenarios after technical failures. 

`MAX_WALL_HOURS`  Limit only the duration of a single run, checking at complete checkpoint boundaries. Export at the time limit  PARTIAL， You can simply repeat it with the label.  smoke  and development-stage measurements; do not extrapolate full tool-trajectory speed from the first probe alone. 


In [ ]:
#@title  Run technical preflight and the selected stage, saving auditable partial results on exceptions 
V69_RESULT = None
try:
    with V69_NS["writer68"](V69_RUN):
        if RUN_MODE != "export":
            V69_STUDY.prepare69()
        if RUN_MODE == "all":
            V69_STUDY.run69(RUN_STAGE, max_wall_hours=MAX_WALL_HOURS)
except BaseException:
    try:
        with V69_NS["writer68"](V69_RUN):
            V69_RESULT = V69_NS["export69"](V69_STUDY, V69_ASSETS, RUN_STAGE)
        print(" Try to export the completed checkpoints. The original anomaly continues to be thrown. runtime  There will be no automatic disconnection. ")
    except Exception as export_error:
        print(" Partial export did not finish; individual checkpoints remain saved. Export exception type: ", type(export_error).__name__)
    raise


In [ ]:
#@title  Verify all tool terminals, calculate scenario-matching compartments and package to  Drive
V69_RESULT = None
with V69_NS["writer68"](V69_RUN):
    V69_RESULT = V69_NS["export69"](V69_STUDY, V69_ASSETS, RUN_STAGE)
print(json.dumps(V69_RESULT[0], ensure_ascii=False, indent=2))


##  Results, continuation and automatic disconnection 

 Default directory: `MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p9_semantic_routes/routes-main/`。

 The development phase and the confirmation phase each produce the stage name  COMPLETE / PARTIAL ZIP， The completion of the development cannot be considered as confirmation. ZIP  includes signed raw checkpoints, frozen inputs and source code, condition counts, scenario-paired effects, and summary reports. Authenticity is established by  ZIP   CRC、 Summary, list of documents and list of documents  `reports/report.json`  Subject to. 

 Before confirmation, inspect development-task success, free-generation position coverage, self-patching agreement, and runtime. To export existing results only, change the setting to  `RUN_MODE="export"`  Again.  Run all； This mode does not read  HF_TOKEN、 No model is loaded and will not be automatically separated. 


In [ ]:
#@title  Read & Synchronize Full Results  Drive  Automatically release the current  runtime
from google.colab import drive, runtime
V69_NS["finish69"](V69_STUDY, V69_RESULT, AUTO_DISCONNECT and RUN_MODE=="all",
    DRIVE_MOUNT, os.path.ismount, drive.flush_and_unmount, runtime.unassign)
